# RQ2 — VOV4 Full Expansion: Source-Level Eligibility Pool

## Context

- RQ1 is FINAL. This notebook never reads, imports, or touches any RQ1 artifact, frozen-test split, or frozen-test audio.
- Notebook 07 (VOV4 Discovery, 30 latest articles) is FINAL.
- Notebook 08 (VOV4 Media Probe on those 30 articles) is FINAL.
- Notebook 09 (VOV4 Pilot Download + Technical QA) is FINAL: it collected **12** public audio files,
  **8780.216 s (~2.43895 h)**, all **12/12 technical PASS**. Notebook 09 is only a **pilot**, not the
  final RQ2 `U_real` pool.

## Purpose of this notebook

Expand the VOV4 collection into the **full source-level candidate `U_real` pool**: discover many more
Bahnar-section articles across an explicit date window, probe their public media, reuse Notebook 09's
pilot audio where the exact same media already exists, download/extract any genuinely new public audio,
run the same technical QA, and classify each *source* (article + its one primary public media item) into
a conservative eligibility bucket. Per-file human listening is **not** required for every source.

## STRICTLY DO NOT (hard constraints enforced throughout this notebook)

- Run ASR.
- Run MT.
- Pseudo-label anything.
- Segment audio into training clips.
- Train any model.
- Access or use frozen-test content.
- Modify any RQ1 artifact.
- Overwrite Notebook 09 pilot artifacts (`artifacts/rq2/vov4_pilot_qa/**`) — that tree is read-only input here.

A static self-check near the end of this notebook re-reads this notebook's own saved `.ipynb` source and
fails closed if any cell contains a forbidden ASR/MT/pseudo-label/training import or call pattern.

## Methodology note on native-speaker spot-checks (documented, not a hard rule)

Native-speaker spot-check is **supporting evidence only**. It is not a calibrated probability, and it does
**not** by itself set `source_level_status`.

What the current evidence actually is:

- Three pilot sources were spot-checked by a Bahnar-speaking reviewer.
- The reviewer recognized the speech as predominantly Bahnar.
- Subjective confidence was approximately 80–90%.
- Uncertainty is due to regional/dialectal variation.

This notebook does **not** treat that note as a human-verified label on all 12 Notebook 09 pilot files,
and it does **not** rewrite finalized Notebook 09 artifacts to represent it. Optional per-source notes for
this expansion live in `artifacts/rq2/vov4_full/native_spotcheck.csv`, keyed by `source_id` +
`media_url_sha256` (`native_spotcheck_status`, `native_spotcheck_notes`, `reviewer`). If that file is
absent, or a row is missing, `native_spotcheck_status` is `NOT_PERFORMED`. Eligibility still rests only on
the conservative source-provenance rules below.

## 1. Imports + config

Runs on the `bahnar-s2tt` venv (macOS). Reuses the same safe-networking posture already established in
Notebook 07/08/09: http(s) only, public hosts only, `robots.txt` fail-closed, manual (not automatic)
redirects, retries only on transient errors, and explicit byte budgets. All persisted paths are
project-relative POSIX paths (Section 11 of the spec).


In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("bs4") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "beautifulsoup4"])

import hashlib
import ipaddress
import json
import math
import os
import re
import shutil
import socket
import time
import unicodedata
import wave
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import unquote, urljoin, urlparse, urlunparse
from urllib.robotparser import RobotFileParser

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display


def find_project_root() -> Path:
    """Repo root whether the kernel cwd is the thesis root or notebooks/."""
    node = Path.cwd().resolve()
    for cand in [node, *node.parents]:
        if (cand / "requirements.txt").is_file() and (cand / "notebooks").is_dir() and (cand / "src").is_dir():
            return cand
    raise RuntimeError(f"Cannot locate bahnar-s2tt-thesis root from {node}")


PROJECT_ROOT = find_project_root()
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "10_RQ2_VOV4_Full_Expansion_Source_Eligibility.ipynb"

DISCOVERY07_DIR = PROJECT_ROOT / "artifacts" / "rq2" / "vov4_discovery"
PROBE08_DIR = PROJECT_ROOT / "artifacts" / "rq2" / "vov4_media_probe"
PILOT09_DIR = PROJECT_ROOT / "artifacts" / "rq2" / "vov4_pilot_qa"

OUT_DIR = PROJECT_ROOT / "artifacts" / "rq2" / "vov4_full"
DISCOVERY_DIR = OUT_DIR / "discovery"
MEDIA_DIR = OUT_DIR / "media"
RAW_DIR = OUT_DIR / "raw"
WAV_DIR = OUT_DIR / "wav16k"
QA_DIR = OUT_DIR / "qa"

for _d in (OUT_DIR, DISCOVERY_DIR, MEDIA_DIR, RAW_DIR, WAV_DIR, QA_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --- Date scope (Section 1) --------------------------------------------------
PHASE_A_LABEL = "A"
PHASE_A_START = "2026-01-01"
PHASE_A_END = "2026-09-24"
PHASE_B_LABEL = "B"
PHASE_B_START = "2025-01-01"
PHASE_B_END = "2025-12-31"
PHASE_B_TRIGGER_HOURS = 10.0  # Phase B only runs if Phase A eligible-speech hours < this

# --- VOV4 Bahnar section base + safe-crawl budgets --------------------------
BASE_HOST = "vov4.vov.vn"
START_URL = "https://vov4.vov.vn/bahnar"
SLEEP_SECONDS = 1.2
FETCH_ATTEMPTS = 3
RETRY_BACKOFF_SECONDS = (2, 4, 8)
MAX_REDIRECTS = 5
MAX_PAGES_PER_SECTION = 800   # resume cap; still PARTIAL if a section hits it before the date window is covered
STOP_AFTER_EMPTY_PAGES = 2    # consecutive pages whose trusted dates are entirely older than the window

# --- Download budgets (new network bytes only; reused pilot bytes are separate) ---
CHUNK_BYTES = 1024 * 256


def positive_int_env(name: str, default: int) -> int:
    """Explicit positive-integer override. Empty/unset keeps default. Anything else fails closed."""
    raw = os.environ.get(name, "").strip()
    if raw == "":
        value = int(default)
    elif raw.isdigit() and int(raw) > 0:
        value = int(raw)
    else:
        raise RuntimeError(f"{name} must be a positive integer, got {raw!r}")
    if value <= 0:
        raise RuntimeError(f"{name} must be a positive integer, got {value}")
    return value


MAX_SINGLE_FILE_BYTES = positive_int_env("BAHNAR_VOV4_MAX_SINGLE_FILE_BYTES", 128 * 1024 ** 2)
# Conservative floor is 1 GiB. This interrupted run resumes at 8 GiB unless
# BAHNAR_VOV4_MAX_TOTAL_DOWNLOAD_BYTES overrides it. The resume check compares
# this value with checkpointed streamed bytes, not only successful bytes.
MAX_TOTAL_DOWNLOAD_BYTES = positive_int_env("BAHNAR_VOV4_MAX_TOTAL_DOWNLOAD_BYTES", 12 * 1024 ** 3)

QA_METRIC_FIELDS = (
    "peak_abs", "peak_dbfs", "rms", "rms_dbfs", "clipping_fraction", "silence_fraction_energy",
)
MIME_TO_AUDIO_EXT = {
    "audio/mpeg": "mp3",
    "audio/mp3": "mp3",
    "audio/aac": "aac",
    "audio/x-aac": "aac",
    "audio/mp4": "m4a",
    "audio/x-m4a": "m4a",
    "audio/wav": "wav",
    "audio/x-wav": "wav",
    "audio/wave": "wav",
}
MIME_TO_VIDEO_EXT = {"video/mp4": "mp4"}
SUPPORTED_MEDIA_TYPES = sorted(set(MIME_TO_AUDIO_EXT) | set(MIME_TO_VIDEO_EXT))
NATIVE_SPOTCHECK_COLUMNS = (
    "source_id", "media_url_sha256", "native_spotcheck_status", "native_spotcheck_notes", "reviewer",
)

USER_AGENT = (
    "BahnarS2TT-Thesis-Research/1.0 "
    "(+non-commercial academic full-expansion discovery; public media only)"
)

session = requests.Session()
session.headers.update({
    "User-Agent": USER_AGENT,
    "Accept-Language": "vi,en;q=0.8",
})

NETWORK_STATE = {
    "new_network_bytes_streamed_this_run": 0,   # every chunk of a *new* download, success or not
    "new_network_bytes_successful": 0,          # only bytes of downloads that finished successfully
    "reused_pilot_bytes": 0,                    # bytes reused from Notebook 09 pilot artifacts (not re-streamed)
}


def find_executable(name: str) -> str:
    found = shutil.which(name)
    if found:
        return found
    for extra in ("/opt/homebrew/bin", "/usr/local/bin"):
        candidate = Path(extra) / name
        if candidate.is_file():
            return str(candidate)
    return ""


FFMPEG = find_executable("ffmpeg")
FFPROBE = find_executable("ffprobe")
if not FFMPEG or not FFPROBE:
    raise RuntimeError(
        "ffmpeg/ffprobe not found on PATH or in /opt/homebrew/bin or /usr/local/bin. "
        "Install with `brew install ffmpeg` before running network/technical-QA cells."
    )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUT_DIR:", OUT_DIR)
print("PHASE_A:", PHASE_A_START, "->", PHASE_A_END)
print("PHASE_B (conditional):", PHASE_B_START, "->", PHASE_B_END)
print("FFMPEG:", FFMPEG)
print("FFPROBE:", FFPROBE)


PROJECT_ROOT: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis
OUT_DIR: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/artifacts/rq2/vov4_full
PHASE_A: 2026-01-01 -> 2026-09-24
PHASE_B (conditional): 2025-01-01 -> 2025-12-31
FFMPEG: /opt/homebrew/bin/ffmpeg
FFPROBE: /opt/homebrew/bin/ffprobe


/Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. Path portability helpers

Every persisted artifact path in this notebook is a project-relative POSIX path, resolvable on any
machine as `PROJECT_ROOT / relative_path` (Section 11). This also lets us store a Notebook 09 pilot path
(`artifacts/rq2/vov4_pilot_qa/raw/...`) side by side with a new Notebook 10 path
(`artifacts/rq2/vov4_full/raw/...`) in the same manifest column without any migration needed — both are
already project-relative.


In [2]:
FORBIDDEN_PATH_MARKERS = ("/Users/", "/workspace/", "/home/", "C:/Users/")
ARTIFACT_PATH_FIELDS = ("raw_local_path", "wav_local_path", "video_container_local_path")


def to_project_relative(path) -> str:
    """Absolute or relative Path/str -> canonical project-relative POSIX string."""
    p = Path(path)
    if not p.is_absolute():
        p = (PROJECT_ROOT / p)
    return p.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()


def resolve_project_path(value: str) -> Path:
    """Project-relative POSIX string (or already-absolute path) -> absolute Path."""
    p = Path(value)
    if p.is_absolute():
        return p
    return (PROJECT_ROOT / value).resolve()


def is_portable_path_value(value: str) -> bool:
    if not value:
        return True
    if Path(value).is_absolute():
        return False
    if any(marker in value for marker in FORBIDDEN_PATH_MARKERS):
        return False
    if str(PROJECT_ROOT) in value or str(PROJECT_ROOT.resolve()) in value:
        return False
    return True


def assert_portable_artifact_paths(rows, fields=ARTIFACT_PATH_FIELDS) -> None:
    """Fail closed if any persisted path is absolute or machine-specific."""
    for row in rows:
        for field in fields:
            value = row.get(field, "") if isinstance(row, dict) else getattr(row, field, "")
            if not is_portable_path_value(str(value)):
                raise RuntimeError(f"Non-portable artifact path in field '{field}': {value!r}")


def _synthetic_portability_roundtrip() -> None:
    sample = PROJECT_ROOT / "artifacts" / "rq2" / "vov4_full" / "raw" / "x.mp3"
    rel = to_project_relative(sample)
    if rel != "artifacts/rq2/vov4_full/raw/x.mp3":
        raise AssertionError(rel)
    back = resolve_project_path(rel)
    if back != sample.resolve():
        raise AssertionError(back)
    if not is_portable_path_value(rel):
        raise AssertionError("relative path flagged as non-portable")
    if is_portable_path_value("/Users/someone/artifacts/rq2/vov4_full/raw/x.mp3"):
        raise AssertionError("absolute path was not flagged")
    try:
        assert_portable_artifact_paths([{"raw_local_path": "/Users/someone/x.mp3", "wav_local_path": ""}])
        raise AssertionError("assert_portable_artifact_paths did not raise on absolute path")
    except RuntimeError:
        pass


_synthetic_portability_roundtrip()
print("portability helpers self-check: PASS")


portability helpers self-check: PASS


## 3. Load + validate upstream artifacts (Notebook 07 / 08 / 09) + fingerprints

Read-only. This notebook never writes into `vov4_discovery/`, `vov4_media_probe/`, or `vov4_pilot_qa/`.
It fingerprints each upstream `summary.json` (and the key manifest files it actually reuses) so
`contract.json` can record exactly which upstream state this run was built against.


In [3]:
def utc_now() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).strftime("%Y-%m-%dT%H:%M:%SZ")


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def is_sha256_hex(value: str) -> bool:
    return bool(re.fullmatch(r"[0-9a-f]{64}", (value or "").strip().lower()))


NB07_SUMMARY_PATH = DISCOVERY07_DIR / "summary.json"
NB07_CANDIDATES_CSV = DISCOVERY07_DIR / "vov4_candidates.csv"
NB08_SUMMARY_PATH = PROBE08_DIR / "summary.json"
NB08_PROBE_CSV = PROBE08_DIR / "vov4_media_probe.csv"
NB09_SUMMARY_PATH = PILOT09_DIR / "summary.json"
NB09_MANIFEST_CSV = PILOT09_DIR / "pilot_manifest.csv"
NB09_TECHNICAL_QA_CSV = PILOT09_DIR / "technical_qa.csv"

for _required in (
    NB07_SUMMARY_PATH, NB07_CANDIDATES_CSV,
    NB08_SUMMARY_PATH, NB08_PROBE_CSV,
    NB09_SUMMARY_PATH, NB09_MANIFEST_CSV, NB09_TECHNICAL_QA_CSV,
):
    if not _required.is_file():
        raise RuntimeError(f"Missing required upstream artifact: {_required}")

nb07_summary = load_json(NB07_SUMMARY_PATH)
nb08_summary = load_json(NB08_SUMMARY_PATH)
nb09_summary = load_json(NB09_SUMMARY_PATH)

if nb07_summary.get("status") != "SUCCESS_VOV4_DISCOVERY":
    raise RuntimeError(f"Notebook 07 is not PASS: {nb07_summary.get('status')}")
if nb08_summary.get("status") != "SUCCESS_VOV4_MEDIA_PROBE":
    raise RuntimeError(f"Notebook 08 is not PASS: {nb08_summary.get('status')}")
if nb09_summary.get("status") != "SUCCESS_VOV4_PILOT_QA":
    raise RuntimeError(f"Notebook 09 is not PASS: {nb09_summary.get('status')}")

UPSTREAM_FINGERPRINTS = {
    "nb07_candidates_csv_sha256": sha256_file(NB07_CANDIDATES_CSV),
    "nb07_summary_sha256": sha256_file(NB07_SUMMARY_PATH),
    "nb08_media_probe_csv_sha256": sha256_file(NB08_PROBE_CSV),
    "nb08_summary_sha256": sha256_file(NB08_SUMMARY_PATH),
    "nb09_pilot_manifest_csv_sha256": sha256_file(NB09_MANIFEST_CSV),
    "nb09_technical_qa_csv_sha256": sha256_file(NB09_TECHNICAL_QA_CSV),
    "nb09_summary_sha256": sha256_file(NB09_SUMMARY_PATH),
    "nb09_reported_n_technical_pass": nb09_summary.get("n_technical_pass"),
    "nb09_reported_downloaded_duration_hours": nb09_summary.get("downloaded_duration_hours"),
}

pilot_df = pd.read_csv(NB09_MANIFEST_CSV, dtype=str).fillna("")
if pilot_df["media_url_sha256"].duplicated().any():
    raise RuntimeError("Notebook 09 pilot_manifest.csv has duplicate media_url_sha256")
if pilot_df["source_id"].duplicated().any():
    raise RuntimeError("Notebook 09 pilot_manifest.csv has duplicate source_id")
PILOT_BY_MEDIA_HASH = {row["media_url_sha256"]: row for row in pilot_df.to_dict(orient="records")}

pilot_qa_df = pd.read_csv(NB09_TECHNICAL_QA_CSV, dtype=str).fillna("")
if pilot_qa_df["source_id"].duplicated().any():
    raise RuntimeError("Notebook 09 technical_qa.csv has duplicate source_id")
for _field in QA_METRIC_FIELDS:
    if _field not in pilot_qa_df.columns:
        raise RuntimeError(f"Notebook 09 technical_qa.csv missing metric column {_field}")
PILOT_QA_BY_SOURCE_ID = {row["source_id"]: row for row in pilot_qa_df.to_dict(orient="records")}

print("nb07 status:", nb07_summary.get("status"), "| nb08 status:", nb08_summary.get("status"))
print("nb09 status:", nb09_summary.get("status"),
      "| n_technical_pass:", nb09_summary.get("n_technical_pass"),
      "| downloaded_duration_hours:", nb09_summary.get("downloaded_duration_hours"))
print("pilot media indexed for reuse:", len(PILOT_BY_MEDIA_HASH))
print("pilot technical_qa rows indexed:", len(PILOT_QA_BY_SOURCE_ID))
print("configured MAX_SINGLE_FILE_BYTES:", MAX_SINGLE_FILE_BYTES)
print("configured MAX_TOTAL_DOWNLOAD_BYTES:", MAX_TOTAL_DOWNLOAD_BYTES)

nb07 status: SUCCESS_VOV4_DISCOVERY | nb08 status: SUCCESS_VOV4_MEDIA_PROBE
nb09 status: SUCCESS_VOV4_PILOT_QA | n_technical_pass: 12 | downloaded_duration_hours: 2.438948888888889
pilot media indexed for reuse: 12
pilot technical_qa rows indexed: 12
configured MAX_SINGLE_FILE_BYTES: 134217728
configured MAX_TOTAL_DOWNLOAD_BYTES: 12884901888


## 4. Safe networking helpers (reused from Notebook 07/08/09)

Same fail-closed posture: http(s) only, no localhost/private/link-local/multicast/reserved/non-global
hosts, `robots.txt` must be verifiable (200, not a redirect, contains "user-agent") or the request is
refused, redirects are followed manually (max 5 hops) re-checking host + robots at every hop, and only
transient errors (429 / 5xx / timeout / connection error) are retried.


In [4]:
class RobotsUnverified(RuntimeError):
    pass


class NonPublicHost(RuntimeError):
    pass


class TransientHTTP(RuntimeError):
    pass


class TotalDownloadBudgetExceeded(RuntimeError):
    """The run-wide new-download budget was exceeded. This is systemic, not a per-file QA failure."""


class FetchFailed(RuntimeError):
    def __init__(self, page_url: str, error_type: str, error_message: str, attempts: int):
        self.page_url = page_url
        self.error_type = error_type
        self.error_message = error_message
        self.attempts = int(attempts)
        super().__init__(f"{error_type} after {attempts} attempt(s): {error_message}")


ROBOTS = {}
PUBLIC_HOSTS = {}


def canonicalize(base_url: str, raw_url: str) -> str:
    absolute = urljoin(base_url, (raw_url or "").strip())
    parts = urlparse(absolute)
    if parts.scheme not in {"http", "https"}:
        return ""
    return urlunparse((parts.scheme.lower(), parts.netloc.lower(), parts.path or "/", "", parts.query, ""))


def assert_public_http_host(url: str) -> None:
    host = (urlparse(url).hostname or "").lower().rstrip(".")
    if not host:
        raise NonPublicHost(f"host khong hop le: {url}")
    if host in PUBLIC_HOSTS:
        if PUBLIC_HOSTS[host] is not None:
            raise NonPublicHost(PUBLIC_HOSTS[host])
        return
    if host in {"localhost", "localhost.localdomain"} or host.endswith(".localhost") or host.endswith(".local"):
        message = f"local host bi chan: {host}"
        PUBLIC_HOSTS[host] = message
        raise NonPublicHost(message)
    addresses = []
    try:
        addresses.append(ipaddress.ip_address(host))
    except ValueError:
        try:
            infos = socket.getaddrinfo(host, None)
        except socket.gaierror as exc:
            message = f"khong resolve duoc {host}: {exc}"
            PUBLIC_HOSTS[host] = message
            raise NonPublicHost(message) from exc
        if not infos:
            message = f"khong resolve duoc {host}"
            PUBLIC_HOSTS[host] = message
            raise NonPublicHost(message)
        for info in infos:
            addresses.append(ipaddress.ip_address(info[4][0].split("%", 1)[0]))
    for ip in addresses:
        if (
            ip.is_private or ip.is_loopback or ip.is_link_local or ip.is_multicast
            or ip.is_reserved or ip.is_unspecified or not ip.is_global
        ):
            message = f"non-public address bi chan: {host} -> {ip}"
            PUBLIC_HOSTS[host] = message
            raise NonPublicHost(message)
    PUBLIC_HOSTS[host] = None


def robots_for(url: str) -> RobotFileParser:
    parts = urlparse(url)
    if parts.scheme not in {"http", "https"} or not parts.netloc:
        raise RobotsUnverified(f"URL khong phai http(s): {url}")
    assert_public_http_host(url)
    origin = f"{parts.scheme}://{parts.netloc}"
    if origin in ROBOTS:
        return ROBOTS[origin]
    robots_url = origin + "/robots.txt"
    time.sleep(SLEEP_SECONDS)
    response = session.get(robots_url, timeout=20, allow_redirects=False)
    try:
        if response.status_code in {301, 302, 303, 307, 308}:
            raise RobotsUnverified(f"robots.txt redirect, khong xac minh duoc: {robots_url}")
        if response.status_code != 200 or "user-agent" not in response.text.lower():
            raise RobotsUnverified(f"khong xac minh duoc robots.txt: {robots_url} HTTP {response.status_code}")
        parser = RobotFileParser()
        parser.set_url(robots_url)
        parser.parse(response.text.splitlines())
    finally:
        response.close()
    ROBOTS[origin] = parser
    return parser


def robots_decision(url: str) -> str:
    parser = robots_for(url)
    return "allow" if parser.can_fetch(USER_AGENT, url) else "disallow"


def _is_transient_fetch_error(exc: BaseException) -> bool:
    if isinstance(exc, (TransientHTTP, requests.exceptions.ReadTimeout, requests.exceptions.ConnectionError)):
        return True
    response = getattr(exc, "response", None)
    code_ = getattr(response, "status_code", 0) or 0
    return code_ == 429 or 500 <= code_ <= 599


def get_html_once(url: str, timeout: int = 30):
    """Manual redirects only. No hop is requested before host + robots checks."""
    current = canonicalize(url, url)
    seen = []
    for _hop in range(MAX_REDIRECTS):
        if not current:
            raise RuntimeError(f"URL khong hop le: {url}")
        if current in seen:
            raise RuntimeError(f"Redirect loop: {current}")
        seen.append(current)
        assert_public_http_host(current)
        if urlparse(current).netloc.lower() != BASE_HOST:
            raise RuntimeError(f"Redirect roi {BASE_HOST}: {current}")
        if robots_decision(current) == "disallow":
            raise PermissionError(f"robots.txt khong cho crawl: {current}")
        response = session.get(current, timeout=timeout, allow_redirects=False)
        try:
            if response.status_code in {301, 302, 303, 307, 308}:
                location = (response.headers.get("Location") or "").strip()
                if not location:
                    raise RuntimeError(f"Redirect thieu Location: {current}")
                current = canonicalize(current, location)
                continue
            if response.status_code == 429 or 500 <= response.status_code <= 599:
                raise TransientHTTP(f"HTTP {response.status_code} {current}")
            if response.status_code != 200:
                raise RuntimeError(f"HTTP {response.status_code} khong phai HTML: {current}")
            ctype = (response.headers.get("Content-Type") or "").lower()
            if "text/html" not in ctype and "application/xhtml" not in ctype:
                raise RuntimeError(f"Khong phai HTML: {ctype} -- {current}")
            return response.content, current
        finally:
            response.close()
    raise RuntimeError(f"Qua {MAX_REDIRECTS} redirect hops: {url}")


def fetch_html(url: str, timeout: int = 30):
    last_error = None
    for attempt in range(1, FETCH_ATTEMPTS + 1):
        try:
            content, final_url = get_html_once(url, timeout=timeout)
            return content, final_url, attempt
        except (FetchFailed, TotalDownloadBudgetExceeded):
            raise
        except Exception as exc:
            if not _is_transient_fetch_error(exc):
                raise FetchFailed(url, type(exc).__name__, str(exc), attempt) from exc
            last_error = exc
            if attempt >= FETCH_ATTEMPTS:
                break
            delay = RETRY_BACKOFF_SECONDS[attempt - 1]
            print(f"RETRY {attempt}/{FETCH_ATTEMPTS - 1} in {delay}s: {type(exc).__name__} {url}")
            time.sleep(delay)
    raise FetchFailed(url, type(last_error).__name__, str(last_error), FETCH_ATTEMPTS)


def check_content_length(declared: str, written: int, content_encoding: str) -> None:
    if content_encoding:
        return
    if not declared:
        return
    try:
        expected = int(declared)
    except ValueError:
        return
    if expected != written:
        raise RuntimeError(f"Content-Length mismatch: declared={expected}, actual={written}")


def account_new_network_chunk(nbytes: int) -> None:
    NETWORK_STATE["new_network_bytes_streamed_this_run"] += int(nbytes)
    if NETWORK_STATE["new_network_bytes_streamed_this_run"] > MAX_TOTAL_DOWNLOAD_BYTES:
        raise TotalDownloadBudgetExceeded(
            f"MAX_TOTAL_DOWNLOAD_BYTES exceeded for this run: "
            f"{NETWORK_STATE['new_network_bytes_streamed_this_run']} > {MAX_TOTAL_DOWNLOAD_BYTES}"
        )


def stream_media_to_file(url: str, dest_part: Path) -> dict:
    """Manual-redirect GET, stream to a .part file. Caller renames atomically on success."""
    current = canonicalize(url, url)
    seen = []
    for _hop in range(MAX_REDIRECTS):
        if not current:
            raise RuntimeError(f"URL khong hop le: {url}")
        if current in seen:
            raise RuntimeError(f"Redirect loop: {current}")
        seen.append(current)
        assert_public_http_host(current)
        if robots_decision(current) == "disallow":
            raise PermissionError(f"robots.txt khong cho tai: {current}")
        time.sleep(SLEEP_SECONDS)
        response = session.get(current, timeout=60, stream=True, allow_redirects=False)
        try:
            if response.status_code in {301, 302, 303, 307, 308}:
                location = response.headers.get("Location") or ""
                if not location.strip():
                    raise RuntimeError(f"Redirect thieu Location: {current}")
                current = canonicalize(current, location.strip())
                continue
            if response.status_code == 429 or 500 <= response.status_code <= 599:
                raise TransientHTTP(f"HTTP {response.status_code} tai {current}")
            if response.status_code not in {200, 206}:
                raise RuntimeError(f"HTTP {response.status_code} khong tai duoc: {current}")
            declared_length = response.headers.get("Content-Length") or ""
            if declared_length and int(declared_length) > MAX_SINGLE_FILE_BYTES:
                raise RuntimeError(f"File vuot MAX_SINGLE_FILE_BYTES: {declared_length}")
            written = 0
            with dest_part.open("wb") as handle:
                for chunk in response.iter_content(chunk_size=CHUNK_BYTES):
                    if not chunk:
                        continue
                    written += len(chunk)
                    account_new_network_chunk(len(chunk))
                    if written > MAX_SINGLE_FILE_BYTES:
                        raise RuntimeError("MAX_SINGLE_FILE_BYTES exceeded mid-stream")
                    handle.write(chunk)
            check_content_length(declared_length, written, response.headers.get("Content-Encoding") or "")
            NETWORK_STATE["new_network_bytes_successful"] += written
            return {
                "final_url": current,
                "http_status": str(response.status_code),
                "content_type": (response.headers.get("Content-Type") or "").split(";")[0].strip().lower(),
                "content_length_header": declared_length,
                "actual_file_bytes": written,
                "etag": response.headers.get("ETag") or "",
                "last_modified": response.headers.get("Last-Modified") or "",
            }
        finally:
            response.close()
    raise RuntimeError(f"Qua {MAX_REDIRECTS} redirect hops: {url}")


## 5. Bahnar section map + date-window helper

The VOV4 Bahnar landing page fans out into 12 category ("section") listing pages, each independently
paginated with a Drupal-style `?page=N` (0-indexed) query string. This mapping was confirmed by inspecting
`https://vov4.vov.vn/bahnar` and one category page's pager links; it is **not** used as a language ground
truth (per the repeated project rule) — it only assigns a coarse `content_class` per section, which
Section 7's conservative eligibility rules consume alongside real per-article checks. Three sections are
explicitly music/song sections and are always excluded from eligibility.


In [5]:
SECTION_CONTENT_CLASS = {
    "bai-hori-tay-nguyen-bai-hat-hay-tay-nguyen": "music",
    "choh-jang-sa-nha-nong": "agriculture",
    "johngom-pran-suc-khoe": "health",
    "kotong-ang-lom-topol-thoi-su-xa-hoi": "news",
    "nor-pore-phat-thanh": "radio_report",
    "pom-tong-san-khau-truyen-thanh-tieu-pham-cuoi-tuan": "radio_drama",
    "todrong-hori-tay-nguyen-dan-ca-tay-nguyen": "music",
    "todrong-juat-long-dong-so-van-hoa": "general_speech",
    "todrong-toroi-so-ki-van-nghe-dan-gian": "mixed",
    "toring-torang-nao-nong-thon-moi": "rural_affairs",
    "trong-jang-teh-dak-tojur-an-pang-todrong-horih-chinh-sach-va-cuoc-song": "news",
    "video-clip-bai-joh-hori-video-clip-ca-nhac": "music",
}
ALL_SECTION_SLUGS = tuple(SECTION_CONTENT_CLASS.keys())
MUSIC_SECTIONS = {slug for slug, cls in SECTION_CONTENT_CLASS.items() if cls == "music"}
ALLOWED_CONTENT_CLASSES = {
    "news", "radio_report", "interview", "health", "agriculture", "rural_affairs",
    "spoken_narrative", "general_speech", "radio_drama", "music", "mixed", "unknown",
}
SPEECH_ELIGIBLE_CLASSES = {
    "news", "radio_report", "interview", "health", "agriculture",
    "rural_affairs", "spoken_narrative", "general_speech",
}
AMBIGUOUS_NEEDS_REVIEW_CLASSES = {"radio_drama", "mixed", "unknown"}


def section_content_class(section_slug: str) -> str:
    return SECTION_CONTENT_CLASS.get(section_slug, "unknown")


def in_phase_window(date_str: str, start: str, end: str) -> bool:
    value = (date_str or "").strip()
    if not value:
        return False
    return start <= value <= end


def _synthetic_phase_window() -> None:
    if not in_phase_window("2026-05-01", PHASE_A_START, PHASE_A_END):
        raise AssertionError("date inside window should pass")
    if in_phase_window("2024-01-01", PHASE_A_START, PHASE_A_END):
        raise AssertionError("date outside window should fail")
    if in_phase_window("", PHASE_A_START, PHASE_A_END):
        raise AssertionError("empty date should fail")


_synthetic_phase_window()
print("sections mapped:", len(SECTION_CONTENT_CLASS), "| music sections:", sorted(MUSIC_SECTIONS))


sections mapped: 12 | music sections: ['bai-hori-tay-nguyen-bai-hat-hay-tay-nguyen', 'todrong-hori-tay-nguyen-dan-ca-tay-nguyen', 'video-clip-bai-joh-hori-video-clip-ca-nhac']


## 6. Article URL + article-metadata helpers (ported from Notebook 07)

Identical article-shape rule to Notebook 07: a Bahnar article is
`https://vov4.vov.vn/bahnar/<section>/<slug>.vov4`, never a bare section index and never a media file.
`source_id` uses the exact same scheme as Notebook 07/08/09 (`VOV4_` + upper SHA1 of the canonical page
URL), so if a Notebook 07 pilot article happens to also be discovered again here, it gets the *same*
`source_id` — deduplication across notebooks falls out for free.


In [6]:
BLOCKED_MEDIA_EXT = (".jpg", ".jpeg", ".png", ".gif", ".webp", ".mp3", ".mp4", ".pdf", ".xml", ".rss")


def is_bahnar_article(url: str) -> bool:
    parts = urlparse(url)
    if parts.scheme not in {"http", "https"} or parts.netloc.lower() != BASE_HOST:
        return False
    if not parts.path.startswith("/bahnar/"):
        return False
    segments = [seg for seg in parts.path.split("/") if seg]
    if len(segments) < 3 or segments[0].lower() != "bahnar":
        return False
    slug = segments[-1].lower()
    if not segments[1] or not slug:
        return False
    if slug.endswith(BLOCKED_MEDIA_EXT):
        return False
    return slug.endswith(".vov4")


def article_section_slug(url: str) -> str:
    segments = [seg for seg in urlparse(url).path.split("/") if seg]
    if len(segments) >= 3 and segments[0].lower() == "bahnar":
        return segments[1]
    return ""


def source_id(url: str) -> str:
    return "VOV4_" + hashlib.sha1(url.encode("utf-8")).hexdigest().upper()


def normalize_text(value: str) -> str:
    return re.sub(r"\s+", " ", value or "").strip()


def parse_date_to_iso(text: str) -> str:
    for pattern in (
        re.compile(r"\b(\d{1,2})/(\d{1,2})/(\d{4})\b"),
        re.compile(r"\b(\d{1,2})-(\d{1,2})-(\d{4})\b"),
    ):
        match = pattern.search(text or "")
        if not match:
            continue
        day, month, year = map(int, match.groups())
        try:
            return datetime(year, month, day).date().isoformat()
        except ValueError:
            continue
    return ""


def _date_from_raw(raw: str) -> str:
    match = re.search(r"\b(\d{4})-(\d{2})-(\d{2})\b", raw or "")
    if match:
        return "-".join(match.groups())
    return parse_date_to_iso(raw)


def extract_title(soup: BeautifulSoup) -> str:
    for selector in ('meta[property="og:title"]', 'meta[name="twitter:title"]'):
        node = soup.select_one(selector)
        if node and node.get("content"):
            return normalize_text(node.get("content"))
    heading = soup.find("h1")
    return normalize_text(heading.get_text(" ", strip=True)) if heading else ""


def extract_description(soup: BeautifulSoup) -> str:
    for selector in ('meta[property="og:description"]', 'meta[name="description"]', 'meta[name="twitter:description"]'):
        node = soup.select_one(selector)
        if node and node.get("content"):
            return normalize_text(node.get("content"))
    return ""


def extract_published_date(soup: BeautifulSoup):
    """Trusted dates are metadata or the article's own publication-time node.

    VOV4 renders that time as div.article-date, div.time-published, or
    div.video__time inside <article>. A date anywhere else on the page,
    including the site license line, stays body_fallback.
    """
    for selector in ('meta[property="article:published_time"]', 'meta[name="pubdate"]', 'meta[name="publish-date"]'):
        node = soup.select_one(selector)
        if node is None:
            continue
        parsed = _date_from_raw((node.get("content") or "").strip())
        if parsed:
            return parsed, "metadata"
    time_node = soup.select_one("article time[datetime]") or soup.select_one("time[datetime]")
    if time_node is not None:
        raw = (time_node.get("datetime") or time_node.get("content") or "").strip()
        parsed = _date_from_raw(raw)
        if parsed:
            return parsed, "time"
    article = soup.select_one("article")
    if article is not None:
        for selector in ("div.article-date", "div.time-published", "div.video__time"):
            node = article.select_one(selector)
            if node is None:
                continue
            parsed = parse_date_to_iso(normalize_text(node.get_text(" ", strip=True)))
            if parsed:
                return parsed, "time"
    parsed = parse_date_to_iso(normalize_text(soup.get_text(" ", strip=True)))
    if parsed:
        return parsed, "body_fallback"
    return "", "missing"


## 7. Media-extraction helpers (ported from Notebook 08, unchanged logic)

Only media that appears **inside the primary article scope**
(`article[role=article][about=<pathname>]`) counts; related/sidebar/header/footer widgets never do.
`DATA_URL_ATTRS` is the same fixed whitelist as Notebook 08. This block adds no new discovery heuristics —
it is a direct port so that the same public-media surface Notebook 08 already validated is reused as-is.


In [7]:
MEDIA_EXTENSIONS = (".m3u8", ".m4a", ".mp3", ".mp4", ".aac", ".wav")
AUDIO_EXTENSIONS = {"mp3", "aac", "m4a", "wav"}
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".webp", ".gif")
STRONG_MEDIA_KEYS = {"file", "audio", "video", "media", "playlist", "stream", "sources"}
DATA_URL_ATTRS = {"data-url", "data-src", "data-file", "data-audio", "data-video", "data-stream", "data-playlist"}
HLS_TYPES = {"application/vnd.apple.mpegurl", "application/x-mpegurl"}
BINARY_TYPES = {"application/octet-stream", "binary/octet-stream"}

INLINE_URL_RE = re.compile("(?P<q>[\"\'])(?P<url>(?:https?:)?//[^\\s\"\'<>]+|/[^\\s\"\'<>]+)(?P=q)")
INLINE_ASSIGN_RE = re.compile(
    "(?<![A-Za-z0-9_])"
    "(?P<key>sources|source|playlist|stream|audio|video|media|file|src)"
    "\\s*[:=]\\s*"
    "(?P<q>[\"\'])"
    "(?P<url>[^\"\'\\s<>]+)"
    "(?P=q)",
    re.IGNORECASE,
)


def media_extension(url: str) -> str:
    path = urlparse(url).path.lower()
    for ext in MEDIA_EXTENSIONS:
        if path.endswith(ext):
            return ext[1:]
    return ""


def is_image_url(url: str) -> bool:
    path = urlparse(url).path.lower()
    return any(path.endswith(ext) for ext in IMAGE_EXTENSIONS)


def looks_like_url_ref(raw: str) -> bool:
    value = (raw or "").strip()
    if not value or value.lower().startswith(("data:", "javascript:", "blob:")):
        return False
    lowered = value.lower().split("?", 1)[0].split("#", 1)[0]
    if any(lowered.endswith(ext) for ext in IMAGE_EXTENSIONS):
        return False
    if value.startswith(("http://", "https://", "//", "/", "./", "../")):
        return True
    if "/" in value:
        return True
    return any(lowered.endswith(ext) for ext in MEDIA_EXTENSIONS)


def media_type_for(extension: str, hint: str = "") -> str:
    if extension == "m3u8":
        if hint == "audio":
            return "audio"
        if hint == "video":
            return "video"
        return "unknown_stream"
    if hint in {"audio", "video"}:
        return hint
    if extension in AUDIO_EXTENSIONS:
        return "audio"
    if extension == "mp4":
        return "video"
    return ""


def consider_url(found: dict, page_url: str, raw_url: str, method: str, hint: str = "", strong: bool = False) -> None:
    raw_url = (raw_url or "").strip()
    if not raw_url:
        return
    media_url = canonicalize(page_url, raw_url)
    if not media_url or is_image_url(media_url):
        return
    if method in {"audio_tag", "video_tag", "og_audio", "og_video"}:
        strong = True
    elif method == "source_tag":
        strong = strong or hint in {"audio", "video"}
    elif method == "href_src_scan":
        strong = False
    extension = media_extension(media_url)
    if not extension and not strong:
        return
    kind = media_type_for(extension, hint)
    if not kind and strong:
        kind = "unknown_stream"
    if not kind:
        return
    if media_url in found:
        return
    parts = urlparse(media_url)
    found[media_url] = {
        "media_url": media_url,
        "media_type": kind,
        "media_extension": extension,
        "media_host": parts.netloc.lower(),
        "discovery_method": method,
        "has_query_token": bool(parts.query),
        "is_public_direct_url": False,
    }


def walk_json(node, bucket: list, path: tuple = ()) -> None:
    if isinstance(node, dict):
        raw_type = node.get("@type", node.get("type", ""))
        if isinstance(raw_type, list):
            type_name = " ".join(str(item) for item in raw_type).lower()
        elif isinstance(raw_type, str):
            type_name = raw_type.lower()
        else:
            type_name = ""
        mime = ""
        for child_key, value in node.items():
            if str(child_key).lower() in {"encodingformat", "mime", "mimetype", "contenttype"} and isinstance(value, str):
                mime = value.lower()
        for child_key, value in node.items():
            key = str(child_key).lower()
            walk_json(value, bucket, path + ((key, type_name, mime),))
    elif isinstance(node, list):
        for value in node:
            walk_json(value, bucket, path)
    elif isinstance(node, str):
        bucket.append((path, node))


def json_media_decision(path: tuple) -> tuple:
    if not path:
        return False, ""
    key, type_name, mime = path[-1]
    if not mime and (type_name.startswith("audio/") or type_name.startswith("video/")):
        mime = type_name
    parent_keys = " ".join(item[0] for item in path[:-1])
    hint = ""
    if mime.startswith("audio/") or "audioobject" in type_name or key == "audio":
        hint = "audio"
    elif mime.startswith("video/") or "videoobject" in type_name or key == "video":
        hint = "video"
    if not hint and "audio" in parent_keys and "video" not in parent_keys:
        hint = "audio"
    elif not hint and "video" in parent_keys and "audio" not in parent_keys:
        hint = "video"
    parent_context = any(token in parent_keys for token in ("audio", "video", "player", "media", "stream"))
    object_context = "audioobject" in type_name or "videoobject" in type_name
    mime_context = mime.startswith("audio/") or mime.startswith("video/") or mime in {
        "application/vnd.apple.mpegurl", "application/x-mpegurl",
    }
    strong = key in STRONG_MEDIA_KEYS or object_context or parent_context or mime_context
    return strong, hint


def add_json_value(found: dict, page_url: str, path: tuple, value: str, method: str) -> None:
    if not path or not looks_like_url_ref(value):
        return
    key = path[-1][0]
    if key in {"@type", "type", "encodingformat", "mime", "mimetype", "contenttype"}:
        return
    strong, hint = json_media_decision(path)
    consider_url(found, page_url, value, method, hint, strong=strong)


def inline_statement(script_text: str, start: int) -> str:
    chunk = script_text[max(0, start - 80):start]
    for sep in ("\n", ";", "{"):
        idx = chunk.rfind(sep)
        if idx != -1:
            chunk = chunk[idx + 1:]
    return chunk.lower()


def inline_has_media_context(script_text: str, start: int, end: int) -> bool:
    window = inline_statement(script_text, start)
    return any(token in window for token in ("audio", "video", "player", "media", "stream"))


def extract_inline_script(script_text: str, page_url: str, found: dict) -> None:
    if not script_text:
        return
    for match in INLINE_ASSIGN_RE.finditer(script_text):
        raw = match.group("url")
        if not looks_like_url_ref(raw):
            continue
        key = match.group("key").lower()
        hint = "audio" if key == "audio" else "video" if key == "video" else ""
        contextual = inline_has_media_context(script_text, match.start(), match.end())
        if not hint and key in {"src", "source"}:
            window = inline_statement(script_text, match.start())
            if "audio" in window and "video" not in window:
                hint = "audio"
            elif "video" in window and "audio" not in window:
                hint = "video"
        strong = key in STRONG_MEDIA_KEYS or (key in {"src", "source"} and contextual)
        consider_url(found, page_url, raw, "inline_script", hint, strong=strong)
    for match in INLINE_URL_RE.finditer(script_text):
        raw = match.group("url")
        if not looks_like_url_ref(raw):
            continue
        absolute = canonicalize(page_url, raw)
        if absolute and media_extension(absolute):
            consider_url(found, page_url, raw, "inline_script", "", strong=False)


def find_primary_article(soup: BeautifulSoup, page_url: str):
    pathname = urlparse(page_url).path
    if not pathname:
        return None
    matches = [
        node for node in soup.find_all("article")
        if node.get("role") == "article" and (node.get("about") or "") == pathname
    ]
    if len(matches) != 1:
        return None
    return matches[0]


def extract_media(root, page_url: str) -> list:
    found = {}
    for tag in root.find_all("audio"):
        if tag.get("src"):
            consider_url(found, page_url, tag.get("src"), "audio_tag", "audio")
        for source in tag.find_all("source"):
            raw = source.get("src") or source.get("data-src")
            if raw:
                consider_url(found, page_url, raw, "source_tag", "audio")
    for tag in root.find_all("video"):
        if tag.get("src"):
            consider_url(found, page_url, tag.get("src"), "video_tag", "video")
        for source in tag.find_all("source"):
            raw = source.get("src") or source.get("data-src")
            if raw:
                consider_url(found, page_url, raw, "source_tag", "video")
    for source in root.find_all("source"):
        if source.find_parent(["audio", "video"]) is not None:
            continue
        raw = source.get("src") or source.get("data-src")
        mime = (source.get("type") or "").lower()
        hint = "audio" if mime.startswith("audio/") else "video" if mime.startswith("video/") else ""
        hls_mime = mime in HLS_TYPES
        if raw:
            consider_url(found, page_url, raw, "source_tag", hint, strong=hls_mime)
    for prop, method, hint in (("og:audio", "og_audio", "audio"), ("og:video", "og_video", "video")):
        for node in root.select(f'meta[property="{prop}"]'):
            if node.get("content"):
                consider_url(found, page_url, node.get("content"), method, hint)
    for script in root.find_all("script", attrs={"type": "application/ld+json"}):
        pairs = []
        try:
            walk_json(json.loads(script.string or ""), pairs)
        except (TypeError, json.JSONDecodeError):
            continue
        for path, value in pairs:
            add_json_value(found, page_url, path, value, "json_ld")
    for script in root.find_all("script", attrs={"type": "application/json"}):
        pairs = []
        try:
            walk_json(json.loads(script.string or ""), pairs)
        except (TypeError, json.JSONDecodeError):
            continue
        for path, value in pairs:
            add_json_value(found, page_url, path, value, "script_config")
    for script in root.find_all("script"):
        script_type = (script.get("type") or "").lower()
        if script_type in {"application/json", "application/ld+json"}:
            continue
        extract_inline_script(script.string or "", page_url, found)
    for tag in root.find_all(True):
        classes = " ".join(tag.get("class") or []).lower()
        hint = "audio" if "audio" in classes else "video" if "video" in classes else ""
        for key, value in tag.attrs.items():
            if str(key).lower() not in DATA_URL_ATTRS or not isinstance(value, str):
                continue
            attr_key = str(key)[5:].lower()
            strong = hint in {"audio", "video"} or attr_key in {"audio", "video", "media", "playlist", "stream"}
            consider_url(found, page_url, value, "data_attribute", hint, strong=strong)
    for tag in root.find_all(True):
        for attr in ("src", "href"):
            value = tag.get(attr)
            if isinstance(value, str):
                consider_url(found, page_url, value, "href_src_scan")
    return list(found.values())


def apply_content_type(media: dict, ctype: str) -> None:
    if ctype.startswith("audio/"):
        media["media_type"] = "audio"
    elif ctype.startswith("video/"):
        media["media_type"] = "video"
    elif ctype in HLS_TYPES and media["media_type"] not in {"audio", "video"}:
        media["media_type"] = "unknown_stream"


def content_type_is_media(ctype: str, media_type: str) -> bool:
    if ctype.startswith("audio/") or ctype.startswith("video/") or ctype in HLS_TYPES:
        return True
    return ctype in BINARY_TYPES and media_type in {"audio", "video", "unknown_stream"}


def content_type_is_non_media(ctype: str) -> bool:
    if ctype.startswith("image/") or ctype.startswith("font/"):
        return True
    return ctype in {
        "text/css", "text/javascript", "application/javascript", "application/x-javascript",
        "application/font-woff", "application/x-font-ttf", "application/x-font-woff",
    }


## 8. Section-scoped link extraction (pure, offline-testable)

`extract_section_links_from_html` keeps a URL only when it is a Bahnar article inside the page's
main listing container (`.view-content`, otherwise `main`) **and** `article_section_slug(url)`
equals the listing `section_slug`. Same-section links in sidebar, related, footer, or header blocks
are not listing evidence. A page with no proven main-listing container is `LISTING_SCOPE_UNVERIFIED`
and is not parsed as every anchor on the page. The network crawl fetches each canonical article at
most once and checks the section again from the final URL after redirects.

In [8]:
_EXCLUDED_LISTING_TAGS = frozenset({"aside", "footer", "header", "nav"})
_EXCLUDED_LISTING_ROLES = frozenset({"complementary", "banner", "contentinfo", "navigation"})
_EXCLUDED_LISTING_TOKENS = frozenset({
    "sidebar", "related", "recommend", "recommended", "footer", "header",
    "featured", "widget", "breadcrumb",
})


def _listing_class_tokens(node) -> set:
    raw = " ".join(node.get("class") or []) + " " + str(node.get("id") or "")
    return set(raw.lower().replace("_", " ").replace("-", " ").split())


def _is_pager_container(node) -> bool:
    """Drupal/VOV4 pager containers. A site-header nav is not a pager."""
    if not hasattr(node, "get"):
        return False
    tokens = _listing_class_tokens(node)
    if "pager" in tokens or "pagination" in tokens:
        return True
    # Production listings wrap ul.pagination in nav.mt-3, without a pager class on the nav.
    if getattr(node, "name", None) == "nav" and node.select_one("ul.pagination"):
        return True
    return False


def _excluded_listing_node(node) -> bool:
    if _is_pager_container(node):
        return False
    name = getattr(node, "name", None) or ""
    if name in _EXCLUDED_LISTING_TAGS:
        return True
    if not hasattr(node, "get"):
        return False
    if str(node.get("role") or "").lower() in _EXCLUDED_LISTING_ROLES:
        return True
    if "side-news" in set(node.get("class") or []):
        return True
    return bool(_listing_class_tokens(node) & _EXCLUDED_LISTING_TOKENS)


def _inside_excluded_listing_block(node) -> bool:
    current = node
    while current is not None and getattr(current, "name", None):
        if _excluded_listing_node(current):
            return True
        current = getattr(current, "parent", None)
    return False


def _append_observed_listing_root(roots: list, node) -> None:
    if _inside_excluded_listing_block(node):
        return
    parent = getattr(node, "parent", None)
    while parent is not None and getattr(parent, "name", None):
        if parent in roots:
            return
        parent = getattr(parent, "parent", None)
    roots.append(node)


def classify_listing_scope(soup):
    """Return (roots, strategy).

    Strategies: view-content, taxonomy-views, main-fallback, unverified.
    taxonomy-views is the production VOV4 taxonomy page: .taxonomy-block1
    article cards plus the paged .views-element-container list. .side-news
    stays excluded. There is no whole-document anchor fallback.
    """
    view_roots = []
    for node in soup.select(".view-content"):
        if _inside_excluded_listing_block(node):
            continue
        view_roots.append(node)
    if view_roots:
        return view_roots, "view-content"
    observed = []
    for node in soup.select(".taxonomy-block1, .views-element-container"):
        _append_observed_listing_root(observed, node)
    if observed:
        return observed, "taxonomy-views"
    main_roots = []
    for node in soup.select("main, [role=main]"):
        if _inside_excluded_listing_block(node):
            continue
        main_roots.append(node)
    if main_roots:
        return main_roots, "main-fallback"
    return [], "unverified"


def main_listing_roots(soup):
    """Main listing containers only.

    .view-content remains the first choice. Production VOV4 taxonomy pages
    use .taxonomy-block1 and .views-element-container. main / [role=main]
    is only the fallback when neither of those exists. Sidebar, related,
    footer, and header blocks are never listing evidence.
    """
    roots, _strategy = classify_listing_scope(soup)
    return roots


def _scoped_article_urls(roots, base_url: str, section_slug: str) -> list:
    links = []
    seen = set()
    for root in roots:
        for anchor in root.find_all("a", href=True):
            if _inside_excluded_listing_block(anchor):
                continue
            url = canonicalize(base_url, anchor["href"])
            if not url or not is_bahnar_article(url) or url in seen:
                continue
            if article_section_slug(url) != section_slug:
                continue
            seen.add(url)
            links.append(url)
    return links


def _views_boundary_roots(roots) -> list:
    """Chronological list containers. Featured .taxonomy-block1 cards are not included."""
    found = []
    for root in roots:
        if "views-element-container" in set(root.get("class") or []):
            found.append(root)
            continue
        for node in root.select(".views-element-container"):
            if not _inside_excluded_listing_block(node):
                found.append(node)
    return found


def extract_scoped_listing_links(html_bytes: bytes, base_url: str, section_slug: str) -> dict:
    """Return discovery links and chronological boundary links.

    taxonomy-views discovers .taxonomy-block1 and .views-element-container.
    Only .views-element-container is valid date-boundary evidence.
    """
    if not section_slug:
        raise RuntimeError("section_slug is required; refusing an unscoped listing parse")
    soup = BeautifulSoup(html_bytes, "html.parser")
    roots, strategy = classify_listing_scope(soup)
    if strategy == "unverified":
        return {
            "links": [],
            "boundary_links": [],
            "scope_status": "LISTING_SCOPE_UNVERIFIED",
            "listing_scope_strategy": "unverified",
        }
    links = _scoped_article_urls(roots, base_url, section_slug)
    if strategy == "taxonomy-views":
        boundary_links = _scoped_article_urls(_views_boundary_roots(roots), base_url, section_slug)
    else:
        boundary_links = list(links)
    return {
        "links": links,
        "boundary_links": boundary_links,
        "scope_status": "SCOPED",
        "listing_scope_strategy": strategy,
    }


def extract_section_links_from_html(html_bytes: bytes, base_url: str, section_slug: str) -> list:
    """Article URLs in the main listing whose path section equals section_slug."""
    return extract_scoped_listing_links(html_bytes, base_url, section_slug)["links"]


def href_page_number(href: str):
    """Return the integer Drupal ?page=N value, or None. page=1 does not match page=10."""
    raw = href or ""
    query = urlparse(raw).query
    if not query and "?" in raw:
        query = raw.split("?", 1)[1].split("#", 1)[0]
    for part in query.split("&"):
        if not part.startswith("page="):
            continue
        value = part.split("=", 1)[1]
        if value.isdigit():
            return int(value)
    return None


def _rel_tokens(anchor) -> list:
    rel = anchor.get("rel") or []
    if isinstance(rel, str):
        return rel.lower().split()
    return [str(part).lower() for part in rel]


def _anchor_in_trusted_pager(anchor) -> bool:
    """True when the anchor sits in a pager that is not inside a sidebar/footer/header."""
    if _inside_excluded_listing_block(anchor):
        return False
    current = anchor
    while current is not None and getattr(current, "name", None):
        if _is_pager_container(current):
            return True
        current = getattr(current, "parent", None)
    return False


def listing_has_next_page(html_bytes: bytes, page_number: int) -> bool:
    """True only when a trusted listing pager exposes the next page.

    Arbitrary ``?page=N+1`` anchors and ``rel=next`` links in sidebar, footer,
    or header blocks are not pager evidence.
    """
    soup = BeautifulSoup(html_bytes, "html.parser")
    for anchor in soup.find_all("a", href=True):
        if not _anchor_in_trusted_pager(anchor):
            continue
        if "next" in _rel_tokens(anchor):
            return True
        if href_page_number(anchor.get("href") or "") == page_number + 1:
            return True
        parent = anchor.find_parent("li")
        classes = " ".join((parent.get("class") or []) if parent else []).lower()
        if "pager" in classes and "next" in classes:
            return True
    return False


def listing_has_verified_end_marker(html_bytes: bytes) -> bool:
    """No scoped end-of-list marker is accepted.

    An unscoped div.view-empty can belong to a sidebar. Zero links without a
    next pager stay LISTING_EMPTY_UNVERIFIED.
    """
    return False


def inspect_listing_page(html_bytes: bytes, page_number: int, section_slug: str) -> dict:
    """Classify one listing page without treating a parser miss as the end of the site."""
    scoped = extract_scoped_listing_links(html_bytes, f"{START_URL}/{section_slug}", section_slug)
    has_next = listing_has_next_page(html_bytes, page_number)
    strategy = scoped.get("listing_scope_strategy") or "unverified"
    if scoped["scope_status"] != "SCOPED":
        return {
            "status": "LISTING_SCOPE_UNVERIFIED",
            "links": [],
            "boundary_links": [],
            "has_next": has_next,
            "scope_status": "LISTING_SCOPE_UNVERIFIED",
            "listing_scope_strategy": strategy,
        }
    links = scoped["links"]
    boundary_links = list(scoped.get("boundary_links") or [])
    evidence = boundary_links if strategy == "taxonomy-views" else links
    if evidence and not has_next:
        status = "LISTING_EXHAUSTED"
    elif not evidence and has_next:
        status = "LISTING_PARSE_EMPTY"
    elif not evidence:
        status = "LISTING_EMPTY_UNVERIFIED"
    else:
        status = "LISTING_CONTINUE"
    return {
        "status": status,
        "links": links,
        "boundary_links": boundary_links,
        "has_next": has_next,
        "scope_status": "SCOPED",
        "listing_scope_strategy": strategy,
    }


## 9. Article fetch + media probe per candidate URL (network)

One HTTP GET per article: extracts metadata (title/date/description/`page_sha256`) **and** the
article-scoped media candidates in the same pass (avoids re-fetching the same page for Notebook 08's
media step). Articles whose `published_date` falls outside the active phase window are recorded as
`out_of_window` and skipped (never downloaded), never inserted into the pool, and never treated as a
failure.


In [9]:
ARTICLE_CACHE = {}


def _note_listing_section(result: dict, listing_section: str) -> None:
    """Trust the final article URL's section, not the listing that linked it."""
    if not listing_section or result.get("kind") != "article":
        return
    derived = result["article_row"]["source_section"]
    result.setdefault("discovered_sections", set())
    if derived == listing_section:
        result["discovered_sections"].add(listing_section)


def fetch_article_once(url: str, listing_section: str = "") -> dict:
    """Fetch metadata and article-scoped media at most once per canonical URL."""
    cached = ARTICLE_CACHE.get(url)
    if cached is not None:
        _note_listing_section(cached, listing_section)
        return cached
    try:
        html_bytes, final_url, attempts = fetch_html(url)
    except FetchFailed as exc:
        result = {
            "kind": "fetch_failure",
            "article_url": url,
            "error_type": exc.error_type,
            "error_message": exc.error_message,
            "attempts": exc.attempts,
        }
        ARTICLE_CACHE[url] = result
        return result
    cached_final = ARTICLE_CACHE.get(final_url)
    if cached_final is not None and cached_final.get("kind") == "article":
        ARTICLE_CACHE[url] = cached_final
        _note_listing_section(cached_final, listing_section)
        return cached_final
    derived_section = article_section_slug(final_url)
    if not is_bahnar_article(final_url) or not derived_section:
        result = {
            "kind": "fetch_failure",
            "article_url": final_url or url,
            "source_id": source_id(final_url) if final_url else "",
            "error_type": "NOT_ARTICLE",
            "error_message": f"Final URL is not a Bahnar article: {final_url}",
            "attempts": attempts,
        }
        ARTICLE_CACHE[url] = result
        if final_url:
            ARTICLE_CACHE[final_url] = result
        return result
    soup = BeautifulSoup(html_bytes, "html.parser")
    sid = source_id(final_url)
    published_date, published_date_source = extract_published_date(soup)
    article_row = {
        "source_id": sid,
        "article_url": final_url,
        "article_date": published_date,
        "published_date_source": published_date_source,
        "article_title": extract_title(soup),
        "article_description": extract_description(soup),
        "source_section": derived_section,
        "page_sha256": hashlib.sha256(html_bytes).hexdigest(),
        "phase": "",
        "discovered_at_utc": utc_now(),
    }
    article = find_primary_article(soup, final_url)
    if article is None:
        result = {
            "kind": "article",
            "article_row": article_row,
            "media_candidates": [],
            "article_scope_error": "ERROR_ARTICLE_SCOPE_NOT_FOUND",
            "discovered_sections": set(),
        }
    else:
        result = {
            "kind": "article",
            "article_row": article_row,
            "media_candidates": extract_media(article, final_url),
            "article_scope_error": "",
            "discovered_sections": set(),
        }
    ARTICLE_CACHE[url] = result
    ARTICLE_CACHE[final_url] = result
    ARTICLE_CACHE["id:" + sid] = result
    _note_listing_section(result, listing_section)
    return result

## 10. Media verification + primary-media selection + Notebook 09 pilot-reuse matching

`verify_media_url` is the same safe HEAD-first / ranged-GET-fallback probe as Notebook 08. Verified
results are cached per canonical `media_url` so a media file shared by more than one article (rare, but
possible) is only ever probed once. `pick_primary_media` deterministically chooses at most one usable
public media item per article (audio preferred over video). `match_pilot_media` looks the chosen media's
`media_url_sha256` up in the Notebook 09 pilot index; on a hit it verifies the pilot file is still present
and byte-identical before reusing it (falls through to a fresh download otherwise, never assuming reuse
is valid).


In [10]:
VERIFY_CACHE = {}
VERIFY_ATTEMPTS = 3
VERIFY_BACKOFF_SECONDS = (2, 4, 8)


def _verify_status_is_transient(status_code: int) -> bool:
    return status_code == 429 or 500 <= status_code <= 599


def _is_transient_verify_error(exc: BaseException) -> bool:
    """Retry only 429, 5xx, timeout, and connection failures."""
    return isinstance(exc, (TransientHTTP, requests.exceptions.Timeout, requests.exceptions.ConnectionError))


def _follow_verify_hop(url: str) -> dict:
    current = canonicalize(url, url)
    seen = []
    for _hop in range(MAX_REDIRECTS):
        if not current or current in seen:
            raise RuntimeError(f"Redirect loop or invalid URL: {url}")
        seen.append(current)
        assert_public_http_host(current)
        if robots_decision(current) == "disallow":
            return {"outcome": "protected", "notes": f"robots.txt disallow: {current}", "final_url": current}
        time.sleep(SLEEP_SECONDS)
        response = session.head(current, timeout=20, allow_redirects=False)
        if response.status_code in {405, 501}:
            response.close()
            response = session.get(current, timeout=20, headers={"Range": "bytes=0-0"}, stream=True, allow_redirects=False)
        try:
            if response.status_code in {301, 302, 303, 307, 308}:
                location = response.headers.get("Location") or ""
                if not location.strip():
                    raise RuntimeError(f"Redirect thieu Location: {current}")
                current = canonicalize(current, location.strip())
                continue
            return {
                "outcome": "ok",
                "final_url": current,
                "http_status": str(response.status_code),
                "content_type": (response.headers.get("Content-Type") or "").split(";")[0].strip().lower(),
                "content_length": response.headers.get("Content-Length") or "",
                "notes": "",
            }
        finally:
            response.close()
    raise RuntimeError(f"Qua {MAX_REDIRECTS} redirect hops: {url}")


def verify_media_url(media_url: str) -> dict:
    if media_url in VERIFY_CACHE:
        return VERIFY_CACHE[media_url]
    result = {"probe_status": "ERROR", "is_public_direct_url": False, "http_status": "", "content_type": "", "notes": ""}
    hop_result = None
    for attempt in range(1, VERIFY_ATTEMPTS + 1):
        try:
            hop_result = _follow_verify_hop(media_url)
        except (NonPublicHost, RobotsUnverified) as exc:
            result.update(probe_status="PROTECTED_OR_UNSUPPORTED", notes=str(exc))
            VERIFY_CACHE[media_url] = result
            return result
        except Exception as exc:
            if not _is_transient_verify_error(exc) or attempt >= VERIFY_ATTEMPTS:
                result.update(probe_status="ERROR", notes=f"{type(exc).__name__}: {exc}")
                VERIFY_CACHE[media_url] = result
                return result
            time.sleep(VERIFY_BACKOFF_SECONDS[attempt - 1])
            continue
        if hop_result.get("outcome") == "protected":
            result.update(probe_status="PROTECTED_OR_UNSUPPORTED", notes=hop_result.get("notes", ""))
            VERIFY_CACHE[media_url] = result
            return result
        code_ = int(hop_result.get("http_status") or 0)
        if _verify_status_is_transient(code_) and attempt < VERIFY_ATTEMPTS:
            time.sleep(VERIFY_BACKOFF_SECONDS[attempt - 1])
            continue
        break
    ctype = hop_result.get("content_type", "")
    code_ = int(hop_result.get("http_status") or 0)
    result["http_status"] = hop_result.get("http_status", "")
    result["content_type"] = ctype
    if code_ in {401, 403}:
        result["probe_status"] = "PROTECTED_OR_UNSUPPORTED"
    elif ctype.startswith("text/html"):
        result["probe_status"] = "PROTECTED_OR_UNSUPPORTED"
        result["notes"] = "response is HTML, not a direct media file"
    elif content_type_is_non_media(ctype):
        result["probe_status"] = "NON_MEDIA"
    elif code_ in {200, 206} and content_type_is_media(ctype, ""):
        result["probe_status"] = "MEDIA_FOUND_PUBLIC"
        result["is_public_direct_url"] = True
    else:
        result["probe_status"] = "MEDIA_FOUND_UNVERIFIED"
    VERIFY_CACHE[media_url] = result
    return result


def pick_primary_media(media_candidates: list) -> dict:
    """Deterministic: prefer verified public audio, then verified public video, else None."""
    verified = []
    for media in media_candidates:
        probe = verify_media_url(media["media_url"])
        merged = {**media, **probe}
        apply_content_type(merged, probe.get("content_type", ""))
        verified.append(merged)
    public = [m for m in verified if m["probe_status"] == "MEDIA_FOUND_PUBLIC" and m["is_public_direct_url"]]
    if not public:
        return None
    for preferred_type in ("audio", "video"):
        for media in public:
            if media["media_type"] == preferred_type:
                return media
    return public[0]


def read_wav_pcm(path: Path) -> dict:
    with wave.open(str(path), "rb") as handle:
        if handle.getcomptype() != "NONE":
            raise RuntimeError(f"WAV compression is not PCM: {handle.getcomptype()}")
        info = {
            "sample_rate": handle.getframerate(),
            "channels": handle.getnchannels(),
            "sample_width": handle.getsampwidth(),
            "nframes": handle.getnframes(),
        }
        info["pcm_bytes"] = handle.readframes(info["nframes"])
    info["pcm16_sha256"] = hashlib.sha256(info["pcm_bytes"]).hexdigest()
    return info


def verify_pilot_integrity(pilot_row: dict) -> tuple:
    """Returns (ok, reason, qa_metrics). Does not copy hashes unless every check passes."""
    if str(pilot_row.get("technical_status", "")).strip().upper() != "PASS":
        return False, "pilot technical_status is not PASS", {}
    raw_sha = (pilot_row.get("raw_sha256") or "").strip().lower()
    wav_sha = (pilot_row.get("wav_sha256") or "").strip().lower()
    pcm_sha = (pilot_row.get("pcm16_sha256") or "").strip().lower()
    if not (is_sha256_hex(raw_sha) and is_sha256_hex(wav_sha) and is_sha256_hex(pcm_sha)):
        return False, "pilot hash is not 64-hex", {}
    try:
        duration = float(pilot_row.get("duration_seconds") or 0)
    except (TypeError, ValueError):
        return False, "duration is not numeric", {}
    if not math.isfinite(duration) or not duration > 0:
        return False, "duration is not finite and > 0", {}
    raw_path = resolve_project_path(pilot_row.get("raw_local_path") or "")
    wav_path = resolve_project_path(pilot_row.get("wav_local_path") or "")
    if not raw_path.is_file() or not wav_path.is_file():
        return False, "pilot raw or wav file is missing", {}
    if sha256_file(raw_path) != raw_sha:
        return False, "raw_sha256 does not match file", {}
    if sha256_file(wav_path) != wav_sha:
        return False, "wav_sha256 does not match file", {}
    try:
        wav_info = read_wav_pcm(wav_path)
    except Exception as exc:
        return False, f"wav unreadable: {exc}", {}
    if wav_info["pcm16_sha256"] != pcm_sha:
        return False, "decoded PCM hash does not match pcm16_sha256", {}
    if wav_info["sample_rate"] != 16000 or wav_info["channels"] != 1 or wav_info["sample_width"] != 2:
        return False, "WAV is not 16 kHz mono signed 16-bit", {}
    if wav_info["nframes"] <= 0:
        return False, "WAV has no PCM frames", {}
    qa_row = PILOT_QA_BY_SOURCE_ID.get(pilot_row.get("source_id", ""))
    if not qa_row:
        return False, "Notebook 09 technical_qa.csv has no row for this source_id", {}
    if (qa_row.get("pcm16_sha256") or "").strip().lower() not in ("", pcm_sha):
        return False, "technical_qa pcm16_sha256 disagrees with pilot manifest", {}
    if (qa_row.get("technical_status") or "").strip().upper() not in ("", "PASS"):
        return False, "technical_qa technical_status is not PASS", {}
    qa_metrics = {}
    for field in QA_METRIC_FIELDS:
        value = str(qa_row.get(field, "")).strip()
        if value == "":
            return False, f"technical_qa missing {field}", {}
        qa_metrics[field] = value
    return True, "", qa_metrics


def match_pilot_media(media_url_sha256: str):
    """Reuse a Notebook 09 pilot row only after full raw/wav/PCM/QA integrity checks."""
    pilot_row = PILOT_BY_MEDIA_HASH.get(media_url_sha256)
    if pilot_row is None:
        return None
    ok, reason, qa_metrics = verify_pilot_integrity(pilot_row)
    if not ok:
        print("PILOT_INTEGRITY_FAILED", media_url_sha256[:12], reason)
        return None
    reused = dict(pilot_row)
    reused["_qa_metrics"] = qa_metrics
    return reused


def verify_pass_artifact(record: dict) -> list:
    """Integrity problems for a row that claims technical PASS. Ordinary failures are not passed here."""
    problems = []
    label = record.get("source_id") or record.get("article_url") or "?"
    for field in ("raw_local_path", "wav_local_path", "video_container_local_path"):
        value = str(record.get(field, "") or "")
        if not is_portable_path_value(value):
            problems.append(f"{label}: non-portable {field}")
    for field in ("raw_sha256", "wav_sha256", "pcm16_sha256"):
        if not is_sha256_hex(str(record.get(field, "") or "")):
            problems.append(f"{label}: {field} is not 64-hex")
    try:
        duration = float(record.get("duration_seconds") or 0)
    except (TypeError, ValueError):
        duration = 0.0
    if not math.isfinite(duration) or not duration > 0:
        problems.append(f"{label}: PASS duration is not finite and > 0")
    if problems:
        return problems
    raw_path = resolve_project_path(record["raw_local_path"])
    wav_path = resolve_project_path(record["wav_local_path"])
    if not raw_path.is_file():
        problems.append(f"{label}: raw file missing")
    elif sha256_file(raw_path) != record["raw_sha256"].strip().lower():
        problems.append(f"{label}: stored raw SHA does not match file")
    if not wav_path.is_file():
        problems.append(f"{label}: wav file missing")
        return problems
    if sha256_file(wav_path) != record["wav_sha256"].strip().lower():
        problems.append(f"{label}: stored WAV SHA does not match file")
    try:
        wav_info = read_wav_pcm(wav_path)
    except Exception as exc:
        problems.append(f"{label}: wav unreadable: {exc}")
        return problems
    if wav_info["pcm16_sha256"] != record["pcm16_sha256"].strip().lower():
        problems.append(f"{label}: stored PCM SHA does not match decoded PCM")
    if wav_info["sample_rate"] != 16000 or wav_info["channels"] != 1 or wav_info["sample_width"] != 2:
        problems.append(f"{label}: PASS wav is not 16 kHz mono signed 16-bit")
    video_rel = str(record.get("video_container_local_path") or "")
    video_sha = str(record.get("video_container_sha256") or "").strip().lower()
    if not video_rel:
        if video_sha:
            problems.append(f"{label}: video_container_sha256 set without a container path")
        return problems
    if not is_sha256_hex(video_sha):
        problems.append(f"{label}: video_container_sha256 is not 64-hex")
        return problems
    video_path = resolve_project_path(video_rel)
    if not video_path.is_file():
        problems.append(f"{label}: video container missing")
        return problems
    if sha256_file(video_path) != video_sha:
        problems.append(f"{label}: stored video container SHA does not match file")
    try:
        expected_bytes = int(record.get("video_container_bytes"))
    except (TypeError, ValueError):
        expected_bytes = -1
    if video_path.stat().st_size != expected_bytes:
        problems.append(f"{label}: stored video container byte size does not match file")
    return problems


## 11. Download new public audio, or download + extract audio from new public video

Only reached for `artifact_origin = "nb10_full"` (no verified Notebook 09 pilot match). The audio
extension comes from a known URL extension or from the verified MIME type. An unknown extension/MIME
fails with `UNSUPPORTED_AUDIO_EXTENSION_OR_MIME` and is never defaulted to `.mp3`. MP4 candidates are
streamed, then ffmpeg extracts the audio track (`-c:a copy`, then AAC if copy fails). The extracted
audio is what QA measures. The video container is kept only as provenance.

In [11]:
def run_cmd(args) -> tuple:
    completed = subprocess.run(args, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return completed.returncode, completed.stdout, completed.stderr


def stem_for(source_id_: str, media_url: str) -> str:
    short_hash = hashlib.sha256(media_url.encode("utf-8")).hexdigest()[:12]
    return f"{source_id_}__{short_hash}"


def resolve_audio_extension(media_url: str, content_type: str = "") -> str:
    """Known URL extension, else verified audio MIME. Never default an unknown type to mp3."""
    ext = media_extension(media_url)
    if ext in AUDIO_EXTENSIONS:
        return ext
    mime = (content_type or "").split(";")[0].strip().lower()
    mapped = MIME_TO_AUDIO_EXT.get(mime, "")
    if mapped in AUDIO_EXTENSIONS:
        return mapped
    raise RuntimeError("UNSUPPORTED_AUDIO_EXTENSION_OR_MIME")


def resolve_video_extension(media_url: str, content_type: str = "") -> str:
    ext = media_extension(media_url)
    mime = (content_type or "").split(";")[0].strip().lower()
    if ext == "mp4" or MIME_TO_VIDEO_EXT.get(mime) == "mp4":
        return "mp4"
    raise RuntimeError("UNSUPPORTED_AUDIO_EXTENSION_OR_MIME")


def _is_transient_download_error(exc: BaseException) -> bool:
    if isinstance(exc, TotalDownloadBudgetExceeded):
        return False
    return isinstance(exc, (TransientHTTP, requests.exceptions.Timeout, requests.exceptions.ConnectionError))


def stream_media_with_retries(url: str, dest_part: Path) -> dict:
    """Retry timeout, connection, 429, and 5xx. Delete the partial file before every attempt."""
    last_error = None
    for attempt in range(1, FETCH_ATTEMPTS + 1):
        if dest_part.exists():
            dest_part.unlink()
        try:
            return stream_media_to_file(url, dest_part)
        except TotalDownloadBudgetExceeded:
            if dest_part.exists():
                dest_part.unlink()
            raise
        except Exception as exc:
            if dest_part.exists():
                dest_part.unlink()
            last_error = exc
            if not _is_transient_download_error(exc) or attempt >= FETCH_ATTEMPTS:
                raise
            delay = RETRY_BACKOFF_SECONDS[attempt - 1]
            print(f"RETRY download {attempt}/{FETCH_ATTEMPTS - 1} in {delay}s: {type(exc).__name__} {url}")
            time.sleep(delay)
    raise last_error


def download_new_media(source_id_: str, media_url: str, media_type: str, media_extension_: str, content_type: str = "") -> dict:
    """Network + local ffmpeg. Returns download/extraction metadata; raises on unrecoverable failure."""
    stem = stem_for(source_id_, media_url)
    if media_type == "audio":
        ext = resolve_audio_extension(media_url, content_type)
        dest = RAW_DIR / f"{stem}.{ext}"
        part = dest.with_suffix(dest.suffix + ".part")
        meta = stream_media_with_retries(media_url, part)
        os.replace(part, dest)
        return {
            "raw_local_path": to_project_relative(dest),
            "video_container_local_path": "",
            "video_container_sha256": "",
            "video_container_bytes": "",
            **meta,
        }
    if media_type == "video":
        resolve_video_extension(media_url, content_type)
        video_dest = RAW_DIR / f"{stem}.mp4"
        part = video_dest.with_suffix(video_dest.suffix + ".part")
        meta = stream_media_with_retries(media_url, part)
        os.replace(part, video_dest)
        audio_dest = RAW_DIR / f"{stem}.m4a"
        rc, _out, err = run_cmd([FFMPEG, "-y", "-i", str(video_dest), "-vn", "-acodec", "copy", str(audio_dest)])
        if rc != 0 or not audio_dest.is_file() or audio_dest.stat().st_size == 0:
            rc2, _out2, err2 = run_cmd([FFMPEG, "-y", "-i", str(video_dest), "-vn", "-acodec", "aac", "-b:a", "192k", str(audio_dest)])
            if rc2 != 0 or not audio_dest.is_file() or audio_dest.stat().st_size == 0:
                if audio_dest.exists():
                    audio_dest.unlink()
                if video_dest.exists():
                    video_dest.unlink()
                raise RuntimeError(f"ffmpeg audio extraction failed for {video_dest.name}: {err.decode(errors='replace')[-500:]} / {err2.decode(errors='replace')[-500:]}")
        return {
            "raw_local_path": to_project_relative(audio_dest),
            "video_container_local_path": to_project_relative(video_dest),
            "video_container_sha256": sha256_file(video_dest),
            "video_container_bytes": video_dest.stat().st_size,
            **meta,
        }
    raise RuntimeError(f"Unsupported media_type for download: {media_type}")


## 12. ffprobe + full decode validation + WAV16k conversion (reused from Notebook 09)

Applied uniformly whether the raw audio came from a downloaded MP3, a Notebook 09 pilot reuse, or an
MP4-extracted `.m4a`. `wav_sha256` hashes the WAV *container* bytes; `pcm16_sha256` hashes only the decoded
PCM samples (via `wave.readframes`), matching Notebook 09's exact scheme so hashes are directly comparable.


In [12]:
def ffprobe_audio(path: Path) -> dict:
    rc, out, err = run_cmd([
        FFPROBE, "-v", "error", "-print_format", "json",
        "-show_format", "-show_streams", str(path),
    ])
    if rc != 0:
        raise RuntimeError(f"ffprobe failed for {path.name}: {err.decode(errors='replace')[-500:]}")
    payload = json.loads(out.decode("utf-8", errors="replace"))
    streams = [s for s in payload.get("streams", []) if s.get("codec_type") == "audio"]
    if not streams:
        raise RuntimeError(f"No audio stream in {path.name}")
    stream = streams[0]
    fmt = payload.get("format", {})
    duration = float(fmt.get("duration") or stream.get("duration") or 0.0)
    if not (duration > 0.0) or not np.isfinite(duration):
        raise RuntimeError(f"Invalid duration for {path.name}: {duration}")
    return {
        "duration_seconds": duration,
        "codec_name": stream.get("codec_name", ""),
        "format_name": fmt.get("format_name", ""),
        "sample_rate": int(stream.get("sample_rate") or 0),
        "channels": int(stream.get("channels") or 0),
        "channel_layout": stream.get("channel_layout", ""),
        "bit_rate": stream.get("bit_rate") or fmt.get("bit_rate") or "",
    }


def decode_audio(path: Path) -> None:
    rc, _out, err = run_cmd([FFMPEG, "-v", "error", "-i", str(path), "-f", "null", "-"])
    if rc != 0:
        raise RuntimeError(f"Full decode failed for {path.name}: {err.decode(errors='replace')[-500:]}")


def convert_wav(src: Path, dest: Path) -> None:
    rc, _out, err = run_cmd([
        FFMPEG, "-y", "-i", str(src), "-vn", "-ac", "1", "-ar", "16000",
        "-acodec", "pcm_s16le", str(dest),
    ])
    if rc != 0:
        raise RuntimeError(f"WAV conversion failed for {src.name}: {err.decode(errors='replace')[-500:]}")


def inspect_wav(path: Path) -> dict:
    ffprobe_info = ffprobe_audio(path)
    if ffprobe_info["sample_rate"] != 16000 or ffprobe_info["channels"] != 1:
        raise RuntimeError(f"WAV_INVALID sample_rate/channels for {path.name}: {ffprobe_info}")
    with wave.open(str(path), "rb") as handle:
        if handle.getsampwidth() != 2:
            raise RuntimeError(f"WAV_INVALID sample width for {path.name}")
        pcm_bytes = handle.readframes(handle.getnframes())
    return {
        "wav_sha256": sha256_file(path),
        "pcm16_sha256": hashlib.sha256(pcm_bytes).hexdigest(),
        "pcm_bytes": pcm_bytes,
        "duration_seconds": ffprobe_info["duration_seconds"],
    }


## 13. Technical QA metrics (reused from Notebook 09)

`peak_dbfs` / `rms_dbfs` / `clipping_fraction` / `silence_fraction_energy` — never named or treated as a
speech ratio, VAD, or language probability. Only `peak_abs <= 0` or `silence_fraction_energy >= 1.0` fails
as `SILENT`.


In [13]:
def dbfs(value: float) -> float:
    return 20.0 * np.log10(value) if value > 0 else float("-inf")


def qa_wav(pcm_bytes: bytes) -> dict:
    samples = np.frombuffer(pcm_bytes, dtype="<i2").astype(np.float64) / 32768.0
    if samples.size == 0:
        raise RuntimeError("Empty PCM payload")
    peak_abs = float(np.max(np.abs(samples)))
    rms = float(np.sqrt(np.mean(np.square(samples))))
    clipping_fraction = float(np.mean(np.abs(samples) >= 0.999))
    frame_len = int(16000 * 0.020)
    n_frames = samples.size // frame_len
    if n_frames == 0:
        silence_fraction_energy = 1.0 if peak_abs <= 0 else 0.0
    else:
        frames = samples[: n_frames * frame_len].reshape(n_frames, frame_len)
        frame_rms = np.sqrt(np.mean(np.square(frames), axis=1))
        frame_dbfs = np.full(frame_rms.shape, -np.inf, dtype=np.float64)
        audible = frame_rms > 0
        if np.any(audible):
            frame_dbfs[audible] = 20.0 * np.log10(frame_rms[audible])
        silence_fraction_energy = float(np.mean(frame_dbfs < -50.0))
    if peak_abs <= 0 or silence_fraction_energy >= 1.0:
        raise RuntimeError("SILENT")
    return {
        "peak_abs": peak_abs,
        "peak_dbfs": dbfs(peak_abs),
        "rms": rms,
        "rms_dbfs": dbfs(rms),
        "clipping_fraction": clipping_fraction,
        "silence_fraction_energy": silence_fraction_energy,
    }


## 14. Metadata-language hint (never a gold label) + source-level eligibility rules

`metadata_language_hint` only ever looks at the article title and the unquoted media filename tokens; it
is `bahnar_likely`, `non_bahnar_likely`, or `unknown`, and it is **never** used as ground truth. It is one
conservative input to `derive_source_level_status`, alongside `content_class` (from the section map) and
technical QA outcome. Allowed `source_level_status`: `ELIGIBLE_SOURCE`, `REJECTED_SOURCE`, `NEEDS_REVIEW`.


In [14]:
# Folded phrases. Matching is exact token/phrase sequence, never a raw substring.
# No bare "kho" or "phap": those match ordinary Vietnamese words after accent folding.
OTHER_LANGUAGE_PHRASES = (
    "ede", "e de", "jrai", "gia rai", "k ho", "ko ho", "co ho",
    "chu ru", "churu", "raglai", "ra glai", "stieng", "s tieng", "mnong", "m nong",
    "tieng viet", "vietnamese", "tieng phap", "tieng anh", "english",
)
BAHNAR_PHRASES = ("bahnar", "bana", "ba na", "bohnar")


def fold_language_text(text: str) -> str:
    """unquote, NFKD, đ->d, strip accents, lower, collapse punctuation to spaces."""
    raw = unquote(text or "")
    raw = raw.replace("đ", "d").replace("Đ", "D")
    raw = unicodedata.normalize("NFKD", raw)
    raw = "".join(ch for ch in raw if not unicodedata.combining(ch))
    raw = raw.lower().replace("đ", "d")
    raw = re.sub(r"[^a-z0-9]+", " ", raw)
    return re.sub(r"\s+", " ", raw).strip()


def language_tokens(text: str) -> list:
    folded = fold_language_text(text)
    return folded.split() if folded else []


def phrase_in_tokens(tokens: list, phrase: str) -> bool:
    parts = phrase.split()
    width = len(parts)
    if width == 0 or len(tokens) < width:
        return False
    for index in range(len(tokens) - width + 1):
        if tokens[index:index + width] == parts:
            return True
    return False


def metadata_hint(title: str, media_url: str) -> tuple:
    """Returns (hint, conflicting_language_evidence). Hint is never a gold language label."""
    filename = unquote(urlparse(media_url).path.rsplit("/", 1)[-1])
    tokens = language_tokens(title) + language_tokens(filename)
    has_other = any(phrase_in_tokens(tokens, phrase) for phrase in OTHER_LANGUAGE_PHRASES)
    has_bahnar = any(phrase_in_tokens(tokens, phrase) for phrase in BAHNAR_PHRASES)
    if has_other:
        return "non_bahnar_likely", True
    if has_bahnar:
        return "bahnar_likely", False
    return "unknown", False


ALLOWED_SOURCE_LEVEL_STATUS = {"ELIGIBLE_SOURCE", "REJECTED_SOURCE", "NEEDS_REVIEW"}


def derive_source_level_status(*, technical_status: str, content_class: str, conflicting_language_evidence: bool,
                                metadata_hint_value: str, article_title: str) -> tuple:
    """Pure. Returns (source_level_status, exclusion_reason)."""
    if technical_status != "PASS":
        return "REJECTED_SOURCE", "TECHNICAL_FAIL"
    if content_class == "music":
        return "REJECTED_SOURCE", "MUSIC_CONTENT"
    if conflicting_language_evidence:
        return "REJECTED_SOURCE", "CONFLICTING_LANGUAGE_EVIDENCE"
    if content_class in AMBIGUOUS_NEEDS_REVIEW_CLASSES:
        return "NEEDS_REVIEW", f"AMBIGUOUS_CONTENT_CLASS:{content_class}"
    if content_class in SPEECH_ELIGIBLE_CLASSES:
        if not (article_title or "").strip():
            return "NEEDS_REVIEW", "METADATA_UNCLEAR"
        return "ELIGIBLE_SOURCE", ""
    return "NEEDS_REVIEW", f"UNMAPPED_CONTENT_CLASS:{content_class}"


## 15. Native speaker spot-check metadata (supporting evidence only)

Optional file: `artifacts/rq2/vov4_full/native_spotcheck.csv`, keyed by `source_id` + `media_url_sha256`.
Columns: `native_spotcheck_status`, `native_spotcheck_notes`, `reviewer`. A missing file or a missing row
is `NOT_PERFORMED`. These fields are never read by `derive_source_level_status`.

The documented spot-check (three pilot sources, predominantly Bahnar, subjective confidence about
80–90%, uncertainty from regional/dialectal variation) is methodology context only. It is not copied
onto every pilot row and Notebook 09 artifacts are not modified to store it.

In [15]:
NATIVE_SPOTCHECK_PATH = OUT_DIR / "native_spotcheck.csv"


def load_native_spotcheck(path: Path = NATIVE_SPOTCHECK_PATH) -> dict:
    """Map (source_id, media_url_sha256) -> row. Missing file is an empty map, not an error."""
    if not path.is_file():
        return {}
    frame = pd.read_csv(path, dtype=str).fillna("")
    for column in NATIVE_SPOTCHECK_COLUMNS:
        if column not in frame.columns:
            raise RuntimeError(f"native_spotcheck.csv missing column {column}")
    keyed = {}
    for row in frame.to_dict(orient="records"):
        key = (row["source_id"].strip(), row["media_url_sha256"].strip().lower())
        if not key[0] or not key[1]:
            continue
        if key in keyed:
            raise RuntimeError(f"native_spotcheck.csv duplicate key {key[0]}")
        keyed[key] = row
    return keyed


NATIVE_SPOTCHECK_BY_KEY = load_native_spotcheck()


def apply_native_spotcheck(record: dict) -> None:
    key = (str(record.get("source_id", "")).strip(), str(record.get("media_url_sha256", "")).strip().lower())
    row = NATIVE_SPOTCHECK_BY_KEY.get(key)
    if not row:
        record["native_spotcheck_status"] = "NOT_PERFORMED"
        record["native_spotcheck_notes"] = ""
        return
    record["native_spotcheck_status"] = (row.get("native_spotcheck_status") or "").strip() or "NOT_PERFORMED"
    record["native_spotcheck_notes"] = row.get("native_spotcheck_notes", "") or ""

## 16. Per-article pipeline orchestrator (network + local ffmpeg; ties Sections 6-15 together)

`run_article_through_pipeline` handles exactly one already-fetched candidate article result: picks the
primary public media (if any), reuses the Notebook 09 pilot artifact when the hash matches, otherwise
downloads new audio (or downloads+extracts video audio), runs ffprobe/decode/WAV/QA, classifies
eligibility, and attaches the native spot-check note. It never raises for ordinary per-article problems —
those are captured as `technical_status` / `source_level_status` / `exclusion_reason` on the returned
record (or as an entry in `exclusions`) so nothing is silently dropped.


In [16]:
def _blank_qa(record: dict) -> None:
    for field in QA_METRIC_FIELDS:
        record.setdefault(field, "")


def run_article_through_pipeline(article_row: dict, media_candidates: list, article_scope_error: str) -> tuple:
    """Returns (record_or_None, exclusion_row_or_None). Exactly one of them is not None."""
    base_exclusion = {
        "source_id": article_row["source_id"],
        "article_url": article_row["article_url"],
        "article_date": article_row["article_date"],
        "article_title": article_row["article_title"],
        "source_section": article_row["source_section"],
        "phase": article_row["phase"],
        "canonical_source_id": "",
        "media_url_sha256": "",
    }
    if article_scope_error:
        return None, {**base_exclusion, "stage": "media_probe", "exclusion_reason": article_scope_error}
    primary = pick_primary_media(media_candidates)
    if primary is None:
        return None, {**base_exclusion, "stage": "media_probe", "exclusion_reason": "NO_PUBLIC_MEDIA"}

    media_url = primary["media_url"]
    media_url_sha256 = hashlib.sha256(media_url.encode("utf-8")).hexdigest()
    pilot_row = match_pilot_media(media_url_sha256)

    record = {
        "source_id": article_row["source_id"],
        "article_url": article_row["article_url"],
        "article_date": article_row["article_date"],
        "published_date_source": article_row["published_date_source"],
        "article_title": article_row["article_title"],
        "phase": article_row["phase"],
        "source_section": article_row["source_section"],
        "content_class": section_content_class(article_row["source_section"]),
        "media_url": media_url,
        "media_url_sha256": media_url_sha256,
        "media_type": primary["media_type"],
        "media_extension": primary["media_extension"],
        "http_status": primary.get("http_status", ""),
        "content_type": primary.get("content_type", ""),
        "video_container_local_path": "",
        "video_container_sha256": "",
        "video_container_bytes": "",
        "download_attempted": False,
        "download_success": False,
    }
    _blank_qa(record)

    try:
        if pilot_row is not None:
            record["artifact_origin"] = "nb09_pilot"
            record["raw_local_path"] = pilot_row["raw_local_path"]
            record["wav_local_path"] = pilot_row["wav_local_path"]
            record["raw_sha256"] = pilot_row["raw_sha256"]
            record["wav_sha256"] = pilot_row["wav_sha256"]
            record["pcm16_sha256"] = pilot_row["pcm16_sha256"]
            record["duration_seconds"] = float(pilot_row["duration_seconds"])
            record["sample_rate"] = 16000
            record["channels"] = 1
            record["codec_name"] = pilot_row.get("codec_name", "")
            record["technical_status"] = "PASS"
            record["technical_notes"] = "reused from Notebook 09 after raw/wav/PCM integrity verification"
            for field, value in pilot_row.get("_qa_metrics", {}).items():
                record[field] = value
            raw_local_path = resolve_project_path(pilot_row["raw_local_path"])
            NETWORK_STATE["reused_pilot_bytes"] += raw_local_path.stat().st_size
        else:
            record["artifact_origin"] = "nb10_full"
            record["download_attempted"] = True
            download_meta = download_new_media(
                article_row["source_id"],
                media_url,
                primary["media_type"],
                primary["media_extension"],
                primary.get("content_type", ""),
            )
            record["download_success"] = True
            record["raw_local_path"] = download_meta["raw_local_path"]
            record["video_container_local_path"] = download_meta.get("video_container_local_path", "")
            record["video_container_sha256"] = download_meta.get("video_container_sha256", "")
            record["video_container_bytes"] = download_meta.get("video_container_bytes", "")
            record["http_status"] = download_meta.get("http_status", record["http_status"])
            record["content_type"] = download_meta.get("content_type", record["content_type"])
            raw_path = resolve_project_path(record["raw_local_path"])
            record["raw_sha256"] = sha256_file(raw_path)
            probe_info = ffprobe_audio(raw_path)
            decode_audio(raw_path)
            wav_path = WAV_DIR / f"{stem_for(article_row['source_id'], media_url)}.wav"
            convert_wav(raw_path, wav_path)
            wav_info = inspect_wav(wav_path)
            qa_metrics = qa_wav(wav_info["pcm_bytes"])
            record["wav_local_path"] = to_project_relative(wav_path)
            record["wav_sha256"] = wav_info["wav_sha256"]
            record["pcm16_sha256"] = wav_info["pcm16_sha256"]
            record["duration_seconds"] = wav_info["duration_seconds"]
            record["sample_rate"] = 16000
            record["channels"] = 1
            record["codec_name"] = probe_info["codec_name"]
            record["technical_status"] = "PASS"
            record["technical_notes"] = "nb10_full technical QA"
            for field in QA_METRIC_FIELDS:
                record[field] = qa_metrics[field]
    except TotalDownloadBudgetExceeded:
        raise
    except Exception as exc:
        record["artifact_origin"] = record.get("artifact_origin", "nb10_full")
        message = str(exc)
        if "UNSUPPORTED_AUDIO_EXTENSION_OR_MIME" in message:
            record["technical_status"] = "UNSUPPORTED_AUDIO_EXTENSION_OR_MIME"
        elif message == "SILENT":
            record["technical_status"] = "SILENT"
        else:
            record["technical_status"] = "TECHNICAL_FAIL"
        record["technical_notes"] = message
        record["raw_local_path"] = record.get("raw_local_path", "")
        record["video_container_local_path"] = record.get("video_container_local_path", "")
        record["wav_local_path"] = record.get("wav_local_path", "")
        record["raw_sha256"] = record.get("raw_sha256", "")
        record["wav_sha256"] = record.get("wav_sha256", "")
        record["pcm16_sha256"] = record.get("pcm16_sha256", "")
        record["duration_seconds"] = record.get("duration_seconds", 0.0)
        record["sample_rate"] = record.get("sample_rate", 0)
        record["channels"] = record.get("channels", 0)
        record["codec_name"] = record.get("codec_name", "")
        _blank_qa(record)

    hint_value, review_required = metadata_hint(article_row["article_title"], media_url)
    record["metadata_language_hint"] = hint_value
    record["conflicting_language_evidence"] = bool(review_required)
    record["verification_method"] = "automated_source_provenance_v1"
    status, reason = derive_source_level_status(
        technical_status=record["technical_status"],
        content_class=record["content_class"],
        conflicting_language_evidence=record["conflicting_language_evidence"],
        metadata_hint_value=hint_value,
        article_title=article_row["article_title"],
    )
    record["source_level_status"] = status
    record["exclusion_reason"] = reason
    record["verification_notes"] = (
        f"content_class={record['content_class']}; technical_status={record['technical_status']}; "
        f"rule_reason={reason or 'n/a'}"
    )
    apply_native_spotcheck(record)
    return record, None


## 17. Date-aware crawl, one fetch per article, then Phase A / Phase B

This is the only network entry point (`RUN_NETWORK_PIPELINE`).

- Listing links are section-scoped. After fetch, `source_section` is taken from the final canonical URL.
- Each canonical article URL is fetched at most once and stored in `ARTICLE_CACHE`.
- Listings are treated as newest-first. A page increments the older-page streak only when every scoped
  article on that page was fetched and every one has a trusted `metadata` or `time` date before the
  window start. A body-text date or a fetch failure resets the streak. Body text is `UNTRUSTED_DATE`.
  The trusted publication time is the date inside the article's `div.article-date`,
  `div.time-published`, or `div.video__time` node, recorded as source `time`.
  A date outside that node, including the site license line, stays `body_fallback`.
- A section is exhausted only when it still has article links and the pager has no next page.
  Zero links with a next link is `LISTING_PARSE_EMPTY`. Zero links with no next pager is
  `LISTING_EMPTY_UNVERIFIED`, including a page that only has an unrelated `div.view-empty`.
  Both failures keep the cursor on that page and retry it on a later session, up to three
  attempts. A spent retry budget stays coverage-incomplete and does not continue at page N+1.
- A checkpoint loads only when its compatibility signature matches this notebook's upstream fingerprints,
  date windows, section map, media types, eligibility schema, and file cap. Crawl schema 1.1 resumes in place.
  A 1.0 checkpoint migrates to 1.1 by resetting listing cursors to page 0 while keeping verified media reusable.
  Any other crawl-schema drift fails closed. `MAX_PAGES_PER_SECTION` may only increase on a same-version resume; a decrease fails closed. Sections that stopped on the old cap continue from the saved `next_page`.
  Raising `MAX_TOTAL_DOWNLOAD_BYTES` above the saved exhausted ceiling does not discard finished
  downloads. The source that was mid-download stays retryable. A budget that is still at or below the
  bytes already streamed fails closed instead of crawling again.
- Consecutive pages whose trusted dates are entirely older than the window start still stop the walk.
  The crawl does not stop early just because an hour target was reached inside the window.
- Phase A (2026-01-01 through 2026-09-24) is crawled and processed first. Eligible hours are computed
  only after that window is finished. Phase B (2025) resumes pagination from the saved page cursor and
  reuses cached articles; it does not refetch Phase A URLs.
- Crawl state, the article cache, owned media, and byte counters are checkpointed after each listing
  page and each processed source. A later run reloads that checkpoint unless `RESET_CHECKPOINT` is True.
  Restored PASS rows are hash-checked before their `source_id` is trusted; a missing or corrupt file is
  released and may be downloaded again.
- Global dedup is by canonical `article_url`, `source_id`, and `media_url_sha256`. Within one phase the
  owner is the smallest `source_id`. Across phases, Phase A keeps a hash even if Phase B later sees a
  smaller `source_id`. Later articles are `DUPLICATE_MEDIA` exclusions and are not downloaded.
- Exceeding `MAX_TOTAL_DOWNLOAD_BYTES` is systemic: the run stops and the summary is PARTIAL.
- If `MAX_PAGES_PER_SECTION` is hit before the requested window is covered, SUCCESS fails closed. Raising the cap on a later resume reopens only those capped sections and does not restart at page 0.


In [17]:
SECTION_CRAWL_STATE = {}
OWNED_MEDIA = {}
SECTION_STAT_FIELDS = (
    "pages_fetched", "oldest_date_seen", "newest_date_seen",
    "pagination_cap_reached", "date_boundary_reached", "listing_exhausted",
    "listing_parse_empty", "listing_parse_empty_page", "listing_parse_empty_attempts",
    "listing_empty_unverified", "listing_empty_unverified_page", "listing_empty_unverified_attempts",
    "listing_scope_unverified", "listing_scope_unverified_page", "listing_scope_unverified_attempts",
    "listing_fetch_error",
    "older_boundary_streak", "last_boundary_reason", "last_page_number",
    "last_page_scoped_article_count", "last_page_trusted_date_count", "last_page_untrusted_date_count",
    "last_page_min_trusted_date", "last_page_max_trusted_date",
    "last_page_all_trusted_dates_older_than_window", "last_page_listing_scope_strategy",
    "recent_listing_pages", "taxonomy_boundary_repair_applied",
)
TRUSTED_DATE_SOURCES = frozenset({"metadata", "time"})
CHECKPOINT_SCHEMA_VERSION = 1
ELIGIBILITY_SCHEMA_VERSION = "1.0"
PREVIOUS_CRAWL_PIPELINE_SCHEMA_VERSION = "1.0"
CRAWL_PIPELINE_SCHEMA_VERSION = "1.1"
LISTING_PARSE_EMPTY_MAX_ATTEMPTS = 3
CHECKPOINT_DIR = OUT_DIR / "checkpoint"
CHECKPOINT_PATH = CHECKPOINT_DIR / "state.json"
RESET_CHECKPOINT = False
RUN_STATE = {"total_budget_exhausted": False, "processed_source_ids": set(), "reusable_media_by_hash": {}}
ACCUMULATED_PHASES = []
LISTING_PARSE_RETRIED_THIS_PROCESS = set()
LISTING_EMPTY_RETRIED_THIS_PROCESS = set()
LISTING_SCOPE_RETRIED_THIS_PROCESS = set()
LISTING_DIAGNOSTIC_HISTORY = 10


class CheckpointIncompatible(RuntimeError):
    """Saved crawl state was produced under a different notebook configuration."""


def blank_section_state() -> dict:
    return {
        "next_page": 0,
        "pages_fetched": 0,
        "oldest_date_seen": "",
        "newest_date_seen": "",
        "pagination_cap_reached": False,
        "date_boundary_reached": False,
        "listing_exhausted": False,
        "listing_parse_empty": False,
        "listing_parse_empty_page": None,
        "listing_parse_empty_attempts": 0,
        "listing_empty_unverified": False,
        "listing_empty_unverified_page": None,
        "listing_empty_unverified_attempts": 0,
        "listing_scope_unverified": False,
        "listing_scope_unverified_page": None,
        "listing_scope_unverified_attempts": 0,
        "listing_fetch_error": "",
        "older_boundary_streak": 0,
        "last_boundary_reason": "",
        "last_page_number": None,
        "last_page_scoped_article_count": 0,
        "last_page_trusted_date_count": 0,
        "last_page_untrusted_date_count": 0,
        "last_page_min_trusted_date": "",
        "last_page_max_trusted_date": "",
        "last_page_all_trusted_dates_older_than_window": False,
        "last_page_listing_scope_strategy": "",
        "recent_listing_pages": [],
        "taxonomy_boundary_repair_applied": False,
    }


def note_observed_date(state: dict, date_str: str) -> None:
    if not date_str:
        return
    if not state["oldest_date_seen"] or date_str < state["oldest_date_seen"]:
        state["oldest_date_seen"] = date_str
    if not state["newest_date_seen"] or date_str > state["newest_date_seen"]:
        state["newest_date_seen"] = date_str


def section_coverage_incomplete(stats: dict) -> bool:
    """True when the requested date window was not fully covered for this section."""
    if stats.get("listing_parse_empty") or stats.get("listing_empty_unverified") or stats.get("listing_scope_unverified"):
        return True
    if stats.get("listing_exhausted") or stats.get("date_boundary_reached"):
        return False
    return True


def incomplete_section_slugs() -> list:
    """Sections whose Phase A date window is not finished."""
    stats = section_stats_snapshot()
    return [
        slug
        for slug in ALL_SECTION_SLUGS
        if not stats.get(slug) or section_coverage_incomplete(stats[slug])
    ]


def phase_b_allowed(eligible_source_hours: float) -> bool:
    """Phase B is allowed only after every section has finished Phase A."""
    if incomplete_section_slugs():
        return False
    return float(eligible_source_hours) < float(PHASE_B_TRIGGER_HOURS)


def trusted_observed_dates(observations) -> list:
    """Keep only metadata/time dates. Body-text dates are not a pagination signal."""
    dates = []
    for date_str, source in observations:
        if source in TRUSTED_DATE_SOURCES and (date_str or "").strip():
            dates.append(date_str.strip())
    return dates


def page_older_than_window(observations, window_start: str) -> bool:
    """True only when the page has trusted dates and every one of them is before window_start."""
    trusted = trusted_observed_dates(observations)
    if not trusted:
        return False
    return all(date_str < window_start for date_str in trusted)


def page_proves_older_boundary(n_scoped_links, observations, n_article_fetch_failures, window_start: str) -> bool:
    """A page proves the date boundary only when every scoped URL was fetched and is trusted-old."""
    if n_scoped_links <= 0 or n_article_fetch_failures != 0 or len(observations) != n_scoped_links:
        return False
    for observed in observations:
        date_str = (observed[0] or "").strip()
        source = observed[1]
        if source not in TRUSTED_DATE_SOURCES or not date_str or not date_str < window_start:
            return False
    return True


def note_listing_parse_empty(state: dict, page_number: int) -> None:
    """Remember the page that returned no articles while a next link still existed."""
    clear_listing_empty_unverified(state)
    state["listing_parse_empty"] = True
    state["listing_parse_empty_page"] = int(page_number)
    state["listing_parse_empty_attempts"] = int(state.get("listing_parse_empty_attempts") or 0) + 1
    state["listing_exhausted"] = False
    state["date_boundary_reached"] = False
    state["next_page"] = int(page_number)


def clear_listing_parse_empty(state: dict) -> None:
    state["listing_parse_empty"] = False
    state["listing_parse_empty_page"] = None
    state["listing_parse_empty_attempts"] = 0


def prepare_listing_parse_empty_retry(state: dict) -> str:
    """'retry' rewinds to the failed page. 'incomplete' means the retry budget is spent."""
    if not state.get("listing_parse_empty"):
        return "ok"
    attempts = int(state.get("listing_parse_empty_attempts") or 0)
    if attempts >= LISTING_PARSE_EMPTY_MAX_ATTEMPTS:
        return "incomplete"
    page = state.get("listing_parse_empty_page")
    if page is None:
        page = max(0, int(state.get("next_page") or 0) - 1)
    state["next_page"] = int(page)
    return "retry"


def clear_listing_empty_unverified(state: dict) -> None:
    state["listing_empty_unverified"] = False
    state["listing_empty_unverified_page"] = None
    state["listing_empty_unverified_attempts"] = 0


def note_listing_empty_unverified(state: dict, page_number: int) -> None:
    """Keep the cursor on the unverified page. Do not leave it on page N+1."""
    clear_listing_parse_empty(state)
    state["listing_empty_unverified"] = True
    state["listing_empty_unverified_page"] = int(page_number)
    state["listing_empty_unverified_attempts"] = int(state.get("listing_empty_unverified_attempts") or 0) + 1
    state["listing_exhausted"] = False
    state["date_boundary_reached"] = False
    state["next_page"] = int(page_number)


def prepare_listing_empty_unverified_retry(state: dict) -> str:
    """Retry the same page. A spent budget stays incomplete and does not advance."""
    if not state.get("listing_empty_unverified"):
        return "ok"
    page = state.get("listing_empty_unverified_page")
    if page is None:
        page = max(0, int(state.get("next_page") or 0) - 1)
    state["next_page"] = int(page)
    attempts = int(state.get("listing_empty_unverified_attempts") or 0)
    if attempts >= LISTING_PARSE_EMPTY_MAX_ATTEMPTS:
        return "incomplete"
    return "retry"



def note_listing_scope_unverified(state: dict, page_number: int) -> None:
    """Keep the cursor on the page whose main listing container could not be proven."""
    clear_listing_parse_empty(state)
    clear_listing_empty_unverified(state)
    state["listing_scope_unverified"] = True
    state["listing_scope_unverified_page"] = int(page_number)
    state["listing_scope_unverified_attempts"] = int(state.get("listing_scope_unverified_attempts") or 0) + 1
    state["listing_exhausted"] = False
    state["date_boundary_reached"] = False
    state["next_page"] = int(page_number)


def clear_listing_scope_unverified(state: dict) -> None:
    state["listing_scope_unverified"] = False
    state["listing_scope_unverified_page"] = None
    state["listing_scope_unverified_attempts"] = 0


def prepare_listing_scope_unverified_retry(state: dict) -> str:
    """Retry the same page. A spent budget stays incomplete and does not advance."""
    if not state.get("listing_scope_unverified"):
        return "ok"
    page = state.get("listing_scope_unverified_page")
    if page is None:
        page = max(0, int(state.get("next_page") or 0) - 1)
    state["next_page"] = int(page)
    attempts = int(state.get("listing_scope_unverified_attempts") or 0)
    if attempts >= LISTING_PARSE_EMPTY_MAX_ATTEMPTS:
        return "incomplete"
    return "retry"


def update_older_boundary_streak(state: dict, proved: bool) -> int:
    """Persist consecutive older-date pages. A page that does not prove the boundary resets the streak."""
    streak = int(state.get("older_boundary_streak") or 0)
    streak = streak + 1 if proved else 0
    state["older_boundary_streak"] = streak
    if streak >= STOP_AFTER_EMPTY_PAGES:
        state["date_boundary_reached"] = True
    return streak


def classify_listing_boundary_reason(
    scope_status: str,
    n_scoped_links: int,
    observations,
    n_article_fetch_failures: int,
    window_start: str,
    window_end: str = "",
) -> str:
    """Diagnostic only. Does not decide SUCCESS."""
    if scope_status == "LISTING_SCOPE_UNVERIFIED":
        return "LISTING_SCOPE_UNVERIFIED"
    if int(n_scoped_links) <= 0:
        return "NO_SCOPED_ARTICLES"
    trusted = trusted_observed_dates(observations)
    untrusted_observations = 0
    for observed in observations:
        date_str = (observed[0] or "").strip()
        source = observed[1]
        if source not in TRUSTED_DATE_SOURCES or not date_str:
            untrusted_observations += 1
    if n_article_fetch_failures or untrusted_observations or len(observations) != int(n_scoped_links):
        return "HAS_UNTRUSTED_DATES"
    if not trusted:
        return "NO_TRUSTED_DATES"
    if window_end and any(date_str > window_end for date_str in trusted):
        return "HAS_NEWER_DATE"
    if any(date_str >= window_start for date_str in trusted):
        return "HAS_IN_WINDOW_DATE"
    return "ALL_TRUSTED_DATES_OLDER_THAN_WINDOW"


def record_listing_page_diagnostic(
    state: dict,
    page_number: int,
    n_scoped_links: int,
    observations,
    n_article_fetch_failures: int,
    window_start: str,
    window_end: str = "",
    scope_status: str = "SCOPED",
    listing_scope_strategy: str = "unverified",
) -> str:
    """Store a bounded per-page listing diagnostic. Raw HTML is not persisted."""
    if listing_scope_strategy not in ("view-content", "taxonomy-views", "main-fallback", "unverified"):
        listing_scope_strategy = "unverified"
    trusted = trusted_observed_dates(observations)
    reason = classify_listing_boundary_reason(
        scope_status, n_scoped_links, observations, n_article_fetch_failures, window_start, window_end
    )
    entry = {
        "page": int(page_number),
        "n_scoped_links": int(n_scoped_links),
        "n_trusted_dates": len(trusted),
        "n_untrusted_dates": max(0, int(n_scoped_links) - len(trusted)),
        "min_trusted_date": min(trusted) if trusted else "",
        "max_trusted_date": max(trusted) if trusted else "",
        "all_trusted_dates_older_than_window": reason == "ALL_TRUSTED_DATES_OLDER_THAN_WINDOW",
        "boundary_reason": reason,
        "listing_scope_strategy": listing_scope_strategy,
    }
    history = list(state.get("recent_listing_pages") or [])
    history.append(entry)
    state["recent_listing_pages"] = history[-LISTING_DIAGNOSTIC_HISTORY:]
    state["last_page_number"] = entry["page"]
    state["last_page_scoped_article_count"] = entry["n_scoped_links"]
    state["last_page_trusted_date_count"] = entry["n_trusted_dates"]
    state["last_page_untrusted_date_count"] = entry["n_untrusted_dates"]
    state["last_page_min_trusted_date"] = entry["min_trusted_date"]
    state["last_page_max_trusted_date"] = entry["max_trusted_date"]
    state["last_page_all_trusted_dates_older_than_window"] = entry["all_trusted_dates_older_than_window"]
    state["last_boundary_reason"] = reason
    state["last_page_listing_scope_strategy"] = listing_scope_strategy
    return reason


def section_page_url(section_slug: str, page_number: int) -> str:
    section_url = f"{START_URL}/{section_slug}"
    if page_number == 0:
        return section_url
    return f"{section_url}?page={page_number}"


def _export_article_cache() -> list:
    exported = []
    for key, result in ARTICLE_CACHE.items():
        if result.get("kind") == "article":
            row = result.get("article_row") or {}
            if key != row.get("article_url"):
                continue
            exported.append({
                "cache_key": key,
                "kind": "article",
                "article_row": row,
                "media_candidates": result.get("media_candidates") or [],
                "article_scope_error": result.get("article_scope_error") or "",
                "discovered_sections": sorted(result.get("discovered_sections") or []),
            })
            continue
        if result.get("kind") != "fetch_failure" or str(key).startswith("id:"):
            continue
        item = {field: value for field, value in result.items() if field != "discovered_sections"}
        item["cache_key"] = key
        exported.append(item)
    return exported


def canonical_json_bytes(value) -> bytes:
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")


def checkpoint_compatibility_payload() -> dict:
    """Deterministic resume contract. Computed before any network orchestration."""
    return {
        "checkpoint_schema_version": CHECKPOINT_SCHEMA_VERSION,
        "upstream_fingerprints": UPSTREAM_FINGERPRINTS,
        "date_windows": {
            "phase_a": {"label": PHASE_A_LABEL, "start": PHASE_A_START, "end": PHASE_A_END},
            "phase_b": {"label": PHASE_B_LABEL, "start": PHASE_B_START, "end": PHASE_B_END},
        },
        "phase_b_trigger_hours": PHASE_B_TRIGGER_HOURS,
        "section_content_class": dict(SECTION_CONTENT_CLASS),
        "supported_media_types": list(SUPPORTED_MEDIA_TYPES),
        "eligibility_schema_version": ELIGIBILITY_SCHEMA_VERSION,
        "max_single_file_bytes": MAX_SINGLE_FILE_BYTES,
        "max_pages_per_section": MAX_PAGES_PER_SECTION,
        "stop_after_empty_pages": STOP_AFTER_EMPTY_PAGES,
        "listing_parse_empty_max_attempts": LISTING_PARSE_EMPTY_MAX_ATTEMPTS,
        "crawl_pipeline_schema_version": CRAWL_PIPELINE_SCHEMA_VERSION,
    }


def checkpoint_compatibility_sha256() -> str:
    return hashlib.sha256(canonical_json_bytes(checkpoint_compatibility_payload())).hexdigest()


def _require_checkpoint_compatibility(payload: dict) -> dict:
    """Return the saved page cap and whether listing semantics must be migrated.

    Every compatibility field must match except ``max_pages_per_section``, which
    may stay the same or increase, and ``crawl_pipeline_schema_version``, which
    may move from 1.0 to 1.1. Any other version drift fails closed.
    """
    saved = payload.get("checkpoint_compatibility")
    saved_sha = str(payload.get("checkpoint_compatibility_sha256") or "")
    if not isinstance(saved, dict) or not is_sha256_hex(saved_sha):
        raise CheckpointIncompatible("checkpoint is missing a compatibility signature")
    recomputed = hashlib.sha256(canonical_json_bytes(saved)).hexdigest()
    if recomputed != saved_sha:
        raise CheckpointIncompatible("checkpoint compatibility signature does not match its payload")
    current = checkpoint_compatibility_payload()
    if "max_pages_per_section" not in saved or "max_pages_per_section" not in current:
        raise CheckpointIncompatible("checkpoint is missing max_pages_per_section")
    saved_pages = int(saved["max_pages_per_section"])
    current_pages = int(current["max_pages_per_section"])
    ignored = {"max_pages_per_section", "crawl_pipeline_schema_version"}
    saved_rest = {key: value for key, value in saved.items() if key not in ignored}
    current_rest = {key: value for key, value in current.items() if key not in ignored}
    if saved_rest != current_rest:
        raise CheckpointIncompatible(
            "checkpoint compatibility signature does not match the current notebook configuration"
        )
    if current_pages < saved_pages:
        raise CheckpointIncompatible(
            "MAX_PAGES_PER_SECTION decreased from "
            + str(saved_pages)
            + " to "
            + str(current_pages)
            + "; refusing resume"
        )
    saved_version = str(saved.get("crawl_pipeline_schema_version") or "")
    current_version = str(current.get("crawl_pipeline_schema_version") or "")
    if saved_version == current_version:
        migrate = False
    elif saved_version == PREVIOUS_CRAWL_PIPELINE_SCHEMA_VERSION and current_version == CRAWL_PIPELINE_SCHEMA_VERSION:
        migrate = True
    else:
        raise CheckpointIncompatible(
            "unsupported crawl_pipeline_schema_version "
            + saved_version
            + " -> "
            + current_version
        )
    return {"saved_pages": saved_pages, "migrate_listing_semantics": migrate}


def apply_monotonic_page_cap(saved_pages: int) -> int:
    """Reopen sections stopped by an older page cap. Continue from saved next_page."""
    saved_pages = int(saved_pages)
    current_pages = int(MAX_PAGES_PER_SECTION)
    if current_pages < saved_pages:
        raise CheckpointIncompatible(
            "MAX_PAGES_PER_SECTION decreased from "
            + str(saved_pages)
            + " to "
            + str(current_pages)
            + "; refusing resume"
        )
    if current_pages == saved_pages:
        return 0
    reopened = 0
    for state in SECTION_CRAWL_STATE.values():
        if (
            state.get("pagination_cap_reached")
            and not state.get("date_boundary_reached")
            and not state.get("listing_exhausted")
        ):
            state["pagination_cap_reached"] = False
            reopened += 1
    print(
        "PAGE_CAP_RAISED "
        + str(saved_pages)
        + " -> "
        + str(current_pages)
        + "; sections_reopened="
        + str(reopened)
    )
    return reopened


def repair_taxonomy_boundary_capped_sections() -> int:
    """Rewind a capped taxonomy-views section once so the chronological boundary can be re-proved.

    Featured .taxonomy-block1 dates used to keep deep pages looking recent. Re-fetch only the
    last STOP_AFTER_EMPTY_PAGES listing pages. A second call does not rewind again.
    """
    reopened = 0
    for state in SECTION_CRAWL_STATE.values():
        if state.get("taxonomy_boundary_repair_applied"):
            continue
        if not state.get("pagination_cap_reached"):
            continue
        if state.get("date_boundary_reached") or state.get("listing_exhausted"):
            continue
        if state.get("last_page_listing_scope_strategy") != "taxonomy-views":
            continue
        current = int(state.get("next_page") or 0)
        state["next_page"] = max(0, min(current, int(MAX_PAGES_PER_SECTION)) - int(STOP_AFTER_EMPTY_PAGES))
        state["pagination_cap_reached"] = False
        state["older_boundary_streak"] = 0
        state["taxonomy_boundary_repair_applied"] = True
        reopened += 1
    if reopened:
        print("TAXONOMY_BOUNDARY_REPAIR sections_reopened=" + str(reopened))
    return reopened


def repair_orphan_phase_articles() -> dict:
    """Drop phase articles that were persisted before they were accounted for.

    ARTICLE_CACHE is left intact so those rows stay discoverable on resume.
    Idempotent: a second call removes nothing.
    """
    stats = {
        "orphan_articles_removed": 0,
        "processed_ids_released": 0,
        "media_owners_released": 0,
    }
    for phase in ACCUMULATED_PHASES:
        accounted_ids = set()
        accounted_urls = set()
        legitimate_owners = {}
        for bucket in ("records", "exclusions", "fetch_failures"):
            for item in phase.get(bucket) or []:
                source_id_ = str(item.get("source_id") or "")
                article_url = str(item.get("article_url") or "")
                if source_id_:
                    accounted_ids.add(source_id_)
                if article_url:
                    accounted_urls.add(article_url)
                media_hash = str(item.get("media_url_sha256") or "").strip()
                if media_hash and source_id_ and media_hash not in legitimate_owners:
                    legitimate_owners[media_hash] = source_id_
        kept = []
        removed_ids = []
        removed_urls = []
        for row in phase.get("articles") or []:
            source_id_ = str(row.get("source_id") or "")
            if source_id_ and source_id_ in accounted_ids:
                kept.append(row)
                continue
            stats["orphan_articles_removed"] += 1
            removed_ids.append(source_id_)
            article_url = str(row.get("article_url") or "")
            if article_url and article_url not in accounted_urls:
                removed_urls.append(article_url)
        phase["articles"] = kept
        if removed_urls:
            drop_urls = set(removed_urls)
            phase["attempted_article_urls"] = [
                url for url in (phase.get("attempted_article_urls") or []) if url not in drop_urls
            ]
        for source_id_ in removed_ids:
            if source_id_ and source_id_ in RUN_STATE["processed_source_ids"]:
                RUN_STATE["processed_source_ids"].discard(source_id_)
                stats["processed_ids_released"] += 1
            for media_hash, owner in list(OWNED_MEDIA.items()):
                if owner != source_id_:
                    continue
                if media_hash in legitimate_owners:
                    continue
                OWNED_MEDIA.pop(media_hash, None)
                stats["media_owners_released"] += 1
    print(
        "CHECKPOINT_REPAIR: orphan_articles_removed = "
        + str(stats["orphan_articles_removed"])
        + " processed_ids_released = "
        + str(stats["processed_ids_released"])
        + " media_owners_released = "
        + str(stats["media_owners_released"])
    )
    return stats


def audit_restored_checkpoint_records() -> list:
    """Drop PASS rows whose files no longer match. Those source ids may be downloaded again."""
    released = []
    for phase in ACCUMULATED_PHASES:
        kept = []
        released_ids = set()
        released_urls = set()
        for record in phase.get("records") or []:
            if str(record.get("technical_status") or "") == "PASS" and verify_pass_artifact(record):
                source_id_ = record.get("source_id") or ""
                released_ids.add(source_id_)
                if record.get("article_url"):
                    released_urls.add(record["article_url"])
                RUN_STATE["processed_source_ids"].discard(source_id_)
                media_hash = str(record.get("media_url_sha256") or "").strip()
                if media_hash and OWNED_MEDIA.get(media_hash) == source_id_:
                    OWNED_MEDIA.pop(media_hash, None)
                released.append(source_id_)
                print("CHECKPOINT_ARTIFACT_REJECTED", source_id_)
                continue
            kept.append(record)
        phase["records"] = kept
        if released_ids:
            phase["articles"] = [
                row for row in phase.get("articles") or [] if row.get("source_id") not in released_ids
            ]
            phase["attempted_article_urls"] = [
                url for url in phase.get("attempted_article_urls") or [] if url not in released_urls
            ]
        _refresh_phase_hours(phase)
    return released


def save_checkpoint(path: Path = None) -> None:
    """Atomic checkpoint. Audio bytes stay on disk; this file stores crawl and manifest state only."""
    path = Path(path) if path is not None else CHECKPOINT_PATH
    path.parent.mkdir(parents=True, exist_ok=True)
    compatibility = checkpoint_compatibility_payload()
    payload = {
        "checkpoint_compatibility": compatibility,
        "checkpoint_compatibility_sha256": hashlib.sha256(canonical_json_bytes(compatibility)).hexdigest(),
        "section_crawl": SECTION_CRAWL_STATE,
        "owned_media": OWNED_MEDIA,
        "network_state": {
            "new_network_bytes_streamed_this_run": NETWORK_STATE["new_network_bytes_streamed_this_run"],
            "new_network_bytes_successful": NETWORK_STATE["new_network_bytes_successful"],
            "reused_pilot_bytes": NETWORK_STATE["reused_pilot_bytes"],
        },
        "processed_source_ids": sorted(RUN_STATE["processed_source_ids"]),
        "reusable_media_by_hash": RUN_STATE.get("reusable_media_by_hash") or {},
        "total_budget_exhausted": bool(RUN_STATE["total_budget_exhausted"]),
        "configured_total_download_budget_bytes": int(MAX_TOTAL_DOWNLOAD_BYTES),
        "articles": _export_article_cache(),
        "phases": ACCUMULATED_PHASES,
    }
    temporary = path.with_suffix(path.suffix + ".part")
    temporary.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")
    os.replace(temporary, path)



def drop_untrusted_cached_dates() -> int:
    """Drop cached articles whose date is not metadata or time, and restart those section cursors.

    A previous crawl stored every VOV4 publication time as body_fallback, so the
    date boundary never fired. Those rows are fetched again with the publication-time parser.
    """
    dropped_ids = []
    for key, result in list(ARTICLE_CACHE.items()):
        if str(key).startswith("id:") or result.get("kind") != "article":
            continue
        row = result.get("article_row") or {}
        if (row.get("published_date_source") or "") in TRUSTED_DATE_SOURCES:
            continue
        dropped_ids.append(row.get("source_id") or "")
        ARTICLE_CACHE.pop(key, None)
    for source_id_ in dropped_ids:
        if source_id_:
            ARTICLE_CACHE.pop("id:" + source_id_, None)
    restarted = 0
    if dropped_ids:
        for slug, state in list(SECTION_CRAWL_STATE.items()):
            if (state.get("oldest_date_seen") or "").strip():
                continue
            if int(state.get("pages_fetched") or 0) <= 0 and not state.get("pagination_cap_reached"):
                continue
            SECTION_CRAWL_STATE[slug] = blank_section_state()
            restarted += 1
        print(
            "DATE_PARSER_MIGRATION dropped "
            + str(len(dropped_ids))
            + " articles with untrusted dates; restarted "
            + str(restarted)
            + " section cursors"
        )
    return len(dropped_ids)


def load_checkpoint(path: Path = None) -> bool:
    path = Path(path) if path is not None else CHECKPOINT_PATH
    if not path.is_file():
        return False
    payload = json.loads(path.read_text(encoding="utf-8"))
    compatibility = _require_checkpoint_compatibility(payload)
    saved_pages = int(compatibility["saved_pages"])
    migrate_listing_semantics = bool(compatibility["migrate_listing_semantics"])
    ARTICLE_CACHE.clear()
    for item in payload.get("articles") or []:
        key = item.get("cache_key")
        if not key:
            continue
        if item.get("kind") == "article":
            result = {
                "kind": "article",
                "article_row": item.get("article_row") or {},
                "media_candidates": item.get("media_candidates") or [],
                "article_scope_error": item.get("article_scope_error") or "",
                "discovered_sections": set(item.get("discovered_sections") or []),
            }
            ARTICLE_CACHE[key] = result
            source_id_ = (result["article_row"] or {}).get("source_id")
            if source_id_:
                ARTICLE_CACHE["id:" + source_id_] = result
            continue
        ARTICLE_CACHE[key] = {field: value for field, value in item.items() if field != "cache_key"}
    SECTION_CRAWL_STATE.clear()
    for slug, state in (payload.get("section_crawl") or {}).items():
        restored = blank_section_state()
        restored.update(state)
        SECTION_CRAWL_STATE[slug] = restored
    OWNED_MEDIA.clear()
    OWNED_MEDIA.update(payload.get("owned_media") or {})
    for key, value in (payload.get("network_state") or {}).items():
        if key in NETWORK_STATE:
            NETWORK_STATE[key] = value
    RUN_STATE["processed_source_ids"] = set(payload.get("processed_source_ids") or [])
    RUN_STATE["reusable_media_by_hash"] = dict(payload.get("reusable_media_by_hash") or {})
    RUN_STATE["total_budget_exhausted"] = bool(payload.get("total_budget_exhausted"))
    RUN_STATE["budget_exhausted_ceiling_bytes"] = payload.get("configured_total_download_budget_bytes")
    ACCUMULATED_PHASES.clear()
    ACCUMULATED_PHASES.extend(payload.get("phases") or [])
    audit_restored_checkpoint_records()
    drop_untrusted_cached_dates()
    if migrate_listing_semantics:
        migrate_crawl_pipeline_1_0_to_1_1()
    else:
        apply_monotonic_page_cap(saved_pages)
    repair_orphan_phase_articles()
    repair_taxonomy_boundary_capped_sections()
    return True


_REUSED_MEDIA_FIELDS = (
    "artifact_origin",
    "raw_local_path",
    "wav_local_path",
    "video_container_local_path",
    "raw_sha256",
    "wav_sha256",
    "pcm16_sha256",
    "duration_seconds",
    "sample_rate",
    "channels",
    "codec_name",
    "technical_status",
    "technical_notes",
    "video_container_sha256",
    "video_container_bytes",
    "peak_abs",
    "peak_dbfs",
    "rms",
    "rms_dbfs",
    "clipping_fraction",
    "silence_fraction_energy",
)


def migrate_crawl_pipeline_1_0_to_1_1() -> None:
    """Rebuild listing coverage under scoped discovery.

    Old phase rows are not current discovery or current dedup ownership.
    Only a PASS artifact that still verifies is kept, keyed by media hash.
    Files already on disk are not deleted.
    """
    reusable = {}
    for phase in ACCUMULATED_PHASES:
        for record in phase.get("records") or []:
            if str(record.get("technical_status") or "") != "PASS":
                continue
            media_hash = str(record.get("media_url_sha256") or "").strip().lower()
            if not is_sha256_hex(media_hash):
                continue
            if verify_pass_artifact(record):
                continue
            reusable[media_hash] = dict(record)
    RUN_STATE["reusable_media_by_hash"] = reusable
    RUN_STATE.pop("reusable_verified_records", None)
    OWNED_MEDIA.clear()
    for result in list(ARTICLE_CACHE.values()):
        if isinstance(result, dict) and "discovered_sections" in result:
            result["discovered_sections"] = set()
    ACCUMULATED_PHASES.clear()
    RUN_STATE["processed_source_ids"] = set()
    previous_slugs = list(SECTION_CRAWL_STATE.keys())
    SECTION_CRAWL_STATE.clear()
    for slug in list(ALL_SECTION_SLUGS) + previous_slugs:
        SECTION_CRAWL_STATE[slug] = blank_section_state()
    print(
        "CRAWL_PIPELINE_MIGRATION 1.0 -> 1.1: listing coverage reset to page 0; "
        "verified PASS media cached by hash; current ownership cleared"
    )


def bind_reused_media_record(row: dict, primary: dict, media_hash: str):
    """Bind a verified artifact to the current article. Returns None when H is not reusable."""
    cached = (RUN_STATE.get("reusable_media_by_hash") or {}).get(media_hash)
    if not isinstance(cached, dict):
        return None
    if str(cached.get("technical_status") or "") != "PASS":
        return None
    if str(cached.get("media_url_sha256") or "").strip().lower() != media_hash:
        return None
    if verify_pass_artifact(cached):
        return None
    record = {field: cached[field] for field in _REUSED_MEDIA_FIELDS if field in cached}
    record["source_id"] = row["source_id"]
    record["article_url"] = row["article_url"]
    record["article_date"] = row["article_date"]
    record["published_date_source"] = row["published_date_source"]
    record["article_title"] = row.get("article_title") or ""
    record["phase"] = row["phase"]
    record["source_section"] = row["source_section"]
    record["content_class"] = section_content_class(row["source_section"])
    record["media_url"] = primary["media_url"]
    record["media_url_sha256"] = media_hash
    record["media_type"] = primary["media_type"]
    record["media_extension"] = primary["media_extension"]
    record["http_status"] = primary.get("http_status", "")
    record["content_type"] = primary.get("content_type", "")
    record["download_attempted"] = False
    record["download_success"] = False
    hint_value, review_required = metadata_hint(record["article_title"], record["media_url"])
    record["metadata_language_hint"] = hint_value
    record["conflicting_language_evidence"] = bool(review_required)
    record["verification_method"] = "automated_source_provenance_v1"
    status, reason = derive_source_level_status(
        technical_status=record["technical_status"],
        content_class=record["content_class"],
        conflicting_language_evidence=record["conflicting_language_evidence"],
        metadata_hint_value=hint_value,
        article_title=record["article_title"],
    )
    record["source_level_status"] = status
    record["exclusion_reason"] = reason
    record["verification_notes"] = (
        f"content_class={record['content_class']}; technical_status={record['technical_status']}; "
        f"rule_reason={reason or 'n/a'}"
    )
    apply_native_spotcheck(record)
    return record


def crawl_sections_through(window_start: str, window_end: str = "") -> None:
    """Network. Resume each section until trusted dates fall before window_start, or the pager ends."""
    for section_slug in ALL_SECTION_SLUGS:
        state = SECTION_CRAWL_STATE.setdefault(section_slug, blank_section_state())
        if state["pagination_cap_reached"]:
            continue
        if state.get("listing_parse_empty"):
            if section_slug in LISTING_PARSE_RETRIED_THIS_PROCESS:
                continue
            if prepare_listing_parse_empty_retry(state) == "incomplete":
                continue
            LISTING_PARSE_RETRIED_THIS_PROCESS.add(section_slug)
        if state.get("listing_empty_unverified"):
            if section_slug in LISTING_EMPTY_RETRIED_THIS_PROCESS:
                continue
            if prepare_listing_empty_unverified_retry(state) == "incomplete":
                continue
            LISTING_EMPTY_RETRIED_THIS_PROCESS.add(section_slug)
        if state.get("listing_scope_unverified"):
            if section_slug in LISTING_SCOPE_RETRIED_THIS_PROCESS:
                continue
            if prepare_listing_scope_unverified_retry(state) == "incomplete":
                continue
            LISTING_SCOPE_RETRIED_THIS_PROCESS.add(section_slug)
        if state.get("listing_exhausted") or state.get("date_boundary_reached"):
            state["date_boundary_reached"] = True
            continue
        state["listing_fetch_error"] = ""
        older_streak = int(state.get("older_boundary_streak") or 0)
        while state["next_page"] < MAX_PAGES_PER_SECTION:
            page_number = state["next_page"]
            page_url = section_page_url(section_slug, page_number)
            try:
                html_bytes, _final_url, _attempts = fetch_html(page_url)
            except TotalDownloadBudgetExceeded:
                save_checkpoint()
                raise
            except FetchFailed as exc:
                state["listing_fetch_error"] = f"{exc.error_type}: {exc.error_message}"
                print("WARN listing fetch failed:", section_slug, page_number, state["listing_fetch_error"])
                save_checkpoint()
                break
            state["pages_fetched"] += 1
            state["next_page"] += 1
            inspected = inspect_listing_page(html_bytes, page_number, section_slug)
            if inspected["status"] == "LISTING_SCOPE_UNVERIFIED":
                record_listing_page_diagnostic(
                    state, page_number, 0, [], 0, window_start, window_end,
                    "LISTING_SCOPE_UNVERIFIED",
                    inspected.get("listing_scope_strategy") or "unverified",
                )
                older_streak = update_older_boundary_streak(state, False)
                note_listing_scope_unverified(state, page_number)
                LISTING_SCOPE_RETRIED_THIS_PROCESS.add(section_slug)
                print("WARN listing scope unverified:", section_slug, page_number)
                save_checkpoint()
                break
            if inspected["status"] == "LISTING_PARSE_EMPTY":
                record_listing_page_diagnostic(
                    state, page_number, 0, [], 0, window_start, window_end,
                    "SCOPED",
                    inspected.get("listing_scope_strategy") or "unverified",
                )
                older_streak = update_older_boundary_streak(state, False)
                note_listing_parse_empty(state, page_number)
                LISTING_PARSE_RETRIED_THIS_PROCESS.add(section_slug)
                print("WARN listing parse empty while pager still has next:", section_slug, page_number)
                save_checkpoint()
                break
            if inspected["status"] == "LISTING_EMPTY_UNVERIFIED":
                record_listing_page_diagnostic(
                    state, page_number, 0, [], 0, window_start, window_end,
                    "SCOPED",
                    inspected.get("listing_scope_strategy") or "unverified",
                )
                older_streak = update_older_boundary_streak(state, False)
                note_listing_empty_unverified(state, page_number)
                LISTING_EMPTY_RETRIED_THIS_PROCESS.add(section_slug)
                print("WARN listing empty without a verified end marker:", section_slug, page_number)
                save_checkpoint()
                break
            clear_listing_parse_empty(state)
            clear_listing_empty_unverified(state)
            clear_listing_scope_unverified(state)
            for link in inspected["links"]:
                fetch_article_once(link, listing_section=section_slug)
            boundary_links = list(inspected.get("boundary_links") or [])
            if inspected.get("listing_scope_strategy") != "taxonomy-views":
                boundary_links = list(inspected["links"])
            observations = []
            fetch_failures = 0
            for link in boundary_links:
                result = fetch_article_once(link, listing_section=section_slug)
                row = result.get("article_row") or {}
                if result.get("kind") != "article" or row.get("source_section") != section_slug:
                    fetch_failures += 1
                    continue
                observations.append((
                    row.get("article_date") or "",
                    row.get("published_date_source") or "",
                ))
            for date_str in trusted_observed_dates(observations):
                note_observed_date(state, date_str)
            record_listing_page_diagnostic(
                state,
                page_number,
                len(boundary_links),
                observations,
                fetch_failures,
                window_start,
                window_end,
                inspected.get("scope_status") or "SCOPED",
                inspected.get("listing_scope_strategy") or "unverified",
            )
            proved = page_proves_older_boundary(len(boundary_links), observations, fetch_failures, window_start)
            older_streak = update_older_boundary_streak(state, proved)
            if inspected["status"] == "LISTING_EXHAUSTED":
                state["listing_exhausted"] = True
                state["date_boundary_reached"] = True
                save_checkpoint()
                break
            if state.get("date_boundary_reached"):
                save_checkpoint()
                break
            save_checkpoint()
            time.sleep(SLEEP_SECONDS)
        else:
            state["pagination_cap_reached"] = True
            save_checkpoint()


def iter_discovered_articles():
    seen_url = set()
    seen_id = set()
    for key, result in ARTICLE_CACHE.items():
        if result.get("kind") != "article":
            continue
        row = result["article_row"]
        if key != row["article_url"]:
            continue
        if row["source_section"] not in result.get("discovered_sections", ()):
            continue
        if row["article_url"] in seen_url or row["source_id"] in seen_id:
            continue
        seen_url.add(row["article_url"])
        seen_id.add(row["source_id"])
        yield result


def dedup_articles(rows: list) -> tuple:
    """Global dedup by canonical article_url and source_id. Stable owner is sorted article_url."""
    kept = []
    duplicates = []
    seen_url = set()
    seen_id = set()
    for row in sorted(rows, key=lambda item: (item["article_url"], item["source_id"])):
        if row["article_url"] in seen_url or row["source_id"] in seen_id:
            duplicates.append(row)
            continue
        seen_url.add(row["article_url"])
        seen_id.add(row["source_id"])
        kept.append(row)
    return kept, duplicates


def dedup_media_rows(rows: list, already_owned: dict = None) -> tuple:
    """Within this call, the smallest source_id owns the hash. already_owned (earlier phase) wins."""
    owned = dict(already_owned or {})
    kept = []
    duplicates = []
    for row in sorted(rows, key=lambda item: (item["source_id"], item["article_url"])):
        media_hash = row["media_url_sha256"]
        if media_hash in owned:
            duplicates.append({
                "source_id": row["source_id"],
                "article_url": row["article_url"],
                "article_date": row.get("article_date", ""),
                "article_title": row.get("article_title", ""),
                "source_section": row.get("source_section", ""),
                "phase": row.get("phase", ""),
                "stage": "dedup",
                "exclusion_reason": "DUPLICATE_MEDIA",
                "canonical_source_id": owned[media_hash],
                "media_url_sha256": media_hash,
            })
            continue
        owned[media_hash] = row["source_id"]
        kept.append(row)
    return kept, duplicates


def _exclusion_from_article(article_row: dict, stage: str, reason: str, **extra) -> dict:
    payload = {
        "source_id": article_row["source_id"],
        "article_url": article_row["article_url"],
        "article_date": article_row.get("article_date", ""),
        "article_title": article_row.get("article_title", ""),
        "source_section": article_row.get("source_section", ""),
        "phase": article_row.get("phase", ""),
        "stage": stage,
        "exclusion_reason": reason,
        "canonical_source_id": "",
        "media_url_sha256": "",
    }
    payload.update(extra)
    return payload


def _phase_for(phase_label: str, start: str, end: str) -> dict:
    for phase in ACCUMULATED_PHASES:
        if phase.get("phase_label") == phase_label:
            phase["start"] = start
            phase["end"] = end
            return phase
    phase = empty_phase(phase_label, start, end)
    ACCUMULATED_PHASES.append(phase)
    return phase


def _refresh_phase_hours(phase: dict) -> None:
    phase["eligible_source_hours"] = sum(
        float(record["duration_seconds"])
        for record in phase["records"]
        if record.get("source_level_status") == "ELIGIBLE_SOURCE"
    ) / 3600.0


def process_phase(start: str, end: str, phase_label: str) -> dict:
    """Network + local ffmpeg for articles whose trusted date is inside [start, end]."""
    phase = _phase_for(phase_label, start, end)
    discovered = []
    for result in iter_discovered_articles():
        row = dict(result["article_row"])
        if row["source_id"] in RUN_STATE["processed_source_ids"]:
            continue
        if (row.get("published_date_source") or "") not in TRUSTED_DATE_SOURCES:
            continue
        if not in_phase_window(row["article_date"], start, end):
            continue
        row["phase"] = phase_label
        result["article_row"]["phase"] = phase_label
        discovered.append((row, result))
    def _account_article(article_row: dict) -> None:
        phase["articles"].append(article_row)
        phase["attempted_article_urls"].append(article_row["article_url"])

    prepared = []
    for row, result in discovered:
        if result.get("article_scope_error"):
            _account_article(row)
            phase["exclusions"].append(_exclusion_from_article(row, "media_probe", result["article_scope_error"]))
            RUN_STATE["processed_source_ids"].add(row["source_id"])
            save_checkpoint()
            continue
        primary = pick_primary_media(result["media_candidates"])
        if not primary:
            _account_article(row)
            phase["exclusions"].append(_exclusion_from_article(row, "media_probe", "NO_PUBLIC_MEDIA"))
            RUN_STATE["processed_source_ids"].add(row["source_id"])
            save_checkpoint()
            continue
        media_hash = hashlib.sha256(primary["media_url"].encode("utf-8")).hexdigest()
        prepared.append((row, result["media_candidates"], primary, media_hash))
    prepared.sort(key=lambda item: (item[0]["source_id"], item[0]["article_url"]))
    for row, media_candidates, primary, media_hash in prepared:
        if media_hash in OWNED_MEDIA:
            _account_article(row)
            phase["exclusions"].append(_exclusion_from_article(
                row, "dedup", "DUPLICATE_MEDIA",
                canonical_source_id=OWNED_MEDIA[media_hash],
                media_url_sha256=media_hash,
            ))
            RUN_STATE["processed_source_ids"].add(row["source_id"])
            save_checkpoint()
            continue
        OWNED_MEDIA[media_hash] = row["source_id"]
        reused = bind_reused_media_record(row, primary, media_hash)
        if reused is not None:
            _account_article(row)
            phase["records"].append(reused)
            RUN_STATE["processed_source_ids"].add(row["source_id"])
            _refresh_phase_hours(phase)
            save_checkpoint()
            continue
        try:
            record, exclusion = run_article_through_pipeline(row, media_candidates, "")
        except TotalDownloadBudgetExceeded:
            # This row and every later prepared row were never persisted as articles.
            note_total_download_budget_exhausted(row["source_id"], media_hash)
            _refresh_phase_hours(phase)
            save_checkpoint()
            raise
        if record is None and exclusion is None:
            OWNED_MEDIA.pop(media_hash, None)
            continue
        _account_article(row)
        if record is not None:
            phase["records"].append(record)
        if exclusion is not None:
            phase["exclusions"].append(exclusion)
        RUN_STATE["processed_source_ids"].add(row["source_id"])
        _refresh_phase_hours(phase)
        save_checkpoint()
    _refresh_phase_hours(phase)
    save_checkpoint()
    return phase


def collect_fetch_failures() -> list:
    seen = set()
    failures = []
    for key, result in ARTICLE_CACHE.items():
        if result.get("kind") != "fetch_failure":
            continue
        article_url = result.get("article_url") or key
        if article_url in seen or str(key).startswith("id:"):
            continue
        seen.add(article_url)
        failures.append(result)
    return failures


def collect_undated_exclusions() -> list:
    exclusions = []
    for result in iter_discovered_articles():
        row = result["article_row"]
        if (row.get("published_date_source") or "") == "body_fallback":
            continue
        if (row.get("article_date") or "").strip():
            continue
        exclusions.append(_exclusion_from_article(row, "discovery", "UNDATED_ARTICLE"))
    return exclusions


def collect_untrusted_date_exclusions() -> list:
    """body_fallback and any other non-metadata date stay out of the phase window."""
    exclusions = []
    for result in iter_discovered_articles():
        row = result["article_row"]
        source = row.get("published_date_source") or ""
        if source in TRUSTED_DATE_SOURCES:
            continue
        if source == "body_fallback" or (row.get("article_date") or "").strip():
            exclusions.append(_exclusion_from_article(row, "discovery", "UNTRUSTED_DATE"))
    return exclusions


def empty_phase(phase_label: str, start: str, end: str) -> dict:
    return {
        "phase_label": phase_label, "start": start, "end": end,
        "articles": [], "records": [], "exclusions": [], "fetch_failures": [],
        "eligible_source_hours": 0.0, "attempted_article_urls": [],
    }


def merge_phase_results(*phase_results: dict) -> dict:
    merged = {"articles": [], "records": [], "exclusions": [], "fetch_failures": [], "attempted_article_urls": []}
    for phase in phase_results:
        merged["articles"].extend(phase["articles"])
        merged["records"].extend(phase["records"])
        merged["exclusions"].extend(phase["exclusions"])
        merged["fetch_failures"].extend(phase["fetch_failures"])
        merged["attempted_article_urls"].extend(phase.get("attempted_article_urls", []))
    return merged


def section_stats_snapshot() -> dict:
    snapshot = {}
    for slug in ALL_SECTION_SLUGS:
        state = SECTION_CRAWL_STATE.get(slug, blank_section_state())
        blank = blank_section_state()
        snapshot[slug] = {field: state.get(field, blank[field]) for field in SECTION_STAT_FIELDS}
    return snapshot


def note_total_download_budget_exhausted(source_id_: str, media_hash: str = "") -> None:
    """Stop the run without treating the interrupted source as finished."""
    if media_hash:
        OWNED_MEDIA.pop(media_hash, None)
    RUN_STATE["processed_source_ids"].discard(source_id_)
    RUN_STATE["total_budget_exhausted"] = True
    RUN_STATE["budget_exhausted_ceiling_bytes"] = int(MAX_TOTAL_DOWNLOAD_BYTES)


def apply_resumed_download_budget() -> None:
    """Continue only when the configured total budget was raised above the exhausted ceiling and the bytes already streamed."""
    if not RUN_STATE.get("total_budget_exhausted"):
        return
    streamed = int(NETWORK_STATE["new_network_bytes_streamed_this_run"])
    try:
        ceiling = int(RUN_STATE.get("budget_exhausted_ceiling_bytes"))
    except (TypeError, ValueError):
        ceiling = None
    current = int(MAX_TOTAL_DOWNLOAD_BYTES)
    if ceiling is not None and current > ceiling and current > streamed:
        RUN_STATE["total_budget_exhausted"] = False
        print(
            "download budget raised to "
            + str(current)
            + " bytes; interrupted sources stay retryable"
        )
        return
    raise TotalDownloadBudgetExceeded(
        "TOTAL_DOWNLOAD_BUDGET_EXHAUSTED: configured_total_download_budget_bytes="
        + str(current)
        + ", saved_exhausted_ceiling="
        + str(ceiling)
        + ", bytes_already_streamed="
        + str(streamed)
        + ". Set BAHNAR_VOV4_MAX_TOTAL_DOWNLOAD_BYTES above b oth the saved ceiling and the bytes already streamed, then resume."
    )


# Signature is fixed before the network block so a later resume can fail closed on drift.
CHECKPOINT_COMPATIBILITY = checkpoint_compatibility_payload()
CHECKPOINT_COMPATIBILITY_SHA256 = checkpoint_compatibility_sha256()

def refuse_if_nb10_finalized(summary_path: Path) -> None:
    """SUCCESS is immutable. A later network run must not replace that generation."""
    summary_path = Path(summary_path)
    if not summary_path.is_file():
        return
    payload = json.loads(summary_path.read_text(encoding="utf-8"))
    if payload.get("status") == "SUCCESS_RQ2_VOV4_FULL_SOURCE_POOL":
        raise RuntimeError(
            "NB10 already has a finalized SUCCESS artifact set; refusing network rerun. Preserve FINAL artifacts."
        )


RUN_NETWORK_PIPELINE = True  # flip to True only when actually running this notebook end-to-end

if RUN_NETWORK_PIPELINE:
    refuse_if_nb10_finalized(OUT_DIR / "summary.json")
    ACCUMULATED_PHASES.clear()
    if RESET_CHECKPOINT or not load_checkpoint():
        ARTICLE_CACHE.clear()
        SECTION_CRAWL_STATE.clear()
        OWNED_MEDIA.clear()
        RUN_STATE["total_budget_exhausted"] = False
        RUN_STATE["processed_source_ids"] = set()
        RUN_STATE["reusable_media_by_hash"] = {}
        NETWORK_STATE["new_network_bytes_streamed_this_run"] = 0
        NETWORK_STATE["new_network_bytes_successful"] = 0
        NETWORK_STATE["reused_pilot_bytes"] = 0
        print("checkpoint reset; starting a fresh crawl")
    else:
        print("resumed checkpoint:", to_project_relative(CHECKPOINT_PATH))
    apply_resumed_download_budget()
    PHASE_B_TRIGGERED = False
    DATE_SCOPE_START = PHASE_A_START
    DATE_SCOPE_END = PHASE_A_END
    try:
        crawl_sections_through(PHASE_A_START, PHASE_A_END)
        phase_a_result = process_phase(PHASE_A_START, PHASE_A_END, PHASE_A_LABEL)
        print(f"Phase A eligible_source_hours = {phase_a_result['eligible_source_hours']:.4f}")
        incomplete = incomplete_section_slugs()
        if incomplete:
            PHASE_B_TRIGGERED = False
            print("PHASE_A_COVERAGE_INCOMPLETE sections=" + ",".join(incomplete))
        elif phase_a_result["eligible_source_hours"] < PHASE_B_TRIGGER_HOURS:
            PHASE_B_TRIGGERED = True
            print("Phase A below trigger hours; resuming crawl into Phase B ...")
            DATE_SCOPE_START = PHASE_B_START
            crawl_sections_through(PHASE_B_START, PHASE_B_END)
            process_phase(PHASE_B_START, PHASE_B_END, PHASE_B_LABEL)
        else:
            PHASE_B_TRIGGERED = False
            print("Phase A already meets trigger hours; Phase B is not required.")
            ACCUMULATED_PHASES.append(empty_phase(PHASE_B_LABEL, PHASE_B_START, PHASE_B_END))
    except TotalDownloadBudgetExceeded:
        RUN_STATE["total_budget_exhausted"] = True
        print("TOTAL_DOWNLOAD_BUDGET_EXCEEDED: writing a partial manifest")
        save_checkpoint()
    if not ACCUMULATED_PHASES:
        ACCUMULATED_PHASES.append(empty_phase(PHASE_A_LABEL, PHASE_A_START, PHASE_A_END))
    MERGED = merge_phase_results(*ACCUMULATED_PHASES)
    undated_rows = collect_undated_exclusions()
    untrusted_rows = collect_untrusted_date_exclusions()
    MERGED["exclusions"].extend(undated_rows)
    MERGED["exclusions"].extend(untrusted_rows)
    MERGED["fetch_failures"] = collect_fetch_failures()
    MERGED["attempted_article_urls"].extend(item["article_url"] for item in MERGED["fetch_failures"])
    MERGED["attempted_article_urls"].extend(item["article_url"] for item in undated_rows)
    MERGED["attempted_article_urls"].extend(item["article_url"] for item in untrusted_rows)
    MERGED["section_stats"] = section_stats_snapshot()
    MERGED["require_full_coverage"] = True
    MERGED["total_budget_exhausted"] = bool(RUN_STATE["total_budget_exhausted"])
    MERGED["n_articles_discovered_total"] = sum(1 for _result in iter_discovered_articles())
    MERGED["n_articles_in_scope"] = len(MERGED["articles"])
else:
    MERGED = {
        "articles": [], "records": [], "exclusions": [], "fetch_failures": [],
        "attempted_article_urls": [], "section_stats": {}, "require_full_coverage": False,
        "total_budget_exhausted": False,
        "n_articles_discovered_total": 0,
        "n_articles_in_scope": 0,
    }
    DATE_SCOPE_START = PHASE_A_START
    DATE_SCOPE_END = PHASE_A_END
    PHASE_B_TRIGGERED = False
    print("RUN_NETWORK_PIPELINE is False: skipping the live crawl. Flip the flag above to run it for real.")

print("articles:", len(MERGED["articles"]), "| records:", len(MERGED["records"]),
      "| exclusions:", len(MERGED["exclusions"]), "| fetch_failures:", len(MERGED["fetch_failures"]))

CHECKPOINT_REPAIR: orphan_articles_removed = 0 processed_ids_released = 0 media_owners_released = 0
resumed checkpoint: artifacts/rq2/vov4_full/checkpoint/state.json
download budget raised to 12884901888 bytes; interrupted sources stay retryable
Phase A eligible_source_hours = 70.7965
Phase A already meets trigger hours; Phase B is not required.
articles: 2493 | records: 680 | exclusions: 1813 | fetch_failures: 0


## 18. Write output artifacts (Section 9): discovery, media, qa, source pool manifest

All paths are asserted project-relative before anything is written. `exclusions.csv` always contains
every non-eligible / failed row (technical failures, `REJECTED_SOURCE`, `NEEDS_REVIEW`, fetch failures,
missing-media articles) — nothing is silently removed.


In [18]:
SOURCE_POOL_COLUMNS = [
    "source_id", "article_url", "article_date", "article_title", "media_url", "media_url_sha256",
    "media_type", "media_extension", "artifact_origin", "raw_local_path", "wav_local_path",
    "raw_sha256", "wav_sha256", "pcm16_sha256", "duration_seconds", "technical_status",
    "source_section", "metadata_language_hint", "content_class", "conflicting_language_evidence",
    "verification_method", "native_spotcheck_status", "verification_notes", "source_level_status",
    "exclusion_reason", "phase", "published_date_source", "sample_rate", "channels", "codec_name",
    "technical_notes", "http_status", "content_type", "native_spotcheck_notes",
    "video_container_local_path", "video_container_sha256", "video_container_bytes",
    "download_attempted", "download_success",
]

ARTICLE_COLUMNS = ["source_id", "article_url", "article_date", "published_date_source", "article_title",
                    "article_description", "source_section", "page_sha256", "phase", "discovered_at_utc"]

EXCLUSION_COLUMNS = ["source_id", "article_url", "article_date", "article_title", "source_section",
                      "phase", "stage", "exclusion_reason", "canonical_source_id", "media_url_sha256"]

TECHNICAL_QA_COLUMNS = [
    "source_id", "media_url_sha256", "artifact_origin", "technical_status", "technical_notes",
    "duration_seconds", "sample_rate", "channels", "codec_name",
    "peak_abs", "peak_dbfs", "rms", "rms_dbfs", "clipping_fraction", "silence_fraction_energy",
    "raw_sha256", "wav_sha256", "pcm16_sha256",
]


def write_jsonl(path: Path, rows: list) -> None:
    lines = [json.dumps(row, ensure_ascii=False) for row in rows]
    path.write_text(("\n".join(lines) + "\n") if lines else "", encoding="utf-8")


def write_manifests(merged: dict) -> None:
    assert_portable_artifact_paths(merged["records"])

    articles_df = pd.DataFrame(merged["articles"], columns=ARTICLE_COLUMNS)
    articles_df.to_csv(DISCOVERY_DIR / "articles.csv", index=False, encoding="utf-8-sig")
    write_jsonl(DISCOVERY_DIR / "articles.jsonl", merged["articles"])

    media_rows = [
        {
            "source_id": r["source_id"], "article_url": r["article_url"], "media_url": r["media_url"],
            "media_url_sha256": r["media_url_sha256"], "media_type": r["media_type"],
            "media_extension": r["media_extension"], "http_status": r["http_status"],
            "content_type": r["content_type"], "artifact_origin": r["artifact_origin"],
        }
        for r in merged["records"]
    ]
    media_df = pd.DataFrame(media_rows)
    media_df.to_csv(MEDIA_DIR / "media_manifest.csv", index=False, encoding="utf-8-sig")
    write_jsonl(MEDIA_DIR / "media_manifest.jsonl", media_rows)

    technical_rows = []
    for r in merged["records"]:
        technical_rows.append({column: r.get(column, "") for column in TECHNICAL_QA_COLUMNS})
    pd.DataFrame(technical_rows, columns=TECHNICAL_QA_COLUMNS).to_csv(
        QA_DIR / "technical_qa.csv", index=False, encoding="utf-8-sig"
    )

    exclusion_rows = list(merged["exclusions"])
    for r in merged["records"]:
        if r["source_level_status"] != "ELIGIBLE_SOURCE":
            exclusion_rows.append({
                "source_id": r["source_id"], "article_url": r["article_url"], "article_date": r["article_date"],
                "article_title": r["article_title"], "source_section": r["source_section"], "phase": r["phase"],
                "stage": "eligibility" if r["technical_status"] == "PASS" else "technical_qa",
                "exclusion_reason": r["exclusion_reason"] or r["source_level_status"],
            })
    for f in merged["fetch_failures"]:
        exclusion_rows.append({
            "source_id": f.get("source_id", ""), "article_url": f["article_url"], "article_date": "",
            "article_title": "", "source_section": "", "phase": "",
            "stage": "discovery", "exclusion_reason": f"FETCH_FAILED:{f['error_type']}",
        })
    exclusions_path = QA_DIR / "exclusions.csv"
    if exclusion_rows:
        pd.DataFrame(exclusion_rows, columns=EXCLUSION_COLUMNS).to_csv(exclusions_path, index=False, encoding="utf-8-sig")
    else:
        pd.DataFrame(columns=EXCLUSION_COLUMNS).to_csv(exclusions_path, index=False, encoding="utf-8-sig")

    pool_rows = [{column: record.get(column, "") for column in SOURCE_POOL_COLUMNS} for record in merged["records"]]
    pool_df = pd.DataFrame(pool_rows, columns=SOURCE_POOL_COLUMNS)
    pool_df.to_csv(OUT_DIR / "source_pool_manifest.csv", index=False, encoding="utf-8-sig")
    write_jsonl(OUT_DIR / "source_pool_manifest.jsonl", pool_rows)


if RUN_NETWORK_PIPELINE:
    write_manifests(MERGED)
    print("Manifests written under:", OUT_DIR)
else:
    print("RUN_NETWORK_PIPELINE is False: manifest files are not (re)written in this pass.")


Manifests written under: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/artifacts/rq2/vov4_full


## 19. Summary + success status

`SUCCESS_RQ2_VOV4_FULL_SOURCE_POOL` means the requested date scope was completed, every discovered or
attempted source is accounted for, duplicates are resolved, eligible rows are technical PASS, no music
or conflicting-language row is eligible, and paths, hashes, PCM, and hour arithmetic check out.

Ordinary per-source problems stay in `exclusions.csv` and in the summary counts. They do not by
themselves make the run `PARTIAL`. That status is reserved for an incomplete date window (pagination
cap or a listing fetch that stops the crawl early), an unresolved duplicate, or an integrity/contract
failure.

In [19]:
def hours_of(records: list, status: str) -> float:
    return sum(float(r["duration_seconds"]) for r in records if r["source_level_status"] == status) / 3600.0


def run_consistency_gates(merged: dict) -> list:
    """Systemic / incomplete-run problems only. Ordinary per-source failures are exclusions."""
    problems = []
    records = merged["records"]
    articles = merged["articles"]
    exclusions = merged.get("exclusions", [])
    fetch_failures = merged.get("fetch_failures", [])

    article_urls = [a["article_url"] for a in articles]
    if len(article_urls) != len(set(article_urls)):
        problems.append("duplicate article_url in discovery/articles")
    article_ids = [a["source_id"] for a in articles]
    if len(article_ids) != len(set(article_ids)):
        problems.append("duplicate source_id in discovery/articles")
    media_hashes = [r["media_url_sha256"] for r in records if r.get("media_url_sha256")]
    if len(media_hashes) != len(set(media_hashes)):
        problems.append("duplicate media_url_sha256 in source_pool_manifest")
    source_ids = [r["source_id"] for r in records]
    if len(source_ids) != len(set(source_ids)):
        problems.append("duplicate source_id in source_pool_manifest")
    if not all(r["source_level_status"] in ALLOWED_SOURCE_LEVEL_STATUS for r in records):
        problems.append("source_level_status outside allowed enum")
    for record in records:
        if record["source_level_status"] == "ELIGIBLE_SOURCE" and record["technical_status"] != "PASS":
            problems.append(f"ELIGIBLE_SOURCE with non-PASS technical_status: {record['source_id']}")
        if record["source_level_status"] == "ELIGIBLE_SOURCE" and record["content_class"] == "music":
            problems.append(f"music content marked ELIGIBLE_SOURCE: {record['source_id']}")
        if record["source_level_status"] == "ELIGIBLE_SOURCE" and record["conflicting_language_evidence"]:
            problems.append(f"conflicting-language row marked ELIGIBLE_SOURCE: {record['source_id']}")
        if record.get("technical_status") == "PASS" or record["source_level_status"] == "ELIGIBLE_SOURCE":
            problems.extend(verify_pass_artifact(record))
    try:
        assert_portable_artifact_paths(records)
    except RuntimeError as exc:
        problems.append(str(exc))

    record_urls = {r["article_url"] for r in records}
    exclusion_urls = {e.get("article_url", "") for e in exclusions}
    if set(article_urls) - (record_urls | exclusion_urls):
        problems.append("discovered article is missing from records and exclusions")
    attempted = merged.get("attempted_article_urls") or []
    if attempted:
        fetch_urls = {f.get("article_url", "") for f in fetch_failures}
        accounted = set(article_urls) | exclusion_urls | fetch_urls | record_urls
        missing = [url for url in attempted if url not in accounted]
        if missing:
            problems.append(f"attempted article URL not accounted for: {missing[:3]}")

    if merged.get("require_full_coverage"):
        stats_map = merged.get("section_stats") or {}
        for slug in ALL_SECTION_SLUGS:
            stats = stats_map.get(slug)
            if not stats or section_coverage_incomplete(stats):
                problems.append(f"date window not fully covered for section {slug}")
    if merged.get("total_budget_exhausted"):
        problems.append("total download budget exhausted before the date window was completed")
    return problems


def build_summary(merged: dict) -> dict:
    records = merged["records"]
    n_public_audio = sum(1 for r in records if r["media_type"] == "audio")
    n_public_video = sum(1 for r in records if r["media_type"] == "video")
    content_class_counts = pd.Series([r["content_class"] for r in records]).value_counts().to_dict() if records else {}
    exclusion_rows = list(merged["exclusions"]) + [
        {"exclusion_reason": r["exclusion_reason"] or r["source_level_status"]}
        for r in records if r["source_level_status"] != "ELIGIBLE_SOURCE"
    ] + [{"exclusion_reason": f"FETCH_FAILED:{f['error_type']}"} for f in merged["fetch_failures"]]
    exclusion_reason_counts = (
        pd.Series([e["exclusion_reason"] for e in exclusion_rows]).value_counts().to_dict() if exclusion_rows else {}
    )
    n_technical_pass = sum(1 for r in records if r["technical_status"] == "PASS")
    n_technical_fail = sum(1 for r in records if r["technical_status"] != "PASS")
    n_eligible = sum(1 for r in records if r["source_level_status"] == "ELIGIBLE_SOURCE")
    n_rejected = sum(1 for r in records if r["source_level_status"] == "REJECTED_SOURCE")
    n_needs_review = sum(1 for r in records if r["source_level_status"] == "NEEDS_REVIEW")
    n_pilot_reused = sum(1 for r in records if r.get("artifact_origin") == "nb09_pilot")
    n_new_download_attempted = sum(1 for r in records if r.get("download_attempted"))
    n_new_download_success = sum(1 for r in records if r.get("download_success"))
    n_new_technical_pass = sum(
        1 for r in records
        if r.get("artifact_origin") == "nb10_full" and r.get("download_success") and r.get("technical_status") == "PASS"
    )
    n_new_downloaded = n_new_download_success
    n_articles_in_scope = len(merged["articles"])
    if "n_articles_discovered_total" in merged:
        n_articles_discovered_total = int(merged["n_articles_discovered_total"])
    else:
        n_articles_discovered_total = sum(1 for _result in iter_discovered_articles())
    problems = run_consistency_gates(merged)
    status = "SUCCESS_RQ2_VOV4_FULL_SOURCE_POOL" if not problems else "PARTIAL_RQ2_VOV4_FULL_SOURCE_POOL"
    return {
        "status": status,
        "date_scope_start": DATE_SCOPE_START,
        "date_scope_end": DATE_SCOPE_END,
        "phase_b_triggered": PHASE_B_TRIGGERED,
        "n_articles_discovered_total": n_articles_discovered_total,
        "n_articles_in_scope": n_articles_in_scope,
        "n_articles_discovered": n_articles_discovered_total,
        "n_public_media": len(records),
        "n_public_audio": n_public_audio,
        "n_public_video": n_public_video,
        "n_pilot_reused": n_pilot_reused,
        "n_new_download_attempted": n_new_download_attempted,
        "n_new_download_success": n_new_download_success,
        "n_new_technical_pass": n_new_technical_pass,
        "n_new_downloaded": n_new_downloaded,
        "new_network_bytes": NETWORK_STATE["new_network_bytes_successful"],
        "reused_pilot_bytes": NETWORK_STATE["reused_pilot_bytes"],
        "n_technical_pass": n_technical_pass,
        "n_technical_fail": n_technical_fail,
        "technical_pass_hours": sum(float(r["duration_seconds"]) for r in records if r["technical_status"] == "PASS") / 3600.0,
        "n_eligible_source": n_eligible,
        "eligible_source_hours": hours_of(records, "ELIGIBLE_SOURCE"),
        "n_rejected_source": n_rejected,
        "rejected_source_hours": hours_of(records, "REJECTED_SOURCE"),
        "n_needs_review": n_needs_review,
        "needs_review_hours": hours_of(records, "NEEDS_REVIEW"),
        "content_class_counts": content_class_counts,
        "exclusion_reason_counts": exclusion_reason_counts,
        "n_fetch_failed": len(merged["fetch_failures"]),
        "total_download_budget_exhausted": bool(merged.get("total_budget_exhausted")),
        "section_crawl": merged.get("section_stats", {}),
        "problems": problems,
        "generated_at_utc": utc_now(),
    }


if RUN_NETWORK_PIPELINE:
    PENDING_SUMMARY = build_summary(MERGED)
    print(json.dumps(PENDING_SUMMARY, ensure_ascii=False, indent=2))
else:
    PENDING_SUMMARY = None
    print("RUN_NETWORK_PIPELINE is False: summary.json is not (re)written in this pass.")


{
  "status": "SUCCESS_RQ2_VOV4_FULL_SOURCE_POOL",
  "date_scope_start": "2026-01-01",
  "date_scope_end": "2026-09-24",
  "phase_b_triggered": false,
  "n_articles_discovered_total": 13721,
  "n_articles_in_scope": 2493,
  "n_articles_discovered": 13721,
  "n_public_media": 680,
  "n_public_audio": 549,
  "n_public_video": 131,
  "n_pilot_reused": 5,
  "n_new_download_attempted": 546,
  "n_new_download_success": 478,
  "n_new_technical_pass": 478,
  "n_new_downloaded": 478,
  "new_network_bytes": 12562495599,
  "reused_pilot_bytes": 120846086,
  "n_technical_pass": 612,
  "n_technical_fail": 68,
  "technical_pass_hours": 105.39306880861116,
  "n_eligible_source": 335,
  "eligible_source_hours": 70.79646098972223,
  "n_rejected_source": 284,
  "rejected_source_hours": 24.40406781888887,
  "n_needs_review": 61,
  "needs_review_hours": 10.192539999999996,
  "content_class_counts": {
    "music": 273,
    "radio_report": 220,
    "news": 43,
    "mixed": 36,
    "rural_affairs": 31,
    "

## 20. Contract (Section 12): machine-readable rules + upstream fingerprints + success-gate record

`contract.json` is the audit trail: which upstream Notebook 07/08/09 state this run trusted, the exact
date windows used, the eligibility/technical-QA rules applied, and a static self-check that this
notebook's own saved source contains no ASR/MT/pseudo-label/training import or call pattern.


In [20]:
FORBIDDEN_CODE_PATTERNS = (
    "import whisper", "faster_whisper", "import torch", "from transformers", "import transformers",
    "import nemo", "import openai", "openai.chatcompletion", "openai.completion",
    "wav2vec", "speechbrain", "pseudo_label", "pseudolabel", "asr_pipeline", "import sentencepiece",
    "import fairseq", "import sacrebleu", "model.fit(", "model.train(", "trainer.train(",
)


def scan_notebook_for_forbidden_code(notebook_path: Path) -> list:
    """Static self-check: read this notebook's own saved .ipynb and scan code-cell source text.

    The cell that *defines* FORBIDDEN_CODE_PATTERNS necessarily contains every pattern string as a
    literal, so that one cell is excluded from the scan (self-reference, not actual usage); every
    other code cell is scanned.
    """
    if not notebook_path.is_file():
        return ["notebook file not found for self-check: " + str(notebook_path)]
    nb = json.loads(notebook_path.read_text(encoding="utf-8"))
    hits = []
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        if "FORBIDDEN_CODE_PATTERNS = (" in source:
            continue
        source_lower = source.lower()
        for pattern in FORBIDDEN_CODE_PATTERNS:
            if pattern in source_lower:
                hits.append(pattern)
    return hits


def build_contract() -> dict:
    forbidden_hits = scan_notebook_for_forbidden_code(NOTEBOOK_PATH)
    return {
        "notebook": "10_RQ2_VOV4_Full_Expansion_Source_Eligibility.ipynb",
        "schema_version": "1.0",
        "date_windows": {
            "phase_a": {"label": PHASE_A_LABEL, "start": PHASE_A_START, "end": PHASE_A_END},
            "phase_b": {"label": PHASE_B_LABEL, "start": PHASE_B_START, "end": PHASE_B_END,
                        "trigger_hours": PHASE_B_TRIGGER_HOURS, "triggered": PHASE_B_TRIGGERED},
        },
        "upstream_fingerprints": UPSTREAM_FINGERPRINTS,
        "supported_media_types": SUPPORTED_MEDIA_TYPES,
        "download_budget": {
            "max_single_file_bytes": MAX_SINGLE_FILE_BYTES,
            "max_total_download_bytes": MAX_TOTAL_DOWNLOAD_BYTES,
            "env_single_file": "BAHNAR_VOV4_MAX_SINGLE_FILE_BYTES",
            "env_total": "BAHNAR_VOV4_MAX_TOTAL_DOWNLOAD_BYTES",
            "total_download_budget_is_systemic": True,
        },
        "media_owner_rule": "phase_priority_then_smallest_source_id",
        "date_boundary_uses_only": ["metadata", "time"],
        "listing_exhaustion_requires_no_next_page": True,
        "section_content_class_map": SECTION_CONTENT_CLASS,
        "eligibility_rules": {
            "allowed_source_level_status": sorted(ALLOWED_SOURCE_LEVEL_STATUS),
            "allowed_content_classes": sorted(ALLOWED_CONTENT_CLASSES),
            "speech_eligible_classes": sorted(SPEECH_ELIGIBLE_CLASSES),
            "ambiguous_needs_review_classes": sorted(AMBIGUOUS_NEEDS_REVIEW_CLASSES),
            "music_classes_always_rejected": sorted({SECTION_CONTENT_CLASS[s] for s in MUSIC_SECTIONS}),
            "requires_technical_pass_for_eligible": True,
            "requires_no_conflicting_language_for_eligible": True,
            "per_file_human_review_required": False,
        },
        "technical_qa_rules": {
            "decode_success_required": True,
            "duration_must_be_finite_and_positive": True,
            "pcm_validation_required": True,
            "metrics": ["peak_abs", "peak_dbfs", "rms", "rms_dbfs", "clipping_fraction", "silence_fraction_energy"],
            "silent_rule": "peak_abs <= 0 or silence_fraction_energy >= 1.0",
        },
        "artifact_schema": {"source_pool_manifest_columns": SOURCE_POOL_COLUMNS},
        "forbidden_code_pattern_hits": forbidden_hits,
        "generated_at_utc": utc_now(),
    }


FINAL_ARTIFACT_RELATIVE = (
    "discovery/articles.csv",
    "discovery/articles.jsonl",
    "media/media_manifest.csv",
    "media/media_manifest.jsonl",
    "qa/technical_qa.csv",
    "qa/exclusions.csv",
    "source_pool_manifest.csv",
    "source_pool_manifest.jsonl",
)


class FinalArtifactError(RuntimeError):
    """A required final artifact could not be written or verified."""


def atomic_write_text(path: Path, text: str) -> None:
    """Write text via a temporary sibling, fsync, then replace the destination."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".part")
    try:
        with temporary.open("w", encoding="utf-8") as handle:
            handle.write(text)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    except Exception:
        if temporary.is_file():
            temporary.unlink()
        raise


def verify_final_artifact_hashes(out_dir: Path, expected: dict) -> None:
    out_dir = Path(out_dir)
    for relative, digest in expected.items():
        artifact = out_dir / relative
        if not artifact.is_file():
            raise FinalArtifactError("missing final artifact: " + relative)
        actual = sha256_file(artifact)
        if actual != digest:
            raise FinalArtifactError("hash mismatch: " + relative)
    if not (out_dir / "contract.json").is_file():
        raise FinalArtifactError("missing contract.json")


def _read_text_if_file(path: Path):
    path = Path(path)
    if not path.is_file():
        return None
    return path.read_text(encoding="utf-8")


def finalize_run_publication(out_dir: Path, summary: dict, contract: dict, contract_writer=None, after_contract_commit=None) -> None:
    """Publish one generation: contract.json, then summary.json last.

    Summary consistency is checked before either file is replaced. Required
    artifact hashes are computed before the contract commit. If anything fails
    after the contract commit and before the summary commit, the previous
    contract is restored and the previous summary is left untouched.
    """
    out_dir = Path(out_dir)
    if summary.get("status") == "SUCCESS_RQ2_VOV4_FULL_SOURCE_POOL" and summary.get("problems"):
        raise FinalArtifactError("refusing to publish SUCCESS while consistency problems remain")
    expected = {}
    for relative in FINAL_ARTIFACT_RELATIVE:
        artifact = out_dir / relative
        if not artifact.is_file():
            raise FinalArtifactError("missing final artifact: " + relative)
        expected[relative] = sha256_file(artifact)
    published_contract = dict(contract)
    published_contract["final_artifact_sha256"] = expected
    if published_contract.get("forbidden_code_pattern_hits"):
        raise FinalArtifactError(
            "forbidden code patterns: " + str(published_contract["forbidden_code_pattern_hits"])
        )
    contract_path = out_dir / "contract.json"
    summary_path = out_dir / "summary.json"
    previous_contract = _read_text_if_file(contract_path)
    contract_text = json.dumps(published_contract, ensure_ascii=False, indent=2)
    writer = contract_writer or atomic_write_text
    contract_committed = False
    try:
        writer(contract_path, contract_text)
        contract_committed = True
        verify_final_artifact_hashes(out_dir, expected)
        loaded = json.loads(contract_path.read_text(encoding="utf-8"))
        if loaded.get("final_artifact_sha256") != expected:
            raise FinalArtifactError("contract artifact hashes do not match the files on disk")
        if after_contract_commit is not None:
            after_contract_commit()
        atomic_write_text(summary_path, json.dumps(summary, ensure_ascii=False, indent=2))
    except Exception:
        if contract_committed:
            if previous_contract is None:
                if contract_path.is_file():
                    contract_path.unlink()
            else:
                atomic_write_text(contract_path, previous_contract)
        raise


if RUN_NETWORK_PIPELINE:
    contract = build_contract()
    try:
        finalize_run_publication(OUT_DIR, PENDING_SUMMARY, contract)
    except FinalArtifactError as exc:
        raise RuntimeError("final artifacts were not published: " + str(exc)) from exc
    print(json.dumps(contract, ensure_ascii=False, indent=2)[:2000])
    if PENDING_SUMMARY["status"] != "SUCCESS_RQ2_VOV4_FULL_SOURCE_POOL":
        raise RuntimeError(f"{PENDING_SUMMARY['status']}: {PENDING_SUMMARY['problems']}")
else:
    print("RUN_NETWORK_PIPELINE is False: contract.json and summary.json are not (re)written in this pass.")


{
  "notebook": "10_RQ2_VOV4_Full_Expansion_Source_Eligibility.ipynb",
  "schema_version": "1.0",
  "date_windows": {
    "phase_a": {
      "label": "A",
      "start": "2026-01-01",
      "end": "2026-09-24"
    },
    "phase_b": {
      "label": "B",
      "start": "2025-01-01",
      "end": "2025-12-31",
      "trigger_hours": 10.0,
      "triggered": false
    }
  },
  "upstream_fingerprints": {
    "nb07_candidates_csv_sha256": "6187d00f6557eebc9206018023c0ad8b35696d565a713c9f809d8ead2a96ea8f",
    "nb07_summary_sha256": "e7cb298686a1cb3d18550edf6364ebcdbb09ca5081f065a473045a7089cac396",
    "nb08_media_probe_csv_sha256": "5c161254d11f4075acc6dcc620d424726d85a34a48373d40fe4f04ae3e9575f6",
    "nb08_summary_sha256": "b9d075b0b7b6e0a06dfa66c560193425bb4b3642543d77a4d13922ef30951d08",
    "nb09_pilot_manifest_csv_sha256": "135ad5ee086fa227b22e3704bce731b64b7a0ec1e1fb11d079452e54ff7cf64f",
    "nb09_technical_qa_csv_sha256": "5218e8f87e9d625ca1b5ea9ddf035ebe3ffbc505cce6e7478f5df4b741

## 21. Synthetic tests (Section 13) — fully offline, no network

Covers: duplicate article URL handling, duplicate media URL handling, Notebook 09 pilot media reuse,
relative-path round trip, MP3 classification, MP4 classification, content eligibility rules, music
rejection, explicit non-Bahnar conflict rejection, ambiguous radio-drama -> `NEEDS_REVIEW`, summary hour
arithmetic, duplicate `source_id` fail-closed, and no machine-specific persisted paths.


In [21]:
def _assert_hint(title: str, expected: str) -> None:
    hint, flag = metadata_hint(title, "https://cdn.example.com/clip.mp3")
    if hint != expected or flag is not (expected == "non_bahnar_likely"):
        raise AssertionError((title, hint, flag))


def _rejected_record(source_id_: str, article_url: str, media_hash: str) -> dict:
    return {
        "source_id": source_id_,
        "article_url": article_url,
        "media_url_sha256": media_hash,
        "media_type": "audio",
        "artifact_origin": "nb10_full",
        "technical_status": "TECHNICAL_FAIL",
        "source_level_status": "REJECTED_SOURCE",
        "content_class": "news",
        "conflicting_language_evidence": False,
        "exclusion_reason": "TECHNICAL_FAIL",
        "raw_local_path": "",
        "wav_local_path": "",
        "video_container_local_path": "",
        "duration_seconds": 0.0,
    }


def run_synthetic_checks() -> None:
    # 1. duplicate article URL handling, globally
    article_rows = [
        {"article_url": "https://vov4.vov.vn/bahnar/a/x.vov4", "source_id": "VOV4_B"},
        {"article_url": "https://vov4.vov.vn/bahnar/a/x.vov4", "source_id": "VOV4_A"},
        {"article_url": "https://vov4.vov.vn/bahnar/a/y.vov4", "source_id": "VOV4_C"},
    ]
    kept_articles, dup_articles = dedup_articles(article_rows)
    if len(kept_articles) != 2 or len(dup_articles) != 1:
        raise AssertionError((kept_articles, dup_articles))
    if any(row["article_url"] == "https://vov4.vov.vn/bahnar/a/x.vov4" and row["source_id"] != "VOV4_A" for row in kept_articles):
        raise AssertionError(kept_articles)

    # 2. duplicate media URL handling, global and deterministic
    found = {}
    consider_url(found, "https://vov4.vov.vn/bahnar/a/x.vov4", "https://cdn.example.com/a.mp3", "audio_tag", "audio")
    consider_url(found, "https://vov4.vov.vn/bahnar/a/x.vov4", "https://cdn.example.com/a.mp3", "audio_tag", "audio")
    if len(found) != 1:
        raise AssertionError("duplicate media_url was not deduped")
    shared_hash = "ab" * 32
    media_rows = [
        {"source_id": "VOV4_B", "article_url": "https://vov4.vov.vn/bahnar/a/b.vov4", "article_date": "2026-02-01",
         "article_title": "b", "source_section": "news", "phase": "A", "media_url_sha256": shared_hash},
        {"source_id": "VOV4_A", "article_url": "https://vov4.vov.vn/bahnar/a/a.vov4", "article_date": "2026-02-02",
         "article_title": "a", "source_section": "news", "phase": "A", "media_url_sha256": shared_hash},
    ]
    kept_media, dup_media = dedup_media_rows(media_rows)
    if len(kept_media) != 1 or kept_media[0]["source_id"] != "VOV4_A":
        raise AssertionError(kept_media)
    if len(dup_media) != 1 or dup_media[0]["exclusion_reason"] != "DUPLICATE_MEDIA":
        raise AssertionError(dup_media)
    if dup_media[0]["canonical_source_id"] != "VOV4_A":
        raise AssertionError(dup_media[0])
    kept_again, dup_again = dedup_media_rows(media_rows, already_owned={shared_hash: "VOV4_A"})
    if kept_again or len(dup_again) != 2:
        raise AssertionError("already-owned media was downloaded again")
    phase_b_smaller = [{
        "source_id": "VOV4_A", "article_url": "https://vov4.vov.vn/bahnar/a/later.vov4",
        "article_date": "2025-02-02", "article_title": "later", "source_section": "news",
        "phase": "B", "media_url_sha256": shared_hash,
    }]
    kept_phase_b, dup_phase_b = dedup_media_rows(phase_b_smaller, already_owned={shared_hash: "VOV4_M"})
    if kept_phase_b or dup_phase_b[0]["canonical_source_id"] != "VOV4_M":
        raise AssertionError("Phase A owner lost to a smaller Phase B source_id")

    # 3. Notebook 09 pilot media reuse only after integrity verification
    if not PILOT_BY_MEDIA_HASH:
        raise AssertionError("expected at least one indexed pilot media row")
    some_hash = next(iter(PILOT_BY_MEDIA_HASH))
    pilot_hit = match_pilot_media(some_hash)
    if pilot_hit is None:
        raise AssertionError("expected a real pilot hash to pass integrity and match_pilot_media")
    if any(not str(pilot_hit.get("_qa_metrics", {}).get(field, "")).strip() for field in QA_METRIC_FIELDS):
        raise AssertionError("pilot reuse did not preserve technical_qa metrics")
    if match_pilot_media("0" * 64) is not None:
        raise AssertionError("unknown hash should not match any pilot row")
    tampered = dict(pilot_hit)
    tampered["raw_sha256"] = "0" * 64
    ok_tampered, _reason, _qa = verify_pilot_integrity(tampered)
    if ok_tampered:
        raise AssertionError("tampered pilot hash was trusted")

    # 4. relative-path round trip
    sample = PROJECT_ROOT / "artifacts" / "rq2" / "vov4_full" / "raw" / "unit_test.mp3"
    rel = to_project_relative(sample)
    if resolve_project_path(rel) != sample.resolve():
        raise AssertionError("path round trip failed")

    # 5 & 6. MP3 / MP4 classification and no unknown->mp3 default
    if media_extension("https://x.example.com/a.mp3") != "mp3" or media_type_for("mp3") != "audio":
        raise AssertionError("mp3 not classified as audio")
    if media_extension("https://x.example.com/a.mp4") != "mp4" or media_type_for("mp4") != "video":
        raise AssertionError("mp4 not classified as video")
    if resolve_audio_extension("https://x.example.com/a.mp3") != "mp3":
        raise AssertionError("mp3 URL extension")
    if resolve_audio_extension("https://x.example.com/noext", "audio/mpeg") != "mp3":
        raise AssertionError("audio/mpeg MIME should map to mp3")
    if resolve_video_extension("https://x.example.com/noext", "video/mp4") != "mp4":
        raise AssertionError("video/mp4 MIME should map to mp4")
    for bad_url, bad_mime in (
        ("https://x.example.com/noext", ""),
        ("https://x.example.com/clip.xyz", "application/octet-stream"),
    ):
        try:
            resolve_audio_extension(bad_url, bad_mime)
            raise AssertionError("unknown audio type defaulted instead of failing")
        except RuntimeError as exc:
            if "UNSUPPORTED_AUDIO_EXTENSION_OR_MIME" not in str(exc):
                raise

    # 7. content eligibility rules (baseline eligible)
    status, reason = derive_source_level_status(
        technical_status="PASS", content_class="news", conflicting_language_evidence=False,
        metadata_hint_value="bahnar_likely", article_title="Mot bai bao co tieu de",
    )
    if status != "ELIGIBLE_SOURCE" or reason != "":
        raise AssertionError((status, reason))

    # 8. music rejection
    status, reason = derive_source_level_status(
        technical_status="PASS", content_class="music", conflicting_language_evidence=False,
        metadata_hint_value="unknown", article_title="Bai hat",
    )
    if status != "REJECTED_SOURCE" or reason != "MUSIC_CONTENT":
        raise AssertionError((status, reason))

    # 9. accent-folded language names, token boundaries, explicit conflict rejection
    for title in ("Êđê", "Ede", "Jrai", "Gia Rai", "K'Ho", "Kơ Ho"):
        _assert_hint(title, "non_bahnar_likely")
    _assert_hint("Khoa hoc nong thon va needed report", "unknown")
    hint_value, review_required = metadata_hint("Ban tin tieng Ede", "https://x.example.com/de.mp3")
    if hint_value != "non_bahnar_likely" or not review_required:
        raise AssertionError((hint_value, review_required))
    status, reason = derive_source_level_status(
        technical_status="PASS", content_class="news", conflicting_language_evidence=True,
        metadata_hint_value=hint_value, article_title="Ban tin",
    )
    if status != "REJECTED_SOURCE" or reason != "CONFLICTING_LANGUAGE_EVIDENCE":
        raise AssertionError((status, reason))
    _assert_hint("BANA - Dan ca Ede", "non_bahnar_likely")
    _assert_hint("BANA - Dân ca Êđê", "non_bahnar_likely")
    _assert_hint("Khoa hoc", "unknown")
    _assert_hint("phap luat", "unknown")
    _assert_hint("tieng phap", "non_bahnar_likely")
    if "kho" in OTHER_LANGUAGE_PHRASES or "phap" in OTHER_LANGUAGE_PHRASES:
        raise AssertionError("bare kho/phap tokens must not be other-language evidence")

    # 10. ambiguous radio drama -> NEEDS_REVIEW
    status, reason = derive_source_level_status(
        technical_status="PASS", content_class="radio_drama", conflicting_language_evidence=False,
        metadata_hint_value="unknown", article_title="Tieu pham",
    )
    if status != "NEEDS_REVIEW" or not reason.startswith("AMBIGUOUS_CONTENT_CLASS"):
        raise AssertionError((status, reason))

    # 11. summary hour arithmetic
    fixture_records = [
        {"duration_seconds": 3600.0, "source_level_status": "ELIGIBLE_SOURCE"},
        {"duration_seconds": 1800.0, "source_level_status": "ELIGIBLE_SOURCE"},
        {"duration_seconds": 900.0, "source_level_status": "REJECTED_SOURCE"},
    ]
    if abs(hours_of(fixture_records, "ELIGIBLE_SOURCE") - 1.5) > 1e-9:
        raise AssertionError(hours_of(fixture_records, "ELIGIBLE_SOURCE"))
    if abs(hours_of(fixture_records, "REJECTED_SOURCE") - 0.25) > 1e-9:
        raise AssertionError(hours_of(fixture_records, "REJECTED_SOURCE"))

    # 12. duplicate source_id fail-closed
    dup_records = [
        _rejected_record("VOV4_AAA", "https://vov4.vov.vn/bahnar/a/1.vov4", "11" * 32),
        _rejected_record("VOV4_AAA", "https://vov4.vov.vn/bahnar/a/2.vov4", "22" * 32),
    ]
    dup_problems = run_consistency_gates({
        "records": dup_records,
        "articles": [
            {"article_url": dup_records[0]["article_url"], "source_id": "VOV4_AAA"},
            {"article_url": dup_records[1]["article_url"], "source_id": "VOV4_AAA"},
        ],
        "exclusions": [],
        "fetch_failures": [],
        "require_full_coverage": False,
    })
    if not any("duplicate source_id" in problem for problem in dup_problems):
        raise AssertionError(dup_problems)

    # 13. no machine-specific persisted paths
    try:
        assert_portable_artifact_paths([{"raw_local_path": "/Users/someone/x.mp3", "wav_local_path": "", "video_container_local_path": ""}])
        raise AssertionError("machine-specific path was not rejected")
    except RuntimeError:
        pass
    assert_portable_artifact_paths([{"raw_local_path": "artifacts/rq2/vov4_full/raw/x.mp3", "wav_local_path": "", "video_container_local_path": ""}])

    # cross-section listing: news page must drop a music-section URL
    news_slug = "kotong-ang-lom-topol-thoi-su-xa-hoi"
    music_slug = "video-clip-bai-joh-hori-video-clip-ca-nhac"
    mixed_html = (
        f'<html><body><div class="view-content">'
        f'<a href="/bahnar/{news_slug}/tin-tuc-1.vov4">news</a>'
        f'<a href="/bahnar/{music_slug}/bai-hat-2.vov4">music</a>'
        f'</div></body></html>'
    ).encode("utf-8")
    mixed_links = extract_section_links_from_html(mixed_html, f"https://vov4.vov.vn/bahnar/{news_slug}", news_slug)
    if mixed_links != [f"https://vov4.vov.vn/bahnar/{news_slug}/tin-tuc-1.vov4"]:
        raise AssertionError(mixed_links)

    synthetic_html = (
        b"<html><body><div class=\"view-content\">"
        b"<a href=\"/bahnar/choh-jang-sa-nha-nong/bai-a-1.vov4\">A</a>"
        b"<a href=\"/bahnar/choh-jang-sa-nha-nong/bai-a-1.vov4\">A dup</a>"
        b"<a href=\"/bahnar/choh-jang-sa-nha-nong\">section index, not an article</a>"
        b"<a href=\"/bahnar/choh-jang-sa-nha-nong/photo.jpg\">not an article</a>"
        b"</div></body></html>"
    )
    links = extract_section_links_from_html(
        synthetic_html, "https://vov4.vov.vn/bahnar/choh-jang-sa-nha-nong", "choh-jang-sa-nha-nong"
    )
    if links != ["https://vov4.vov.vn/bahnar/choh-jang-sa-nha-nong/bai-a-1.vov4"]:
        raise AssertionError(links)

    # manual HTML redirects: validate before requesting the next hop
    class _FakeResponse:
        def __init__(self, code, headers, content=b"<html>ok</html>"):
            self.status_code = code
            self.headers = headers
            self.content = content

        def close(self):
            return None

    requested = []
    saved_get = session.get
    saved_robots = robots_decision
    saved_host = assert_public_http_host

    def _fake_robots(url):
        return "allow"

    def _fake_host(url):
        host = (urlparse(url).hostname or "").lower()
        if host != BASE_HOST:
            raise NonPublicHost(host)

    def _fake_get(url, timeout=30, allow_redirects=True, **_kwargs):
        if allow_redirects is not False:
            raise AssertionError("allow_redirects must be False")
        requested.append(url)
        if url == "https://vov4.vov.vn/bahnar/start":
            return _FakeResponse(302, {"Location": "/bahnar/next"})
        if url == "https://vov4.vov.vn/bahnar/next":
            return _FakeResponse(200, {"Content-Type": "text/html; charset=utf-8"})
        raise AssertionError("unexpected request " + url)

    globals()["robots_decision"] = _fake_robots
    globals()["assert_public_http_host"] = _fake_host
    session.get = _fake_get
    try:
        body, final_url = get_html_once("https://vov4.vov.vn/bahnar/start")
        if final_url != "https://vov4.vov.vn/bahnar/next" or body != b"<html>ok</html>":
            raise AssertionError((final_url, body))
        if "evil.example" in " ".join(requested):
            raise AssertionError(requested)
        requested.clear()

        def _redirect_away(url, timeout=30, allow_redirects=True, **_kwargs):
            if allow_redirects is not False:
                raise AssertionError("allow_redirects must be False")
            requested.append(url)
            return _FakeResponse(302, {"Location": "https://evil.example/secret"})

        session.get = _redirect_away
        try:
            get_html_once("https://vov4.vov.vn/bahnar/start")
            raise AssertionError("off-host redirect was followed")
        except (NonPublicHost, RuntimeError):
            pass
        if any("evil.example" in url for url in requested):
            raise AssertionError("request was sent before the redirect target was validated")

        def _loop(url, timeout=30, allow_redirects=True, **_kwargs):
            if allow_redirects is not False:
                raise AssertionError("allow_redirects must be False")
            if url.endswith("/a"):
                return _FakeResponse(302, {"Location": "https://vov4.vov.vn/bahnar/b"})
            return _FakeResponse(302, {"Location": "https://vov4.vov.vn/bahnar/a"})

        session.get = _loop
        try:
            get_html_once("https://vov4.vov.vn/bahnar/a")
            raise AssertionError("redirect loop was not detected")
        except RuntimeError as exc:
            if "Redirect loop" not in str(exc):
                raise
    finally:
        session.get = saved_get
        globals()["robots_decision"] = saved_robots
        globals()["assert_public_http_host"] = saved_host

    # date-window coverage fail-closed, ordinary failures still SUCCESS
    if not section_coverage_incomplete({
        "pagination_cap_reached": True, "date_boundary_reached": False, "listing_exhausted": False,
    }):
        raise AssertionError("pagination cap before the date boundary must be incomplete")
    if section_coverage_incomplete({
        "pagination_cap_reached": False, "date_boundary_reached": True, "listing_exhausted": False,
    }):
        raise AssertionError("date boundary should count as covered")
    article_url = "https://vov4.vov.vn/bahnar/kotong-ang-lom-topol-thoi-su-xa-hoi/tin-1.vov4"
    dead_url = "https://vov4.vov.vn/bahnar/kotong-ang-lom-topol-thoi-su-xa-hoi/missing-1.vov4"
    rejected = _rejected_record("VOV4_DEAD", article_url, "33" * 32)
    complete_stats = {
        slug: {
            "pages_fetched": 2,
            "oldest_date_seen": "2025-12-01",
            "newest_date_seen": "2026-09-01",
            "pagination_cap_reached": False,
            "date_boundary_reached": True,
            "listing_exhausted": False,
            "listing_fetch_error": "",
        }
        for slug in ALL_SECTION_SLUGS
    }
    ordinary = {
        "records": [rejected],
        "articles": [{"article_url": article_url, "source_id": "VOV4_DEAD"}],
        "exclusions": [],
        "fetch_failures": [{"article_url": dead_url, "error_type": "HTTPError"}],
        "attempted_article_urls": [article_url, dead_url],
        "section_stats": complete_stats,
        "require_full_coverage": True,
    }
    ordinary_summary = build_summary(ordinary)
    if ordinary_summary["status"] != "SUCCESS_RQ2_VOV4_FULL_SOURCE_POOL":
        raise AssertionError(ordinary_summary["problems"])
    if ordinary_summary["n_technical_fail"] != 1 or ordinary_summary["n_fetch_failed"] != 1:
        raise AssertionError(ordinary_summary)
    if ordinary_summary.get("total_download_budget_exhausted") is not False:
        raise AssertionError(ordinary_summary.get("total_download_budget_exhausted"))
    saved_article_cache = dict(ARTICLE_CACHE)
    try:
        ARTICLE_CACHE.clear()
        scope_section = "kotong-ang-lom-topol-thoi-su-xa-hoi"
        in_scope_url = "https://vov4.vov.vn/bahnar/" + scope_section + "/in-scope.vov4"
        out_scope_url = "https://vov4.vov.vn/bahnar/" + scope_section + "/out-scope.vov4"
        for article_url, source_id_ in ((in_scope_url, "VOV4_IN_SCOPE"), (out_scope_url, "VOV4_OUT_SCOPE")):
            article_row = {
                "source_id": source_id_,
                "article_url": article_url,
                "source_section": scope_section,
                "article_date": "2026-02-01",
                "published_date_source": "metadata",
            }
            ARTICLE_CACHE[article_url] = {
                "kind": "article",
                "article_row": article_row,
                "media_candidates": [],
                "article_scope_error": "",
                "discovered_sections": {scope_section},
            }
        counted = build_summary({
            "records": [],
            "articles": [{"article_url": in_scope_url, "source_id": "VOV4_IN_SCOPE"}],
            "exclusions": [{"article_url": in_scope_url, "exclusion_reason": "NO_PUBLIC_MEDIA"}],
            "fetch_failures": [],
            "require_full_coverage": False,
        })
        if counted["n_articles_discovered_total"] != 2 or counted["n_articles_in_scope"] != 1:
            raise AssertionError((counted["n_articles_discovered_total"], counted["n_articles_in_scope"]))
        if counted["n_articles_discovered"] != counted["n_articles_discovered_total"]:
            raise AssertionError("n_articles_discovered did not count every discovered article")
    finally:
        ARTICLE_CACHE.clear()
        ARTICLE_CACHE.update(saved_article_cache)
    capped = dict(ordinary)
    capped["section_stats"] = dict(complete_stats)
    capped["section_stats"][ALL_SECTION_SLUGS[0]] = {
        "pages_fetched": MAX_PAGES_PER_SECTION,
        "oldest_date_seen": "2026-05-01",
        "newest_date_seen": "2026-09-01",
        "pagination_cap_reached": True,
        "date_boundary_reached": False,
        "listing_exhausted": False,
        "listing_fetch_error": "",
    }
    capped_summary = build_summary(capped)
    if capped_summary["status"] == "SUCCESS_RQ2_VOV4_FULL_SOURCE_POOL":
        raise AssertionError("pagination cap was not fail-closed")
    budget_case = dict(ordinary)
    budget_case["total_budget_exhausted"] = True
    budget_summary = build_summary(budget_case)
    if budget_summary["status"] == "SUCCESS_RQ2_VOV4_FULL_SOURCE_POOL":
        raise AssertionError("total download budget exhaustion was treated as SUCCESS")
    if not any("budget" in problem for problem in budget_summary["problems"]):
        raise AssertionError(budget_summary["problems"])
    if budget_summary.get("total_download_budget_exhausted") is not True:
        raise AssertionError("summary did not persist total_download_budget_exhausted")
    for column in ("download_attempted", "download_success"):
        if column not in SOURCE_POOL_COLUMNS:
            raise AssertionError(column)

    attempted = _rejected_record("VOV4_ATT", "https://vov4.vov.vn/bahnar/a/att.vov4", "55" * 32)
    attempted["download_attempted"] = True
    attempted["download_success"] = False
    succeeded = _rejected_record("VOV4_OKD", "https://vov4.vov.vn/bahnar/a/okd.vov4", "66" * 32)
    succeeded["download_attempted"] = True
    succeeded["download_success"] = True
    count_summary = build_summary({
        "records": [attempted, succeeded],
        "articles": [
            {"article_url": attempted["article_url"], "source_id": attempted["source_id"]},
            {"article_url": succeeded["article_url"], "source_id": succeeded["source_id"]},
        ],
        "exclusions": [],
        "fetch_failures": [],
        "require_full_coverage": False,
    })
    if count_summary["n_new_download_attempted"] != 2 or count_summary["n_new_download_success"] != 1:
        raise AssertionError(count_summary)
    if count_summary["n_new_downloaded"] != 1 or count_summary["n_new_technical_pass"] != 0:
        raise AssertionError(count_summary)

    saved_streamed = NETWORK_STATE["new_network_bytes_streamed_this_run"]
    NETWORK_STATE["new_network_bytes_streamed_this_run"] = MAX_TOTAL_DOWNLOAD_BYTES
    try:
        account_new_network_chunk(1)
        raise AssertionError("total budget did not raise TotalDownloadBudgetExceeded")
    except TotalDownloadBudgetExceeded:
        pass
    finally:
        NETWORK_STATE["new_network_bytes_streamed_this_run"] = saved_streamed

    saved_pick = pick_primary_media
    saved_match = match_pilot_media
    saved_download = download_new_media

    def _pick_one(_candidates):
        return {
            "media_url": "https://cdn.example.com/budget.mp3",
            "media_type": "audio",
            "media_extension": "mp3",
            "http_status": "200",
            "content_type": "audio/mpeg",
        }

    def _no_pilot(_media_hash):
        return None

    def _raise_budget(*_args, **_kwargs):
        raise TotalDownloadBudgetExceeded("MAX_TOTAL_DOWNLOAD_BYTES exceeded")

    globals()["pick_primary_media"] = _pick_one
    globals()["match_pilot_media"] = _no_pilot
    globals()["download_new_media"] = _raise_budget
    try:
        try:
            run_article_through_pipeline({
                "source_id": "VOV4_BUDGET",
                "article_url": "https://vov4.vov.vn/bahnar/kotong-ang-lom-topol-thoi-su-xa-hoi/budget.vov4",
                "article_date": "2026-06-01",
                "published_date_source": "metadata",
                "article_title": "Budget",
                "phase": "A",
                "source_section": "kotong-ang-lom-topol-thoi-su-xa-hoi",
            }, [{}], "")
            raise AssertionError("budget exhaustion became an ordinary technical failure")
        except TotalDownloadBudgetExceeded:
            pass
    finally:
        globals()["pick_primary_media"] = saved_pick
        globals()["match_pilot_media"] = saved_match
        globals()["download_new_media"] = saved_download

    if page_older_than_window([("2023-01-01", "body_fallback"), ("2026-06-01", "metadata")], "2026-01-01"):
        raise AssertionError("body_fallback date was allowed to mark a 2026 page as older")
    if page_older_than_window([("2023-01-01", "body_fallback")], "2026-01-01"):
        raise AssertionError("a page with only body_fallback dates was treated as older")

    def _pub_html(inner: str) -> bytes:
        return (
            "<html><body><div class=\"article-date\">01/01/2020</div><article>"
            + inner
            + "</article><p>Giấy phép cấp ngày 03/12/2016</p></body></html>"
        ).encode("utf-8")

    publication_cases = (
        ("<div class=\"article-date\">Chủ nhật, 06:00, 20/09/2026</div>", "2026-09-20"),
        ("<div class=\"time-published\"><span>Thứ năm, 13:57, 24/09/2026</span></div>", "2026-09-24"),
        ("<div class=\"video__time\">Thứ năm, 06:00, 22/09/2026</div>", "2026-09-22"),
    )
    for inner, expected_date in publication_cases:
        parsed_date, parsed_source = extract_published_date(BeautifulSoup(_pub_html(inner), "html.parser"))
        if parsed_date != expected_date or parsed_source != "time":
            raise AssertionError((inner, parsed_date, parsed_source))
    footer_only = extract_published_date(BeautifulSoup("<html><body><p>03/12/2016</p></body></html>", "html.parser"))
    if footer_only != ("2016-12-03", "body_fallback"):
        raise AssertionError(footer_only)
    saved_cache = dict(ARTICLE_CACHE)
    saved_sections = {slug: dict(state) for slug, state in SECTION_CRAWL_STATE.items()}
    try:
        ARTICLE_CACHE.clear()
        SECTION_CRAWL_STATE.clear()
        stale_section = "kotong-ang-lom-topol-thoi-su-xa-hoi"
        stale_url = "https://vov4.vov.vn/bahnar/" + stale_section + "/stale-date.vov4"
        stale_row = {
            "source_id": "VOV4_STALE_DATE",
            "article_url": stale_url,
            "source_section": stale_section,
            "article_date": "2026-02-01",
            "published_date_source": "body_fallback",
        }
        ARTICLE_CACHE[stale_url] = {
            "kind": "article",
            "article_row": stale_row,
            "media_candidates": [],
            "article_scope_error": "",
            "discovered_sections": {stale_section},
        }
        ARTICLE_CACHE["id:VOV4_STALE_DATE"] = ARTICLE_CACHE[stale_url]
        capped = blank_section_state()
        capped["next_page"] = 400
        capped["pages_fetched"] = 400
        capped["pagination_cap_reached"] = True
        SECTION_CRAWL_STATE[stale_section] = capped
        if drop_untrusted_cached_dates() != 1:
            raise AssertionError("untrusted cached date was kept")
        if stale_url in ARTICLE_CACHE or "id:VOV4_STALE_DATE" in ARTICLE_CACHE:
            raise AssertionError("stale article cache entry remained")
        if SECTION_CRAWL_STATE[stale_section]["next_page"] != 0 or SECTION_CRAWL_STATE[stale_section]["pagination_cap_reached"]:
            raise AssertionError(SECTION_CRAWL_STATE[stale_section])
    finally:
        ARTICLE_CACHE.clear()
        ARTICLE_CACHE.update(saved_cache)
        SECTION_CRAWL_STATE.clear()
        SECTION_CRAWL_STATE.update(saved_sections)
    if not page_older_than_window([("2025-12-01", "time")], "2026-01-01"):
        raise AssertionError("trusted dates before the window were not treated as older")
    if page_older_than_window([("2025-12-01", "time"), ("2026-02-01", "metadata")], "2026-01-01"):
        raise AssertionError("a mixed trusted page was treated as entirely older")
    if page_proves_older_boundary(
        2, [("2025-01-01", "metadata"), ("2024-06-01", "body_fallback")], 0, "2026-01-01"
    ):
        raise AssertionError("body_fallback was allowed to prove an older boundary")
    if page_proves_older_boundary(1, [("2025-01-01", "time")], 1, "2026-01-01"):
        raise AssertionError("a fetch failure was allowed to prove an older boundary")
    if not page_proves_older_boundary(
        2, [("2025-06-01", "metadata"), ("2025-01-01", "time")], 0, "2026-01-01"
    ):
        raise AssertionError("a fully trusted old page did not prove the boundary")
    if page_proves_older_boundary(
        2, [("2025-06-01", "metadata"), ("2026-02-01", "time")], 0, "2026-01-01"
    ):
        raise AssertionError("a trusted date inside the window proved an older boundary")

    news_for_pager = "kotong-ang-lom-topol-thoi-su-xa-hoi"
    empty_with_next = (
        '<html><body><div class="view-content"></div>'
        '<ul><li class="pager next"><a href="?page=1">next</a></li></ul></body></html>'
    ).encode("utf-8")
    parsed_empty = inspect_listing_page(empty_with_next, 0, news_for_pager)
    if parsed_empty["status"] != "LISTING_PARSE_EMPTY" or parsed_empty["links"]:
        raise AssertionError(parsed_empty)
    if not section_coverage_incomplete({
        "listing_parse_empty": True, "listing_exhausted": True, "date_boundary_reached": True,
    }):
        raise AssertionError("parse-empty listing was treated as covered")
    exhausted_html = (
        '<html><body><div class="view-content"><a href="/bahnar/' + news_for_pager + '/tin-cuoi.vov4">item</a></div></body></html>'
    ).encode("utf-8")
    if inspect_listing_page(exhausted_html, 0, news_for_pager)["status"] != "LISTING_EXHAUSTED":
        raise AssertionError("links without a next pager were not exhausted")
    if inspect_listing_page(b"<html><body><p>end</p></body></html>", 0, news_for_pager)["status"] != "LISTING_SCOPE_UNVERIFIED":
        raise AssertionError("HTML without a main listing container was treated as scoped evidence")
    sidebar_empty = (
        '<html><body><aside><div class="view-empty">other block</div></aside>'
        '<div id="content"></div></body></html>'
    ).encode("utf-8")
    if inspect_listing_page(sidebar_empty, 0, news_for_pager)["status"] != "LISTING_SCOPE_UNVERIFIED":
        raise AssertionError("an unrelated view-empty marker exhausted the target listing")
    if listing_has_verified_end_marker(sidebar_empty):
        raise AssertionError("an unscoped view-empty marker was accepted")
    if not section_coverage_incomplete({
        "listing_empty_unverified": True, "listing_exhausted": True, "date_boundary_reached": True,
    }):
        raise AssertionError("unverified empty listing was treated as covered")
    for field in (
        "listing_parse_empty_page", "listing_parse_empty_attempts",
        "listing_empty_unverified_page", "listing_empty_unverified_attempts",
    ):
        if field not in SECTION_STAT_FIELDS:
            raise AssertionError(field)
    if "listing_empty_unverified_attempts" not in section_stats_snapshot()[ALL_SECTION_SLUGS[0]]:
        raise AssertionError("summary section stats omit unverified-empty retry state")
    failed_page = 5
    unverified_state = blank_section_state()
    unverified_state["next_page"] = failed_page + 1
    note_listing_empty_unverified(unverified_state, failed_page)
    if unverified_state["listing_empty_unverified_page"] != failed_page or unverified_state["next_page"] != failed_page:
        raise AssertionError(unverified_state)
    if prepare_listing_empty_unverified_retry(unverified_state) != "retry" or unverified_state["next_page"] != failed_page:
        raise AssertionError("resume walked to page N+1 after LISTING_EMPTY_UNVERIFIED")
    unverified_state["listing_empty_unverified_attempts"] = LISTING_PARSE_EMPTY_MAX_ATTEMPTS
    unverified_state["next_page"] = failed_page + 1
    if prepare_listing_empty_unverified_retry(unverified_state) != "incomplete":
        raise AssertionError("spent unverified-empty retries were treated as retryable")
    if unverified_state["next_page"] != failed_page or not section_coverage_incomplete(unverified_state):
        raise AssertionError("spent retries advanced past the failed page")
    if listing_has_next_page(b'<html><body><a href="?page=10">10</a></body></html>', 1):
        raise AssertionError("page=10 was treated as the next page after page=1")
    if listing_has_next_page(b'<html><body><a rel="next" href="?page=2">next</a></body></html>', 1):
        raise AssertionError("an unscoped rel=next link was treated as pager evidence")
    retry_state = blank_section_state()
    retry_state["next_page"] = 4
    note_listing_parse_empty(retry_state, 3)
    if retry_state["listing_parse_empty_page"] != 3 or retry_state["next_page"] != 3:
        raise AssertionError(retry_state)
    if retry_state["listing_parse_empty_attempts"] != 1:
        raise AssertionError(retry_state)
    if prepare_listing_parse_empty_retry(retry_state) != "retry" or retry_state["next_page"] != 3:
        raise AssertionError("parse-empty page was not rewound for a later session")
    clear_listing_parse_empty(retry_state)
    if retry_state["listing_parse_empty"] or prepare_listing_parse_empty_retry(retry_state) != "ok":
        raise AssertionError("a successful parse did not clear listing_parse_empty")
    spent = blank_section_state()
    spent["listing_parse_empty"] = True
    spent["listing_parse_empty_page"] = 2
    spent["listing_parse_empty_attempts"] = LISTING_PARSE_EMPTY_MAX_ATTEMPTS
    if prepare_listing_parse_empty_retry(spent) != "incomplete" or not section_coverage_incomplete(spent):
        raise AssertionError("exhausted parse-empty retries were treated as covered")

    retry_part = OUT_DIR / "qa" / "_synthetic_retry.part"
    saved_stream = stream_media_to_file
    saved_sleep = time.sleep
    stream_calls = {"n": 0}

    def _no_sleep(_delay):
        return None

    def _flaky_stream(_url, dest_part):
        stream_calls["n"] += 1
        dest_part.parent.mkdir(parents=True, exist_ok=True)
        dest_part.write_bytes(b"partial")
        if stream_calls["n"] < 3:
            raise TransientHTTP("connection reset")
        return {"http_status": "200", "content_type": "audio/mpeg", "actual_file_bytes": 7}

    def _budget_stream(_url, dest_part):
        stream_calls["n"] += 1
        dest_part.write_bytes(b"partial")
        raise TotalDownloadBudgetExceeded("budget")

    globals()["time"].sleep = _no_sleep
    try:
        stream_calls["n"] = 0
        globals()["stream_media_to_file"] = _budget_stream
        retry_part.write_bytes(b"old")
        try:
            stream_media_with_retries("https://cdn.example.com/a.mp3", retry_part)
            raise AssertionError("budget exhaustion was retried")
        except TotalDownloadBudgetExceeded:
            pass
        if stream_calls["n"] != 1:
            raise AssertionError(stream_calls["n"])
        if retry_part.exists():
            raise AssertionError("partial file survived a budget failure")
        stream_calls["n"] = 0
        globals()["stream_media_to_file"] = _flaky_stream
        meta = stream_media_with_retries("https://cdn.example.com/a.mp3", retry_part)
        if stream_calls["n"] != 3 or meta.get("http_status") != "200":
            raise AssertionError((stream_calls, meta))
    finally:
        globals()["stream_media_to_file"] = saved_stream
        time.sleep = saved_sleep
        retry_part.unlink(missing_ok=True)

    verify_calls = {"n": 0}
    saved_follow = _follow_verify_hop
    saved_verify_sleep = time.sleep

    def _verify_no_sleep(_delay):
        return None

    def _permanent_verify(url):
        verify_calls["n"] += 1
        raise RuntimeError("Redirect loop or invalid URL: " + url)

    def _transient_verify(url):
        verify_calls["n"] += 1
        if verify_calls["n"] < 3:
            raise requests.exceptions.ConnectionError("reset")
        return {
            "outcome": "ok",
            "final_url": url,
            "http_status": "200",
            "content_type": "audio/mpeg",
            "content_length": "1",
            "notes": "",
        }

    globals()["_follow_verify_hop"] = _permanent_verify
    time.sleep = _verify_no_sleep
    try:
        verify_calls["n"] = 0
        verify_media_url("https://cdn.example.com/permanent-redirect.mp3")
        if verify_calls["n"] != 1:
            raise AssertionError(verify_calls["n"])
        verify_calls["n"] = 0
        globals()["_follow_verify_hop"] = _transient_verify
        verify_media_url("https://cdn.example.com/transient-reset.mp3")
        if verify_calls["n"] != 3:
            raise AssertionError(verify_calls["n"])
    finally:
        globals()["_follow_verify_hop"] = saved_follow
        time.sleep = saved_verify_sleep
        VERIFY_CACHE.pop("https://cdn.example.com/permanent-redirect.mp3", None)
        VERIFY_CACHE.pop("https://cdn.example.com/transient-reset.mp3", None)

    checkpoint_dir = OUT_DIR / "qa" / "_synthetic_checkpoint"
    checkpoint_path = checkpoint_dir / "state.json"
    saved_cache = dict(ARTICLE_CACHE)
    saved_sections = {slug: dict(state) for slug, state in SECTION_CRAWL_STATE.items()}
    saved_owned = dict(OWNED_MEDIA)
    saved_network = dict(NETWORK_STATE)
    saved_processed = set(RUN_STATE["processed_source_ids"])
    saved_budget_flag = RUN_STATE["total_budget_exhausted"]
    saved_phases = list(ACCUMULATED_PHASES)
    try:
        ARTICLE_CACHE.clear()
        SECTION_CRAWL_STATE.clear()
        OWNED_MEDIA.clear()
        ACCUMULATED_PHASES.clear()
        sample_url = "https://vov4.vov.vn/bahnar/kotong-ang-lom-topol-thoi-su-xa-hoi/cached.vov4"
        sample_row = {
            "source_id": "VOV4_CACHE",
            "article_url": sample_url,
            "article_date": "2026-03-01",
            "published_date_source": "metadata",
            "article_title": "Cached",
            "source_section": "kotong-ang-lom-topol-thoi-su-xa-hoi",
            "phase": "A",
        }
        ARTICLE_CACHE[sample_url] = {
            "kind": "article",
            "article_row": sample_row,
            "media_candidates": [],
            "article_scope_error": "",
            "discovered_sections": {"kotong-ang-lom-topol-thoi-su-xa-hoi"},
        }
        ARTICLE_CACHE["id:VOV4_CACHE"] = ARTICLE_CACHE[sample_url]
        SECTION_CRAWL_STATE["kotong-ang-lom-topol-thoi-su-xa-hoi"] = blank_section_state()
        SECTION_CRAWL_STATE["kotong-ang-lom-topol-thoi-su-xa-hoi"]["next_page"] = 3
        OWNED_MEDIA["ab" * 32] = "VOV4_A"
        NETWORK_STATE["new_network_bytes_streamed_this_run"] = 12
        RUN_STATE["processed_source_ids"] = {"VOV4_CACHE"}
        RUN_STATE["total_budget_exhausted"] = False
        save_checkpoint(checkpoint_path)
        ARTICLE_CACHE.clear()
        SECTION_CRAWL_STATE.clear()
        OWNED_MEDIA.clear()
        NETWORK_STATE["new_network_bytes_streamed_this_run"] = 0
        RUN_STATE["processed_source_ids"] = set()
        if not load_checkpoint(checkpoint_path):
            raise AssertionError("checkpoint was not loaded")
        restored = ARTICLE_CACHE[sample_url]
        if restored["article_row"]["source_id"] != "VOV4_CACHE":
            raise AssertionError(restored)
        if "kotong-ang-lom-topol-thoi-su-xa-hoi" not in restored["discovered_sections"]:
            raise AssertionError(restored["discovered_sections"])
        if ARTICLE_CACHE["id:VOV4_CACHE"] is not restored:
            raise AssertionError("source_id alias was not restored")
        if SECTION_CRAWL_STATE["kotong-ang-lom-topol-thoi-su-xa-hoi"]["next_page"] != 3:
            raise AssertionError(SECTION_CRAWL_STATE)
        if OWNED_MEDIA.get("ab" * 32) != "VOV4_A":
            raise AssertionError(OWNED_MEDIA)
        if NETWORK_STATE["new_network_bytes_streamed_this_run"] != 12:
            raise AssertionError(NETWORK_STATE)
        saved_doc = json.loads(checkpoint_path.read_text(encoding="utf-8"))
        if saved_doc.get("checkpoint_compatibility_sha256") != checkpoint_compatibility_sha256():
            raise AssertionError("same signature was not stored")
        if not load_checkpoint(checkpoint_path):
            raise AssertionError("same signature was not accepted on resume")

        def _reject_checkpoint(mutate) -> None:
            variant_doc = json.loads(checkpoint_path.read_text(encoding="utf-8"))
            mutate(variant_doc["checkpoint_compatibility"])
            variant_doc["checkpoint_compatibility_sha256"] = hashlib.sha256(
                canonical_json_bytes(variant_doc["checkpoint_compatibility"])
            ).hexdigest()
            variant_path = checkpoint_dir / "variant.json"
            variant_path.write_text(json.dumps(variant_doc), encoding="utf-8")
            before_ids = set(RUN_STATE["processed_source_ids"])
            try:
                load_checkpoint(variant_path)
                raise AssertionError("incompatible checkpoint was loaded")
            except CheckpointIncompatible:
                pass
            if set(RUN_STATE["processed_source_ids"]) != before_ids:
                raise AssertionError("rejected checkpoint mutated resume state")

        _reject_checkpoint(lambda payload: payload["upstream_fingerprints"].__setitem__("nb09_summary_sha256", "0" * 64))
        _reject_checkpoint(lambda payload: payload["date_windows"]["phase_a"].__setitem__("end", "2026-01-02"))
        _reject_checkpoint(lambda payload: payload["section_content_class"].__setitem__(
            "kotong-ang-lom-topol-thoi-su-xa-hoi", "music"
        ))
        _reject_checkpoint(lambda payload: payload.__setitem__("eligibility_schema_version", "9.9"))
        _reject_checkpoint(lambda payload: payload.__setitem__("max_single_file_bytes", 1))
        _reject_checkpoint(lambda payload: payload.__setitem__("max_pages_per_section", 1600))
        _reject_checkpoint(lambda payload: payload.__setitem__("stop_after_empty_pages", 9))
        _reject_checkpoint(lambda payload: payload.__setitem__("listing_parse_empty_max_attempts", 99))
        _reject_checkpoint(lambda payload: payload.__setitem__("crawl_pipeline_schema_version", "0"))

        raw_path = checkpoint_dir / "raw.bin"
        wav_path = checkpoint_dir / "audio.wav"
        raw_path.write_bytes(b"raw-bytes")
        pcm = b"\x01\x00" * 1600
        with wave.open(str(wav_path), "wb") as handle:
            handle.setnchannels(1)
            handle.setsampwidth(2)
            handle.setframerate(16000)
            handle.writeframes(pcm)
        keep_url = "https://vov4.vov.vn/bahnar/kotong-ang-lom-topol-thoi-su-xa-hoi/keep.vov4"
        keep_hash = "77" * 32
        keep_record = {
            "source_id": "VOV4_KEEP",
            "article_url": keep_url,
            "technical_status": "PASS",
            "source_level_status": "ELIGIBLE_SOURCE",
            "duration_seconds": 0.1,
            "raw_local_path": to_project_relative(raw_path),
            "wav_local_path": to_project_relative(wav_path),
            "video_container_local_path": "",
            "raw_sha256": sha256_file(raw_path),
            "wav_sha256": sha256_file(wav_path),
            "pcm16_sha256": hashlib.sha256(pcm).hexdigest(),
            "media_url_sha256": keep_hash,
            "artifact_origin": "nb10_full",
        }
        if verify_pass_artifact(keep_record):
            raise AssertionError(verify_pass_artifact(keep_record))
        pilot = match_pilot_media(next(iter(PILOT_BY_MEDIA_HASH)))
        pilot_hash = pilot["media_url_sha256"]
        pilot_url = pilot.get("article_url") or "https://vov4.vov.vn/bahnar/a/pilot-reuse.vov4"
        pilot_record = {
            "source_id": pilot["source_id"],
            "article_url": pilot_url,
            "technical_status": "PASS",
            "source_level_status": "REJECTED_SOURCE",
            "duration_seconds": float(pilot["duration_seconds"]),
            "raw_local_path": pilot["raw_local_path"],
            "wav_local_path": pilot["wav_local_path"],
            "video_container_local_path": pilot.get("video_container_local_path") or "",
            "raw_sha256": pilot["raw_sha256"],
            "wav_sha256": pilot["wav_sha256"],
            "pcm16_sha256": pilot["pcm16_sha256"],
            "media_url_sha256": pilot_hash,
            "artifact_origin": "nb09_pilot",
        }
        if verify_pass_artifact(pilot_record):
            raise AssertionError(verify_pass_artifact(pilot_record))
        failed_record = {
            "source_id": "VOV4_FAIL",
            "article_url": "https://vov4.vov.vn/bahnar/a/fail.vov4",
            "technical_status": "TECHNICAL_FAIL",
            "source_level_status": "REJECTED_SOURCE",
            "duration_seconds": 0.0,
            "media_url_sha256": "88" * 32,
            "artifact_origin": "nb10_full",
        }
        ACCUMULATED_PHASES.clear()
        ACCUMULATED_PHASES.append({
            "phase_label": "A",
            "start": PHASE_A_START,
            "end": PHASE_A_END,
            "articles": [
                {"source_id": "VOV4_KEEP", "article_url": keep_url},
                {"source_id": pilot["source_id"], "article_url": pilot_url},
                {"source_id": "VOV4_FAIL", "article_url": failed_record["article_url"]},
            ],
            "records": [keep_record, pilot_record, failed_record],
            "exclusions": [],
            "fetch_failures": [],
            "eligible_source_hours": 0.0,
            "attempted_article_urls": [keep_url, pilot_url, failed_record["article_url"]],
        })
        OWNED_MEDIA[keep_hash] = "VOV4_KEEP"
        OWNED_MEDIA[pilot_hash] = pilot["source_id"]
        RUN_STATE["processed_source_ids"].update({"VOV4_KEEP", pilot["source_id"], "VOV4_FAIL"})
        save_checkpoint(checkpoint_path)
        RUN_STATE["processed_source_ids"] = set()
        OWNED_MEDIA.clear()
        if not load_checkpoint(checkpoint_path):
            raise AssertionError("valid restored artifacts were not accepted")
        if "VOV4_KEEP" not in RUN_STATE["processed_source_ids"] or OWNED_MEDIA.get(keep_hash) != "VOV4_KEEP":
            raise AssertionError("valid restored artifact was not kept")
        if pilot["source_id"] not in RUN_STATE["processed_source_ids"] or OWNED_MEDIA.get(pilot_hash) != pilot["source_id"]:
            raise AssertionError("valid Notebook 09 reuse was not kept")
        if "VOV4_FAIL" not in RUN_STATE["processed_source_ids"]:
            raise AssertionError("non-PASS completed record was dropped")

        corrupt_doc = json.loads(checkpoint_path.read_text(encoding="utf-8"))
        for phase in corrupt_doc["phases"]:
            for record in phase["records"]:
                if record["source_id"] == pilot["source_id"]:
                    record["pcm16_sha256"] = "ab" * 32
        corrupt_path = checkpoint_dir / "corrupt.json"
        corrupt_path.write_text(json.dumps(corrupt_doc), encoding="utf-8")
        load_checkpoint(corrupt_path)
        if pilot["source_id"] in RUN_STATE["processed_source_ids"] or pilot_hash in OWNED_MEDIA:
            raise AssertionError("wrong PCM hash was trusted")
        if "VOV4_KEEP" not in RUN_STATE["processed_source_ids"]:
            raise AssertionError("a valid artifact was dropped alongside a corrupt one")
        kept_ids = [record["source_id"] for record in ACCUMULATED_PHASES[0]["records"]]
        if pilot["source_id"] in kept_ids:
            raise AssertionError("corrupt PASS record stayed in the phase")

        wav_path.unlink()
        load_checkpoint(checkpoint_path)
        if "VOV4_KEEP" in RUN_STATE["processed_source_ids"] or keep_hash in OWNED_MEDIA:
            raise AssertionError("missing WAV stayed processed")
        if any(record["source_id"] == "VOV4_KEEP" for record in ACCUMULATED_PHASES[0]["records"]):
            raise AssertionError("missing WAV record stayed in the phase")
        if pilot["source_id"] not in RUN_STATE["processed_source_ids"]:
            raise AssertionError("Notebook 09 reuse was dropped because another file was missing")

        saved_total_budget = MAX_TOTAL_DOWNLOAD_BYTES
        try:
            globals()["MAX_TOTAL_DOWNLOAD_BYTES"] = 1024 ** 3
            OWNED_MEDIA["cc" * 32] = "VOV4_BUDGET_SRC"
            finished_ids = set(RUN_STATE["processed_source_ids"])
            note_total_download_budget_exhausted("VOV4_BUDGET_SRC", "cc" * 32)
            if "VOV4_BUDGET_SRC" in RUN_STATE["processed_source_ids"]:
                raise AssertionError("1 GiB exhaustion marked the source processed")
            if "cc" * 32 in OWNED_MEDIA:
                raise AssertionError("interrupted media stayed owned")
            if not finished_ids.issubset(RUN_STATE["processed_source_ids"]):
                raise AssertionError("finished downloads were discarded when the budget was exhausted")
            NETWORK_STATE["new_network_bytes_streamed_this_run"] = 1024 ** 3
            save_checkpoint(checkpoint_path)
            stored_budget = json.loads(checkpoint_path.read_text(encoding="utf-8"))
            if stored_budget.get("configured_total_download_budget_bytes") != 1024 ** 3:
                raise AssertionError(stored_budget.get("configured_total_download_budget_bytes"))
            if not load_checkpoint(checkpoint_path):
                raise AssertionError("exhausted budget checkpoint did not load")
            if "VOV4_BUDGET_SRC" in RUN_STATE["processed_source_ids"]:
                raise AssertionError("reloaded exhaustion marked the source processed")
            globals()["MAX_TOTAL_DOWNLOAD_BYTES"] = 3 * 1024 ** 3
            apply_resumed_download_budget()
            if RUN_STATE["total_budget_exhausted"]:
                raise AssertionError("3 GiB resume did not clear exhaustion")
            if "VOV4_BUDGET_SRC" in RUN_STATE["processed_source_ids"]:
                raise AssertionError("3 GiB resume did not leave the source retryable")
            globals()["MAX_TOTAL_DOWNLOAD_BYTES"] = 1024 ** 3
            note_total_download_budget_exhausted("VOV4_BUDGET_SRC")
            NETWORK_STATE["new_network_bytes_streamed_this_run"] = 1024 ** 3
            try:
                apply_resumed_download_budget()
                raise AssertionError("resume at or below streamed bytes was accepted")
            except TotalDownloadBudgetExceeded as exc:
                if "BAHNAR_VOV4_MAX_TOTAL_DOWNLOAD_BYTES" not in str(exc):
                    raise
        finally:
            globals()["MAX_TOTAL_DOWNLOAD_BYTES"] = saved_total_budget
    finally:
        ARTICLE_CACHE.clear()
        ARTICLE_CACHE.update(saved_cache)
        SECTION_CRAWL_STATE.clear()
        SECTION_CRAWL_STATE.update(saved_sections)
        OWNED_MEDIA.clear()
        OWNED_MEDIA.update(saved_owned)
        NETWORK_STATE.clear()
        NETWORK_STATE.update(saved_network)
        RUN_STATE["processed_source_ids"] = saved_processed
        RUN_STATE["total_budget_exhausted"] = saved_budget_flag
        ACCUMULATED_PHASES.clear()
        ACCUMULATED_PHASES.extend(saved_phases)
        shutil.rmtree(checkpoint_dir, ignore_errors=True)

    # download budget overrides
    saved_single = os.environ.get("BAHNAR_VOV4_MAX_SINGLE_FILE_BYTES")
    try:
        os.environ["BAHNAR_VOV4_MAX_SINGLE_FILE_BYTES"] = "0"
        try:
            positive_int_env("BAHNAR_VOV4_MAX_SINGLE_FILE_BYTES", 10)
            raise AssertionError("non-positive budget was accepted")
        except RuntimeError:
            pass
        os.environ["BAHNAR_VOV4_MAX_SINGLE_FILE_BYTES"] = "4096"
        if positive_int_env("BAHNAR_VOV4_MAX_SINGLE_FILE_BYTES", 10) != 4096:
            raise AssertionError("positive budget override was ignored")
    finally:
        if saved_single is None:
            os.environ.pop("BAHNAR_VOV4_MAX_SINGLE_FILE_BYTES", None)
        else:
            os.environ["BAHNAR_VOV4_MAX_SINGLE_FILE_BYTES"] = saved_single

    # optional native spot-check file, keyed by source_id + media hash
    spot_dir = OUT_DIR / "qa" / "_synthetic_spotcheck"
    spot_dir.mkdir(parents=True, exist_ok=True)
    spot_path = spot_dir / "native_spotcheck.csv"
    try:
        spot_path.write_text(
            "source_id,media_url_sha256,native_spotcheck_status,native_spotcheck_notes,reviewer\n"
            "VOV4_SPOT,abcd,PILOT_SPOTCHECK_BAHNAR_LIKELY,predominantly Bahnar,bahnar-speaker\n",
            encoding="utf-8",
        )
        loaded = load_native_spotcheck(spot_path)
        spot_record = {"source_id": "VOV4_SPOT", "media_url_sha256": "ABCD"}
        saved_map = dict(NATIVE_SPOTCHECK_BY_KEY)
        NATIVE_SPOTCHECK_BY_KEY.clear()
        NATIVE_SPOTCHECK_BY_KEY.update(loaded)
        try:
            apply_native_spotcheck(spot_record)
        finally:
            NATIVE_SPOTCHECK_BY_KEY.clear()
            NATIVE_SPOTCHECK_BY_KEY.update(saved_map)
        if spot_record["native_spotcheck_status"] != "PILOT_SPOTCHECK_BAHNAR_LIKELY":
            raise AssertionError(spot_record)
        missing = {"source_id": "VOV4_NONE", "media_url_sha256": "ff" * 32}
        apply_native_spotcheck(missing)
        if missing["native_spotcheck_status"] != "NOT_PERFORMED":
            raise AssertionError(missing)
        spot_path.write_text(
            "source_id,media_url_sha256,native_spotcheck_status,native_spotcheck_notes,reviewer\n"
            "VOV4_SPOT,abcd,PILOT_SPOTCHECK_BAHNAR_LIKELY,a,bahnar-speaker\n"
            "VOV4_SPOT,abcd,PILOT_SPOTCHECK_BAHNAR_LIKELY,b,bahnar-speaker\n",
            encoding="utf-8",
        )
        try:
            load_native_spotcheck(spot_path)
            raise AssertionError("duplicate spot-check key was accepted")
        except RuntimeError:
            pass
    finally:
        shutil.rmtree(spot_dir, ignore_errors=True)

    # PASS artifact integrity: hashes must match the files on disk
    synth_dir = OUT_DIR / "qa" / "_synthetic_integrity"
    synth_dir.mkdir(parents=True, exist_ok=True)
    try:
        raw_path = synth_dir / "raw.bin"
        wav_path = synth_dir / "audio.wav"
        raw_path.write_bytes(b"raw-bytes")
        pcm = b"\x01\x00" * 1600
        with wave.open(str(wav_path), "wb") as handle:
            handle.setnchannels(1)
            handle.setsampwidth(2)
            handle.setframerate(16000)
            handle.writeframes(pcm)
        good = {
            "source_id": "VOV4_SYNTH",
            "article_url": "https://vov4.vov.vn/bahnar/a/synth.vov4",
            "media_url_sha256": "44" * 32,
            "technical_status": "PASS",
            "source_level_status": "ELIGIBLE_SOURCE",
            "content_class": "news",
            "conflicting_language_evidence": False,
            "duration_seconds": 0.1,
            "raw_local_path": to_project_relative(raw_path),
            "wav_local_path": to_project_relative(wav_path),
            "video_container_local_path": "",
            "raw_sha256": sha256_file(raw_path),
            "wav_sha256": sha256_file(wav_path),
            "pcm16_sha256": hashlib.sha256(pcm).hexdigest(),
        }
        if verify_pass_artifact(good):
            raise AssertionError(verify_pass_artifact(good))
        bad_pcm = dict(good)
        bad_pcm["pcm16_sha256"] = "ab" * 32
        if not verify_pass_artifact(bad_pcm):
            raise AssertionError("PCM mismatch was not detected")
        video_path = synth_dir / "clip.mp4"
        video_path.write_bytes(b"video-container-bytes")
        with_video = dict(good)
        with_video["video_container_local_path"] = to_project_relative(video_path)
        with_video["video_container_sha256"] = sha256_file(video_path)
        with_video["video_container_bytes"] = video_path.stat().st_size
        if verify_pass_artifact(with_video):
            raise AssertionError(verify_pass_artifact(with_video))
        bad_video = dict(with_video)
        bad_video["video_container_sha256"] = "cd" * 32
        if not verify_pass_artifact(bad_video):
            raise AssertionError("video container SHA mismatch was not detected")
        for column in ("video_container_sha256", "video_container_bytes"):
            if column not in SOURCE_POOL_COLUMNS:
                raise AssertionError(column)
    finally:
        shutil.rmtree(synth_dir, ignore_errors=True)

    contract = build_contract()
    if contract["download_budget"]["max_single_file_bytes"] != MAX_SINGLE_FILE_BYTES:
        raise AssertionError("contract is missing the configured download budget")
    for key in (
        "nb07_candidates_csv_sha256",
        "nb08_media_probe_csv_sha256",
        "nb09_pilot_manifest_csv_sha256",
        "nb09_technical_qa_csv_sha256",
        "nb09_summary_sha256",
    ):
        if not is_sha256_hex(contract["upstream_fingerprints"].get(key, "")):
            raise AssertionError(key)
    if "audio/mpeg" not in contract["supported_media_types"] or "video/mp4" not in contract["supported_media_types"]:
        raise AssertionError(contract["supported_media_types"])
    if contract["forbidden_code_pattern_hits"]:
        raise AssertionError(contract["forbidden_code_pattern_hits"])
    if contract.get("media_owner_rule") != "phase_priority_then_smallest_source_id":
        raise AssertionError(contract.get("media_owner_rule"))
    if contract.get("date_boundary_uses_only") != ["metadata", "time"]:
        raise AssertionError(contract.get("date_boundary_uses_only"))
    if not contract["download_budget"].get("total_download_budget_is_systemic"):
        raise AssertionError(contract["download_budget"])
    saved_stream = stream_media_with_retries
    saved_run_cmd = run_cmd
    orphan_stem = stem_for("VOV4_ORPHAN_VIDEO", "https://cdn.example.com/orphan.mp4")
    orphan_video = RAW_DIR / (orphan_stem + ".mp4")
    orphan_audio = RAW_DIR / (orphan_stem + ".m4a")
    try:
        def _write_video_part(url: str, part: Path) -> dict:
            part.parent.mkdir(parents=True, exist_ok=True)
            part.write_bytes(b"orphan-mp4")
            return {"http_status": "200", "content_type": "video/mp4", "actual_file_bytes": 10}

        def _fail_extract(args):
            return 1, b"", b"extract failed"

        globals()["stream_media_with_retries"] = _write_video_part
        globals()["run_cmd"] = _fail_extract
        try:
            download_new_media(
                "VOV4_ORPHAN_VIDEO",
                "https://cdn.example.com/orphan.mp4",
                "video",
                "mp4",
                "video/mp4",
            )
            raise AssertionError("failed audio extraction was treated as success")
        except RuntimeError:
            pass
        if orphan_video.exists() or orphan_audio.exists():
            raise AssertionError("failed video extraction left an untracked container")
    finally:
        globals()["stream_media_with_retries"] = saved_stream
        globals()["run_cmd"] = saved_run_cmd
        orphan_video.unlink(missing_ok=True)
        orphan_audio.unlink(missing_ok=True)
        part = RAW_DIR / (orphan_stem + ".mp4.part")
        part.unlink(missing_ok=True)
    # orphan checkpoint repair + monotonic page-cap resume
    repair_dir = OUT_DIR / "qa" / "_synthetic_repair"
    repair_path = repair_dir / "state.json"
    saved_cache_r = dict(ARTICLE_CACHE)
    saved_sections_r = {slug: dict(state) for slug, state in SECTION_CRAWL_STATE.items()}
    saved_owned_r = dict(OWNED_MEDIA)
    saved_processed_r = set(RUN_STATE["processed_source_ids"])
    saved_budget_r = RUN_STATE["total_budget_exhausted"]
    saved_phases_r = list(ACCUMULATED_PHASES)
    saved_pages_global = MAX_PAGES_PER_SECTION
    try:
        ARTICLE_CACHE.clear()
        SECTION_CRAWL_STATE.clear()
        OWNED_MEDIA.clear()
        ACCUMULATED_PHASES.clear()
        RUN_STATE["processed_source_ids"] = set()
        RUN_STATE["total_budget_exhausted"] = False
        url_a = "https://vov4.vov.vn/bahnar/kotong-ang-lom-topol-thoi-su-xa-hoi/repair-a.vov4"
        url_b = "https://vov4.vov.vn/bahnar/kotong-ang-lom-topol-thoi-su-xa-hoi/repair-b.vov4"
        url_c = "https://vov4.vov.vn/bahnar/kotong-ang-lom-topol-thoi-su-xa-hoi/repair-c.vov4"
        row_a = {"source_id": "VOV4_REPAIR_A", "article_url": url_a, "article_date": "2026-02-01"}
        row_b = {"source_id": "VOV4_REPAIR_B", "article_url": url_b, "article_date": "2026-02-02"}
        row_c = {"source_id": "VOV4_REPAIR_C", "article_url": url_c, "article_date": "2026-02-03"}
        ARTICLE_CACHE[url_c] = {
            "kind": "article",
            "article_row": dict(row_c),
            "media_candidates": [],
            "article_scope_error": "",
            "discovered_sections": {"kotong-ang-lom-topol-thoi-su-xa-hoi"},
        }
        media_c = "cd" * 32
        OWNED_MEDIA[media_c] = "VOV4_REPAIR_C"
        OWNED_MEDIA["aa" * 32] = "VOV4_REPAIR_A"
        RUN_STATE["processed_source_ids"] = {"VOV4_REPAIR_A", "VOV4_REPAIR_B", "VOV4_REPAIR_C"}
        ACCUMULATED_PHASES.append({
            "phase_label": "A",
            "start": PHASE_A_START,
            "end": PHASE_A_END,
            "articles": [dict(row_a), dict(row_b), dict(row_c)],
            "records": [{
                "source_id": "VOV4_REPAIR_A",
                "article_url": url_a,
                "technical_status": "FAIL",
                "source_level_status": "REJECTED_SOURCE",
                "media_url_sha256": "aa" * 32,
                "duration_seconds": 1.0,
            }],
            "exclusions": [{
                "source_id": "VOV4_REPAIR_B",
                "article_url": url_b,
                "exclusion_reason": "NO_PUBLIC_MEDIA",
                "media_url_sha256": "",
            }],
            "fetch_failures": [],
            "eligible_source_hours": 0.0,
            "attempted_article_urls": [url_a, url_b, url_c],
        })
        first = repair_orphan_phase_articles()
        if first["orphan_articles_removed"] != 1 or first["processed_ids_released"] != 1 or first["media_owners_released"] != 1:
            raise AssertionError(first)
        kept_ids = [row["source_id"] for row in ACCUMULATED_PHASES[0]["articles"]]
        if kept_ids != ["VOV4_REPAIR_A", "VOV4_REPAIR_B"]:
            raise AssertionError(kept_ids)
        if "VOV4_REPAIR_C" in RUN_STATE["processed_source_ids"]:
            raise AssertionError("orphan stayed processed")
        if media_c in OWNED_MEDIA:
            raise AssertionError("orphan media stayed owned")
        if "aa" * 32 not in OWNED_MEDIA:
            raise AssertionError("completed media owner was released")
        if url_c not in ARTICLE_CACHE:
            raise AssertionError("orphan left the article cache")
        if url_c in ACCUMULATED_PHASES[0]["attempted_article_urls"]:
            raise AssertionError("orphan URL stayed attempted")
        second = repair_orphan_phase_articles()
        if second["orphan_articles_removed"] != 0 or second["processed_ids_released"] != 0 or second["media_owners_released"] != 0:
            raise AssertionError(second)
        repaired = {
            "articles": list(ACCUMULATED_PHASES[0]["articles"]),
            "records": list(ACCUMULATED_PHASES[0]["records"]),
            "exclusions": list(ACCUMULATED_PHASES[0]["exclusions"]),
            "fetch_failures": [],
            "attempted_article_urls": list(ACCUMULATED_PHASES[0]["attempted_article_urls"]),
            "require_full_coverage": False,
            "section_stats": {},
            "total_budget_exhausted": False,
        }
        if run_consistency_gates(repaired):
            raise AssertionError(run_consistency_gates(repaired))

        capped = blank_section_state()
        capped["next_page"] = 400
        capped["pages_fetched"] = 400
        capped["oldest_date_seen"] = "2024-08-10"
        capped["pagination_cap_reached"] = True
        capped["date_boundary_reached"] = False
        capped["listing_exhausted"] = False
        done = blank_section_state()
        done["next_page"] = 12
        done["pages_fetched"] = 12
        done["oldest_date_seen"] = "2022-01-01"
        done["listing_exhausted"] = True
        done["date_boundary_reached"] = True
        done["pagination_cap_reached"] = False
        SECTION_CRAWL_STATE.clear()
        SECTION_CRAWL_STATE["kotong-ang-lom-topol-thoi-su-xa-hoi"] = dict(capped)
        SECTION_CRAWL_STATE["nor-pore-phat-thanh"] = dict(capped)
        SECTION_CRAWL_STATE["choh-jang-sa-nha-nong"] = dict(done)
        if apply_monotonic_page_cap(MAX_PAGES_PER_SECTION) != 0:
            raise AssertionError("equal page cap reopened sections")
        if SECTION_CRAWL_STATE["kotong-ang-lom-topol-thoi-su-xa-hoi"]["pagination_cap_reached"] is not True:
            raise AssertionError("equal cap cleared pagination_cap_reached")
        if SECTION_CRAWL_STATE["kotong-ang-lom-topol-thoi-su-xa-hoi"]["next_page"] != 400:
            raise AssertionError("equal cap moved next_page")
        reopened = apply_monotonic_page_cap(400)
        if reopened != 2:
            raise AssertionError(reopened)
        for slug in ("kotong-ang-lom-topol-thoi-su-xa-hoi", "nor-pore-phat-thanh"):
            state = SECTION_CRAWL_STATE[slug]
            if state["pagination_cap_reached"] or state["next_page"] != 400 or state["pages_fetched"] != 400:
                raise AssertionError(state)
        if SECTION_CRAWL_STATE["choh-jang-sa-nha-nong"]["next_page"] != 12 or not SECTION_CRAWL_STATE["choh-jang-sa-nha-nong"]["listing_exhausted"]:
            raise AssertionError(SECTION_CRAWL_STATE["choh-jang-sa-nha-nong"])
        globals()["MAX_PAGES_PER_SECTION"] = 200
        try:
            apply_monotonic_page_cap(400)
            raise AssertionError("page-cap decrease was accepted")
        except CheckpointIncompatible:
            pass
        if SECTION_CRAWL_STATE["kotong-ang-lom-topol-thoi-su-xa-hoi"]["next_page"] != 400:
            raise AssertionError("rejected page-cap decrease reset next_page")
    finally:
        globals()["MAX_PAGES_PER_SECTION"] = saved_pages_global
        ARTICLE_CACHE.clear()
        ARTICLE_CACHE.update(saved_cache_r)
        SECTION_CRAWL_STATE.clear()
        SECTION_CRAWL_STATE.update(saved_sections_r)
        OWNED_MEDIA.clear()
        OWNED_MEDIA.update(saved_owned_r)
        RUN_STATE["processed_source_ids"] = saved_processed_r
        RUN_STATE["total_budget_exhausted"] = saved_budget_r
        ACCUMULATED_PHASES.clear()
        ACCUMULATED_PHASES.extend(saved_phases_r)
        shutil.rmtree(repair_dir, ignore_errors=True)


    # A/B. summary.json is published only after contract.json verifies
    publication_dir = OUT_DIR / "qa" / "_synthetic_publication"
    shutil.rmtree(publication_dir, ignore_errors=True)
    for relative in FINAL_ARTIFACT_RELATIVE:
        artifact = publication_dir / relative
        artifact.parent.mkdir(parents=True, exist_ok=True)
        artifact.write_text(relative + "\n", encoding="utf-8")
    previous_summary = {"status": "PARTIAL_RQ2_VOV4_FULL_SOURCE_POOL", "problems": ["held"]}
    (publication_dir / "summary.json").write_text(json.dumps(previous_summary), encoding="utf-8")
    success_summary = {"status": "SUCCESS_RQ2_VOV4_FULL_SOURCE_POOL", "problems": []}
    success_contract = {"notebook": "10", "forbidden_code_pattern_hits": []}

    def _reject_contract(path, text):
        current = json.loads((publication_dir / "summary.json").read_text(encoding="utf-8"))
        if current.get("status") == "SUCCESS_RQ2_VOV4_FULL_SOURCE_POOL":
            raise AssertionError("SUCCESS summary was published before contract write")
        raise RuntimeError("simulated contract write failure")

    try:
        finalize_run_publication(publication_dir, success_summary, success_contract, contract_writer=_reject_contract)
        raise AssertionError("contract failure published a run")
    except RuntimeError as exc:
        if "simulated contract write failure" not in str(exc):
            raise
    held = json.loads((publication_dir / "summary.json").read_text(encoding="utf-8"))
    if held.get("status") == "SUCCESS_RQ2_VOV4_FULL_SOURCE_POOL":
        raise AssertionError("contract failure published SUCCESS")
    if (publication_dir / "contract.json").is_file():
        raise AssertionError("failed contract write left contract.json")

    def _contract_then_check(path, text):
        current = json.loads((publication_dir / "summary.json").read_text(encoding="utf-8"))
        if current.get("status") == "SUCCESS_RQ2_VOV4_FULL_SOURCE_POOL":
            raise AssertionError("SUCCESS summary was published before contract write")
        atomic_write_text(path, text)

    finalize_run_publication(publication_dir, success_summary, success_contract, contract_writer=_contract_then_check)
    published = json.loads((publication_dir / "summary.json").read_text(encoding="utf-8"))
    if published.get("status") != "SUCCESS_RQ2_VOV4_FULL_SOURCE_POOL":
        raise AssertionError(published)
    loaded_contract = json.loads((publication_dir / "contract.json").read_text(encoding="utf-8"))
    if "final_artifact_sha256" not in loaded_contract:
        raise AssertionError("contract omitted artifact hashes")
    verify_final_artifact_hashes(publication_dir, loaded_contract["final_artifact_sha256"])
    if (publication_dir / "summary.json").stat().st_mtime < (publication_dir / "contract.json").stat().st_mtime:
        raise AssertionError("summary.json was published before contract.json")
    shutil.rmtree(publication_dir, ignore_errors=True)

    # C/E. older_boundary_streak and listing diagnostics survive resume; old checkpoints default cleanly
    diagnostic_dir = OUT_DIR / "qa" / "_synthetic_listing_diagnostics"
    diagnostic_path = diagnostic_dir / "state.json"
    shutil.rmtree(diagnostic_dir, ignore_errors=True)
    saved_cache_d = dict(ARTICLE_CACHE)
    saved_sections_d = {slug: dict(state) for slug, state in SECTION_CRAWL_STATE.items()}
    saved_owned_d = dict(OWNED_MEDIA)
    saved_processed_d = set(RUN_STATE["processed_source_ids"])
    saved_budget_d = RUN_STATE["total_budget_exhausted"]
    saved_ceiling_d = RUN_STATE.get("budget_exhausted_ceiling_bytes")
    saved_phases_d = list(ACCUMULATED_PHASES)
    saved_network_d = dict(NETWORK_STATE)
    try:
        ARTICLE_CACHE.clear()
        SECTION_CRAWL_STATE.clear()
        OWNED_MEDIA.clear()
        ACCUMULATED_PHASES.clear()
        RUN_STATE["processed_source_ids"] = set()
        RUN_STATE["total_budget_exhausted"] = False
        streak_slug = "kotong-ang-lom-topol-thoi-su-xa-hoi"
        streak_state = blank_section_state()
        streak_state["next_page"] = 12
        streak_state["pages_fetched"] = 12
        update_older_boundary_streak(streak_state, True)
        if streak_state["older_boundary_streak"] != 1 or streak_state["date_boundary_reached"]:
            raise AssertionError(streak_state)
        record_listing_page_diagnostic(
            streak_state, 11, 2,
            [("2025-01-01", "time"), ("2025-02-01", "metadata")],
            0, "2026-01-01", "2026-09-24", "SCOPED",
        )
        if len(streak_state["recent_listing_pages"]) != 1:
            raise AssertionError(streak_state["recent_listing_pages"])
        SECTION_CRAWL_STATE[streak_slug] = streak_state
        save_checkpoint(diagnostic_path)
        SECTION_CRAWL_STATE.clear()
        if not load_checkpoint(diagnostic_path):
            raise AssertionError("streak checkpoint did not load")
        restored_streak = SECTION_CRAWL_STATE[streak_slug]
        if restored_streak["older_boundary_streak"] != 1 or restored_streak["next_page"] != 12:
            raise AssertionError(restored_streak)
        if restored_streak["last_boundary_reason"] != "ALL_TRUSTED_DATES_OLDER_THAN_WINDOW":
            raise AssertionError(restored_streak["last_boundary_reason"])
        if restored_streak["pages_fetched"] != 12:
            raise AssertionError("resume reset pages_fetched")
        update_older_boundary_streak(restored_streak, True)
        if restored_streak["older_boundary_streak"] != 2 or not restored_streak["date_boundary_reached"]:
            raise AssertionError(restored_streak)
        reset_state = blank_section_state()
        update_older_boundary_streak(reset_state, True)
        update_older_boundary_streak(reset_state, False)
        if reset_state["older_boundary_streak"] != 0 or reset_state["date_boundary_reached"]:
            raise AssertionError(reset_state)
        newer_reason = classify_listing_boundary_reason(
            "SCOPED", 1, [("2026-10-01", "time")], 0, "2026-01-01", "2026-09-24"
        )
        in_window_reason = classify_listing_boundary_reason(
            "SCOPED", 1, [("2026-03-01", "metadata")], 0, "2026-01-01", "2026-09-24"
        )
        if newer_reason != "HAS_NEWER_DATE" or in_window_reason != "HAS_IN_WINDOW_DATE":
            raise AssertionError((newer_reason, in_window_reason))
        capped_state = blank_section_state()
        for page_number in range(12):
            record_listing_page_diagnostic(
                capped_state, page_number, 0, [], 0, "2026-01-01", "2026-09-24", "LISTING_SCOPE_UNVERIFIED"
            )
        if len(capped_state["recent_listing_pages"]) != LISTING_DIAGNOSTIC_HISTORY:
            raise AssertionError(len(capped_state["recent_listing_pages"]))
        if capped_state["recent_listing_pages"][0]["page"] != 12 - LISTING_DIAGNOSTIC_HISTORY:
            raise AssertionError(capped_state["recent_listing_pages"][0])

        stripped = json.loads(diagnostic_path.read_text(encoding="utf-8"))
        for section_state in stripped["section_crawl"].values():
            for key in (
                "older_boundary_streak", "last_boundary_reason", "last_page_number",
                "last_page_scoped_article_count", "last_page_trusted_date_count",
                "last_page_untrusted_date_count", "last_page_min_trusted_date",
                "last_page_max_trusted_date", "last_page_all_trusted_dates_older_than_window",
                "recent_listing_pages", "listing_scope_unverified", "last_page_listing_scope_strategy",
                "listing_scope_unverified_page", "listing_scope_unverified_attempts",
            ):
                section_state.pop(key, None)
        diagnostic_path.write_text(json.dumps(stripped), encoding="utf-8")
        if not load_checkpoint(diagnostic_path):
            raise AssertionError("legacy checkpoint did not load")
        legacy = SECTION_CRAWL_STATE[streak_slug]
        if legacy["older_boundary_streak"] != 0 or legacy["recent_listing_pages"] != []:
            raise AssertionError(legacy)
        if legacy["next_page"] != 12 or legacy["pages_fetched"] != 12:
            raise AssertionError("legacy checkpoint reset crawl progress")
        if legacy["last_boundary_reason"] != "" or legacy["listing_scope_unverified"]:
            raise AssertionError(legacy)
        if legacy["last_page_listing_scope_strategy"] != "":
            raise AssertionError(legacy["last_page_listing_scope_strategy"])
        if not section_coverage_incomplete(legacy):
            raise AssertionError("a legacy in-progress section was marked covered without a boundary")
    finally:
        ARTICLE_CACHE.clear()
        ARTICLE_CACHE.update(saved_cache_d)
        SECTION_CRAWL_STATE.clear()
        SECTION_CRAWL_STATE.update(saved_sections_d)
        OWNED_MEDIA.clear()
        OWNED_MEDIA.update(saved_owned_d)
        NETWORK_STATE.clear()
        NETWORK_STATE.update(saved_network_d)
        RUN_STATE["processed_source_ids"] = saved_processed_d
        RUN_STATE["total_budget_exhausted"] = saved_budget_d
        if saved_ceiling_d is None:
            RUN_STATE.pop("budget_exhausted_ceiling_bytes", None)
        else:
            RUN_STATE["budget_exhausted_ceiling_bytes"] = saved_ceiling_d
        ACCUMULATED_PHASES.clear()
        ACCUMULATED_PHASES.extend(saved_phases_d)
        shutil.rmtree(diagnostic_dir, ignore_errors=True)

    # D. same-section sidebar link is not main-listing evidence and cannot block the date boundary
    sidebar_slug = "kotong-ang-lom-topol-thoi-su-xa-hoi"
    sidebar_html = (
        "<html><body>"
        '<div class="view-content">'
        + '<a href="/bahnar/' + sidebar_slug + '/old-a.vov4">old a</a>'
        + '<a href="/bahnar/' + sidebar_slug + '/old-b.vov4">old b</a>'
        + "</div>"
        + '<aside class="sidebar related">'
        + '<a href="/bahnar/' + sidebar_slug + '/newer.vov4">newer same section</a>'
        + "</aside>"
        + "</body></html>"
    ).encode("utf-8")
    sidebar_links = extract_section_links_from_html(
        sidebar_html, "https://vov4.vov.vn/bahnar/" + sidebar_slug, sidebar_slug
    )
    expected_main = [
        "https://vov4.vov.vn/bahnar/" + sidebar_slug + "/old-a.vov4",
        "https://vov4.vov.vn/bahnar/" + sidebar_slug + "/old-b.vov4",
    ]
    if sidebar_links != expected_main:
        raise AssertionError(sidebar_links)
    sidebar_inspected = inspect_listing_page(sidebar_html, 40, sidebar_slug)
    if sidebar_inspected["status"] != "LISTING_EXHAUSTED" or sidebar_inspected["links"] != expected_main:
        raise AssertionError(sidebar_inspected)
    old_observations = [("2025-03-01", "metadata"), ("2025-04-01", "time")]
    if not page_proves_older_boundary(len(sidebar_links), old_observations, 0, "2026-01-01"):
        raise AssertionError("scoped old listing did not prove the boundary")
    sidebar_reason = classify_listing_boundary_reason(
        "SCOPED", len(sidebar_links), old_observations, 0, "2026-01-01", "2026-09-24"
    )
    if sidebar_reason != "ALL_TRUSTED_DATES_OLDER_THAN_WINDOW":
        raise AssertionError(sidebar_reason)
    contaminated = old_observations + [("2026-09-01", "time")]
    if page_proves_older_boundary(3, contaminated, 0, "2026-01-01"):
        raise AssertionError("a newer same-section date was allowed to prove the boundary")
    loose_html = (
        '<html><body><a href="/bahnar/' + sidebar_slug + '/loose.vov4">loose</a></body></html>'
    ).encode("utf-8")
    if extract_section_links_from_html(loose_html, "https://vov4.vov.vn/bahnar/" + sidebar_slug, sidebar_slug):
        raise AssertionError("an unscoped same-section anchor was treated as listing evidence")
    boundary_state = blank_section_state()
    update_older_boundary_streak(boundary_state, True)
    update_older_boundary_streak(boundary_state, True)
    if not boundary_state["date_boundary_reached"]:
        raise AssertionError("two scoped older pages did not reach the date boundary")


    # publication generation rollback
    generation_dir = OUT_DIR / "qa" / "_synthetic_publication_generation"
    shutil.rmtree(generation_dir, ignore_errors=True)
    for relative in FINAL_ARTIFACT_RELATIVE:
        artifact = generation_dir / relative
        artifact.parent.mkdir(parents=True, exist_ok=True)
        artifact.write_text("generation-manifest\n", encoding="utf-8")
    previous_contract = json.dumps({"generation": "A", "notebook": "previous"})
    (generation_dir / "contract.json").write_text(previous_contract, encoding="utf-8")
    previous_success = {"status": "SUCCESS_RQ2_VOV4_FULL_SOURCE_POOL", "problems": [], "generation": "A"}
    (generation_dir / "summary.json").write_text(json.dumps(previous_success), encoding="utf-8")

    def _write_contract(path, text):
        atomic_write_text(path, text)

    def _fail_after_contract():
        raise RuntimeError("simulated post-commit failure")

    try:
        finalize_run_publication(
            generation_dir,
            {"status": "SUCCESS_RQ2_VOV4_FULL_SOURCE_POOL", "problems": [], "generation": "B"},
            {"generation": "B", "forbidden_code_pattern_hits": []},
            contract_writer=_write_contract,
            after_contract_commit=_fail_after_contract,
        )
        raise AssertionError("post-commit failure published a new generation")
    except RuntimeError as exc:
        if "simulated post-commit failure" not in str(exc):
            raise
    if (generation_dir / "contract.json").read_text(encoding="utf-8") != previous_contract:
        raise AssertionError("failed publication left contract generation B")
    held_summary = json.loads((generation_dir / "summary.json").read_text(encoding="utf-8"))
    if held_summary.get("generation") != "A" or held_summary.get("status") != "SUCCESS_RQ2_VOV4_FULL_SOURCE_POOL":
        raise AssertionError(held_summary)

    orphan_dir = OUT_DIR / "qa" / "_synthetic_publication_orphan"
    shutil.rmtree(orphan_dir, ignore_errors=True)
    for relative in FINAL_ARTIFACT_RELATIVE:
        artifact = orphan_dir / relative
        artifact.parent.mkdir(parents=True, exist_ok=True)
        artifact.write_text("orphan-manifest\n", encoding="utf-8")
    try:
        finalize_run_publication(
            orphan_dir,
            {"status": "SUCCESS_RQ2_VOV4_FULL_SOURCE_POOL", "problems": [], "generation": "B"},
            {"generation": "B", "forbidden_code_pattern_hits": []},
            contract_writer=_write_contract,
            after_contract_commit=_fail_after_contract,
        )
        raise AssertionError("orphan publication succeeded")
    except RuntimeError as exc:
        if "simulated post-commit failure" not in str(exc):
            raise
    if (orphan_dir / "contract.json").exists():
        raise AssertionError("failed first publication left an orphan contract")
    if (orphan_dir / "summary.json").exists():
        raise AssertionError("failed first publication left a summary")

    clean_dir = OUT_DIR / "qa" / "_synthetic_publication_clean"
    shutil.rmtree(clean_dir, ignore_errors=True)
    for relative in FINAL_ARTIFACT_RELATIVE:
        artifact = clean_dir / relative
        artifact.parent.mkdir(parents=True, exist_ok=True)
        artifact.write_text("clean-manifest\n", encoding="utf-8")

    def _summary_still_absent():
        if (clean_dir / "summary.json").is_file():
            raise AssertionError("summary was published before the final commit")

    finalize_run_publication(
        clean_dir,
        {"status": "SUCCESS_RQ2_VOV4_FULL_SOURCE_POOL", "problems": [], "generation": "B"},
        {"generation": "B", "forbidden_code_pattern_hits": []},
        contract_writer=_write_contract,
        after_contract_commit=_summary_still_absent,
    )
    clean_summary = json.loads((clean_dir / "summary.json").read_text(encoding="utf-8"))
    clean_contract = json.loads((clean_dir / "contract.json").read_text(encoding="utf-8"))
    if clean_summary.get("generation") != "B" or clean_contract.get("generation") != "B":
        raise AssertionError((clean_summary, clean_contract))
    if (clean_dir / "summary.json").stat().st_mtime < (clean_dir / "contract.json").stat().st_mtime:
        raise AssertionError("summary.json was published before contract.json")
    shutil.rmtree(generation_dir, ignore_errors=True)
    shutil.rmtree(orphan_dir, ignore_errors=True)
    shutil.rmtree(clean_dir, ignore_errors=True)

    # scoped pager evidence
    pager_slug = "kotong-ang-lom-topol-thoi-su-xa-hoi"
    pager_article = "/bahnar/" + pager_slug + "/pager-item.vov4"
    trusted_pager_html = (
        "<html><body><div class=\"view-content\">"
        "<a href=\"" + pager_article + "\">item</a>"
        "<ul class=\"pager\"><li class=\"pager__item\"><a href=\"?page=11\">11</a></li></ul>"
        "</div></body></html>"
    ).encode("utf-8")
    if not listing_has_next_page(trusted_pager_html, 10):
        raise AssertionError("a pager inside the main listing was ignored")
    sidebar_pager_html = (
        "<html><body><div class=\"view-content\">"
        "<a href=\"" + pager_article + "\">item</a></div>"
        "<aside class=\"sidebar\"><a href=\"?page=11\">sidebar page</a></aside>"
        "</body></html>"
    ).encode("utf-8")
    if listing_has_next_page(sidebar_pager_html, 10):
        raise AssertionError("a sidebar ?page= link was treated as the listing pager")
    footer_next_html = (
        b"<html><body><footer><a rel=\"next\" href=\"?page=2\">next</a></footer></body></html>"
    )
    if listing_has_next_page(footer_next_html, 1):
        raise AssertionError("a footer rel=next link was treated as the listing pager")
    drupal_pager_html = (
        "<html><body><nav class=\"pager\">"
        "<ul class=\"pager__items\"><li class=\"pager__item pager__item--next\">"
        "<a href=\"?page=11\" rel=\"next\">next</a></li></ul></nav></body></html>"
    ).encode("utf-8")
    if not listing_has_next_page(drupal_pager_html, 10):
        raise AssertionError("a Drupal pager-next link was ignored")
    scoped_page = inspect_listing_page(trusted_pager_html, 10, pager_slug)
    if scoped_page.get("listing_scope_strategy") != "view-content":
        raise AssertionError(scoped_page.get("listing_scope_strategy"))
    main_only_html = (
        "<html><body><main><a href=\"" + pager_article + "\">item</a></main></body></html>"
    ).encode("utf-8")
    main_scoped = extract_scoped_listing_links(main_only_html, "https://vov4.vov.vn/bahnar/" + pager_slug, pager_slug)
    if main_scoped.get("listing_scope_strategy") != "main-fallback" or not main_scoped["links"]:
        raise AssertionError(main_scoped)
    strategy_state = blank_section_state()
    record_listing_page_diagnostic(
        strategy_state, 10, 1, [("2025-01-01", "time")], 0, "2026-01-01", "2026-09-24",
        "SCOPED", "view-content",
    )
    if strategy_state["last_page_listing_scope_strategy"] != "view-content":
        raise AssertionError(strategy_state["last_page_listing_scope_strategy"])
    if strategy_state["recent_listing_pages"][-1]["listing_scope_strategy"] != "view-content":
        raise AssertionError(strategy_state["recent_listing_pages"])
    real_slug = "kotong-ang-lom-topol-thoi-su-xa-hoi"
    other_slug = "choh-jang-sa-nha-nong"
    real_listing_html = (
        "<html><body>"
        "<div class=\"col taxonomy-block1\">"
        "<div class=\"row mt-3\">"
        "<div class=\"col-xl-8 col-lg-7\"><div class=\"taxonomy-large-card\">"
        "<div class=\"article-card\"><a href=\"https://vov4.vov.vn/bahnar/" + real_slug + "/card-one.vov4\">one</a></div>"
        "</div></div>"
        "<div class=\"col-xl-4 col-lg-5 side-news\">"
        "<a href=\"https://vov4.vov.vn/bahnar/" + real_slug + "/sidebar-item.vov4\">side</a>"
        "</div></div></div>"
        "<div class=\"col-lg-9\"><div class=\"mt-2\"><div class=\"views-element-container\">"
        "<div class=\"article-card\"><a href=\"https://vov4.vov.vn/bahnar/" + real_slug + "/card-two.vov4\">two</a></div>"
        "<nav class=\"mt-3\"><ul class=\"pagination justify-content-center\">"
        "<li class=\"page-item\"><a href=\"?page=1\">Trang 2</a></li>"
        "</ul></nav></div></div></div>"
        "<a href=\"https://vov4.vov.vn/bahnar/" + other_slug + "/cross.vov4\">cross</a>"
        "<a href=\"https://vov4.vov.vn/bahnar/" + real_slug + "/loose.vov4\">loose</a>"
        "</body></html>"
    ).encode("utf-8")
    real_scoped = extract_scoped_listing_links(
        real_listing_html, "https://vov4.vov.vn/bahnar/" + real_slug, real_slug
    )
    if real_scoped.get("scope_status") != "SCOPED" or real_scoped.get("listing_scope_strategy") != "taxonomy-views":
        raise AssertionError(real_scoped)
    real_links = real_scoped["links"]
    real_boundary = real_scoped["boundary_links"]
    if len(real_links) != 2 or any(token in " ".join(real_links) for token in ("sidebar-item", "cross.vov4", "loose.vov4")):
        raise AssertionError(real_links)
    if "card-one.vov4" not in real_links[0] or "card-two.vov4" not in real_links[1]:
        raise AssertionError(real_links)
    if real_boundary != [link for link in real_links if "card-two.vov4" in link]:
        raise AssertionError(real_boundary)
    if any(token in " ".join(real_boundary) for token in ("card-one.vov4", "sidebar-item", "cross.vov4", "loose.vov4")):
        raise AssertionError(real_boundary)
    print("TAXONOMY_BOUNDARY_SEPARATION_PASS")
    if not listing_has_next_page(real_listing_html, 0):
        raise AssertionError("production pagination nav was not a next page")

    def _taxonomy_page(featured_slug, view_slugs):
        view_html = "".join(
            "<div class=\"article-card\"><a href=\"https://vov4.vov.vn/bahnar/" + real_slug + "/" + item + ".vov4\">v</a></div>"
            for item in view_slugs
        )
        return (
            "<html><body>"
            "<div class=\"taxonomy-block1\"><div class=\"article-card\">"
            "<a href=\"https://vov4.vov.vn/bahnar/" + real_slug + "/" + featured_slug + ".vov4\">featured</a>"
            "</div></div>"
            "<div class=\"views-element-container\">" + view_html + "</div>"
            "</body></html>"
        ).encode("utf-8")

    sticky_state = blank_section_state()
    for page_number, featured, views, dates in (
        (798, "featured-2026", ("view-798-a", "view-798-b"), ("2022-06-22", "2022-06-23")),
        (799, "featured-2026", ("view-799-a", "view-799-b"), ("2022-06-21", "2022-06-21")),
    ):
        page_html = _taxonomy_page(featured, views)
        inspected = inspect_listing_page(page_html, page_number, real_slug)
        if inspected.get("listing_scope_strategy") != "taxonomy-views":
            raise AssertionError(inspected)
        if len(inspected["links"]) != 3 or len(inspected["boundary_links"]) != 2:
            raise AssertionError((inspected["links"], inspected["boundary_links"]))
        if any("featured-2026" in link for link in inspected["boundary_links"]):
            raise AssertionError(inspected["boundary_links"])
        observations = [(item, "metadata") for item in dates]
        proved = page_proves_older_boundary(len(inspected["boundary_links"]), observations, 0, "2026-01-01")
        if not proved:
            raise AssertionError((page_number, proved))
        contaminated = observations + [("2026-09-26", "metadata")]
        if page_proves_older_boundary(len(contaminated), contaminated, 0, "2026-01-01"):
            raise AssertionError("featured 2026 date was accepted as boundary evidence")
        update_older_boundary_streak(sticky_state, proved)
    if sticky_state["older_boundary_streak"] != 2 or not sticky_state["date_boundary_reached"]:
        raise AssertionError(sticky_state)
    print("TAXONOMY_STICKY_DATE_REGRESSION_PASS")

    saved_sections_boundary = {slug: dict(state) for slug, state in SECTION_CRAWL_STATE.items()}
    try:
        SECTION_CRAWL_STATE.clear()
        capped_slug = "kotong-ang-lom-topol-thoi-su-xa-hoi"
        capped_state = blank_section_state()
        capped_state.update({
            "next_page": 800,
            "pages_fetched": 801,
            "pagination_cap_reached": True,
            "date_boundary_reached": False,
            "listing_exhausted": False,
            "last_page_listing_scope_strategy": "taxonomy-views",
            "older_boundary_streak": 0,
        })
        SECTION_CRAWL_STATE[capped_slug] = capped_state
        if repair_taxonomy_boundary_capped_sections() != 1:
            raise AssertionError("capped taxonomy section was not reopened")
        if capped_state["next_page"] != 798 or capped_state["pagination_cap_reached"] or capped_state["older_boundary_streak"] != 0:
            raise AssertionError(capped_state)
        if capped_state["pages_fetched"] != 801:
            raise AssertionError(capped_state["pages_fetched"])
        if repair_taxonomy_boundary_capped_sections() != 0 or capped_state["next_page"] != 798:
            raise AssertionError(capped_state["next_page"])
        for proved in (True, True):
            update_older_boundary_streak(capped_state, proved)
        if not capped_state["date_boundary_reached"] or capped_state["pagination_cap_reached"]:
            raise AssertionError(capped_state)
    finally:
        SECTION_CRAWL_STATE.clear()
        SECTION_CRAWL_STATE.update(saved_sections_boundary)
    print("CAPPED_BOUNDARY_REPAIR_PASS")
    real_diag = blank_section_state()
    record_listing_page_diagnostic(
        real_diag, 0, len(real_links), [], 0, "2026-01-01", "2026-09-24", "SCOPED", "taxonomy-views",
    )
    if real_diag["last_page_listing_scope_strategy"] != "taxonomy-views":
        raise AssertionError(real_diag["last_page_listing_scope_strategy"])
    if real_diag["recent_listing_pages"][-1]["listing_scope_strategy"] != "taxonomy-views":
        raise AssertionError(real_diag["recent_listing_pages"])
    retry_state = blank_section_state()
    retry_state["listing_scope_unverified"] = True
    retry_state["listing_scope_unverified_page"] = 0
    retry_state["listing_scope_unverified_attempts"] = 1
    retry_state["next_page"] = 1
    if prepare_listing_scope_unverified_retry(retry_state) != "retry" or retry_state["next_page"] != 0:
        raise AssertionError(retry_state)
    page_number = retry_state["next_page"]
    retry_state["pages_fetched"] += 1
    retry_state["next_page"] += 1
    retried = inspect_listing_page(real_listing_html, page_number, real_slug)
    if retried.get("scope_status") != "SCOPED" or retried.get("status") != "LISTING_CONTINUE":
        raise AssertionError(retried)
    clear_listing_scope_unverified(retry_state)
    if retry_state["listing_scope_unverified"] or retry_state["listing_scope_unverified_attempts"] != 0 or retry_state["listing_scope_unverified_page"] is not None:
        raise AssertionError(retry_state)
    if retry_state["next_page"] != 1:
        raise AssertionError(retry_state["next_page"])
    missing_strategy = blank_section_state()
    missing_strategy.update({"next_page": 3, "pages_fetched": 3})
    missing_strategy.pop("last_page_listing_scope_strategy", None)
    filled = blank_section_state()
    filled.update(missing_strategy)
    if filled["last_page_listing_scope_strategy"] != "" or filled["next_page"] != 3:
        raise AssertionError(filled)

    # 1.1 resume preserves the cursor; 1.0 -> 1.1 resets listing coverage and keeps media
    migration_dir = OUT_DIR / "qa" / "_synthetic_schema_migration"
    migration_path = migration_dir / "state.json"
    media_file = migration_dir / "keep.wav"
    shutil.rmtree(migration_dir, ignore_errors=True)
    migration_dir.mkdir(parents=True, exist_ok=True)
    media_file.write_bytes(b"verified-media-bytes")
    saved_cache_m = dict(ARTICLE_CACHE)
    saved_sections_m = {slug: dict(state) for slug, state in SECTION_CRAWL_STATE.items()}
    saved_owned_m = dict(OWNED_MEDIA)
    saved_processed_m = set(RUN_STATE["processed_source_ids"])
    saved_reusable_m = dict(RUN_STATE.get("reusable_media_by_hash") or {})
    saved_budget_m = RUN_STATE["total_budget_exhausted"]
    saved_ceiling_m = RUN_STATE.get("budget_exhausted_ceiling_bytes")
    saved_phases_m = list(ACCUMULATED_PHASES)
    saved_network_m = dict(NETWORK_STATE)
    saved_save_checkpoint = save_checkpoint
    saved_pipeline = run_article_through_pipeline
    saved_verify = dict(VERIFY_CACHE)
    try:
        def _rewrite_compat(path, updates):
            payload = json.loads(Path(path).read_text(encoding="utf-8"))
            payload["checkpoint_compatibility"].update(updates)
            payload["checkpoint_compatibility_sha256"] = hashlib.sha256(
                canonical_json_bytes(payload["checkpoint_compatibility"])
            ).hexdigest()
            Path(path).write_text(json.dumps(payload), encoding="utf-8")

        ARTICLE_CACHE.clear()
        SECTION_CRAWL_STATE.clear()
        OWNED_MEDIA.clear()
        ACCUMULATED_PHASES.clear()
        RUN_STATE["processed_source_ids"] = set()
        RUN_STATE["reusable_media_by_hash"] = {"ab" * 32: {"technical_status": "PASS"}}
        RUN_STATE["total_budget_exhausted"] = False
        resume_slug = "choh-jang-sa-nha-nong"
        resume_state = blank_section_state()
        resume_state["next_page"] = 22
        resume_state["pages_fetched"] = 22
        resume_state["older_boundary_streak"] = 1
        resume_state["last_page_listing_scope_strategy"] = "view-content"
        SECTION_CRAWL_STATE[resume_slug] = resume_state
        save_checkpoint(migration_path)
        SECTION_CRAWL_STATE.clear()
        if not load_checkpoint(migration_path):
            raise AssertionError("1.1 checkpoint did not load")
        resumed = SECTION_CRAWL_STATE[resume_slug]
        if resumed["next_page"] != 22 or resumed["pages_fetched"] != 22 or resumed["older_boundary_streak"] != 1:
            raise AssertionError(resumed)
        if resumed["last_page_listing_scope_strategy"] != "view-content":
            raise AssertionError(resumed["last_page_listing_scope_strategy"])
        if RUN_STATE["reusable_media_by_hash"].get("ab" * 32, {}).get("technical_status") != "PASS":
            raise AssertionError("1.1 resume dropped reusable_media_by_hash")

        capped = blank_section_state()
        capped["next_page"] = 400
        capped["pages_fetched"] = 400
        capped["pagination_cap_reached"] = True
        capped["date_boundary_reached"] = False
        capped["listing_exhausted"] = False
        SECTION_CRAWL_STATE.clear()
        SECTION_CRAWL_STATE[resume_slug] = capped
        save_checkpoint(migration_path)
        _rewrite_compat(migration_path, {"max_pages_per_section": 400})
        if not load_checkpoint(migration_path):
            raise AssertionError("raised page-cap checkpoint did not load")
        reopened = SECTION_CRAWL_STATE[resume_slug]
        if reopened["next_page"] != 400 or reopened["pages_fetched"] != 400 or reopened["pagination_cap_reached"]:
            raise AssertionError(reopened)
        _rewrite_compat(migration_path, {"max_pages_per_section": 900})
        try:
            load_checkpoint(migration_path)
            raise AssertionError("a decreased page cap was accepted")
        except CheckpointIncompatible:
            pass
        if SECTION_CRAWL_STATE[resume_slug]["next_page"] != 400:
            raise AssertionError("rejected page-cap decrease reset the cursor")

        migrate_slug = "kotong-ang-lom-topol-thoi-su-xa-hoi"
        migrate_url = "https://vov4.vov.vn/bahnar/" + migrate_slug + "/migration-keep.vov4"
        migrate_id = source_id(migrate_url)
        fail_url = "https://vov4.vov.vn/bahnar/" + migrate_slug + "/migration-fail.vov4"
        fail_id = source_id(fail_url)
        media_url = "https://cdn.example.com/migration-keep.mp3"
        media_hash = hashlib.sha256(media_url.encode("utf-8")).hexdigest()
        fail_media_url = "https://cdn.example.com/migration-fail.mp3"
        fail_hash = hashlib.sha256(fail_media_url.encode("utf-8")).hexdigest()
        changed_media_url = "https://cdn.example.com/migration-changed.mp3"
        changed_hash = hashlib.sha256(changed_media_url.encode("utf-8")).hexdigest()
        pcm = b"\x01\x00" * 1600
        raw_path = migration_dir / "keep.raw"
        wav_path = migration_dir / "keep.wav"
        raw_path.write_bytes(b"raw-artifact")
        with wave.open(str(wav_path), "wb") as handle:
            handle.setnchannels(1)
            handle.setsampwidth(2)
            handle.setframerate(16000)
            handle.writeframes(pcm)
        wav_bytes = wav_path.read_bytes()

        def _public_audio(url):
            VERIFY_CACHE[url] = {
                "probe_status": "MEDIA_FOUND_PUBLIC",
                "is_public_direct_url": True,
                "http_status": 200,
                "content_type": "audio/mpeg",
                "notes": "",
            }
            return {"media_url": url, "media_type": "audio", "media_extension": "mp3", "source": "audio_tag"}

        def _cache_article(url, date, date_source, title, media):
            row = {
                "source_id": source_id(url),
                "article_url": url,
                "article_date": date,
                "published_date_source": date_source,
                "article_title": title,
                "source_section": migrate_slug,
            }
            ARTICLE_CACHE[url] = {
                "kind": "article",
                "article_row": row,
                "media_candidates": [_public_audio(media)] if media else [],
                "article_scope_error": "",
                "discovered_sections": {migrate_slug},
            }
            return row

        def _reset_current_run():
            ACCUMULATED_PHASES.clear()
            RUN_STATE["processed_source_ids"] = set()
            OWNED_MEDIA.clear()

        def _only_discover(*urls):
            for result in ARTICLE_CACHE.values():
                if isinstance(result, dict) and "discovered_sections" in result:
                    result["discovered_sections"] = set()
            for url in urls:
                _note_listing_section(ARTICLE_CACHE[url], migrate_slug)

        migrate_row = _cache_article(migrate_url, "2026-02-01", "metadata", "tin cu", media_url)
        fail_row = _cache_article(fail_url, "2026-02-01", "metadata", "tin loi", fail_media_url)
        pass_record = {
            "source_id": migrate_id,
            "article_url": migrate_url,
            "article_date": "2020-01-01",
            "published_date_source": "body_fallback",
            "article_title": "old title",
            "phase": PHASE_A_LABEL,
            "source_section": migrate_slug,
            "content_class": "news",
            "media_url": media_url,
            "media_url_sha256": media_hash,
            "media_type": "audio",
            "media_extension": "mp3",
            "artifact_origin": "new_download",
            "raw_local_path": to_project_relative(raw_path),
            "wav_local_path": to_project_relative(wav_path),
            "video_container_local_path": "",
            "raw_sha256": hashlib.sha256(raw_path.read_bytes()).hexdigest(),
            "wav_sha256": hashlib.sha256(wav_bytes).hexdigest(),
            "pcm16_sha256": hashlib.sha256(pcm).hexdigest(),
            "duration_seconds": 0.1,
            "sample_rate": 16000,
            "channels": 1,
            "codec_name": "pcm_s16le",
            "technical_status": "PASS",
            "technical_notes": "verified",
            "source_level_status": "REJECTED_SOURCE",
            "exclusion_reason": "MUSIC_CONTENT",
            "verification_notes": "content_class=music; technical_status=PASS; rule_reason=MUSIC_CONTENT",
            "native_spotcheck_status": "CONFIRMED_BAHNAR",
            "native_spotcheck_notes": "spot-check belonged to old source A",
            "video_container_sha256": "",
            "video_container_bytes": "",
        }
        pass_problems = verify_pass_artifact(pass_record)
        if pass_problems:
            raise AssertionError(pass_problems)
        fail_record = {
            "source_id": fail_id,
            "article_url": fail_url,
            "article_date": "2026-02-01",
            "published_date_source": "metadata",
            "media_url": fail_media_url,
            "media_url_sha256": fail_hash,
            "technical_status": "TECHNICAL_FAIL",
            "source_level_status": "REJECTED_SOURCE",
            "duration_seconds": 1.0,
        }
        OWNED_MEDIA.clear()
        OWNED_MEDIA[media_hash] = migrate_id
        OWNED_MEDIA[fail_hash] = fail_id
        RUN_STATE["processed_source_ids"] = {migrate_id, fail_id}
        RUN_STATE["reusable_media_by_hash"] = {}
        ACCUMULATED_PHASES.clear()
        ACCUMULATED_PHASES.append({
            "phase_label": PHASE_A_LABEL,
            "start": PHASE_A_START,
            "end": PHASE_A_END,
            "articles": [dict(migrate_row), dict(fail_row)],
            "records": [dict(pass_record), dict(fail_record)],
            "exclusions": [],
            "fetch_failures": [],
            "eligible_source_hours": 0.0,
            "attempted_article_urls": [migrate_url, fail_url],
        })
        old_cursor = blank_section_state()
        old_cursor["next_page"] = 400
        old_cursor["pages_fetched"] = 400
        old_cursor["pagination_cap_reached"] = True
        old_cursor["date_boundary_reached"] = True
        old_cursor["older_boundary_streak"] = 2
        SECTION_CRAWL_STATE.clear()
        SECTION_CRAWL_STATE[migrate_slug] = old_cursor
        save_checkpoint(migration_path)
        _rewrite_compat(migration_path, {"crawl_pipeline_schema_version": "1.0"})
        if not load_checkpoint(migration_path):
            raise AssertionError("1.0 checkpoint did not migrate")
        migrated = SECTION_CRAWL_STATE[migrate_slug]
        if migrated["next_page"] != 0 or migrated["pages_fetched"] != 0 or migrated["date_boundary_reached"]:
            raise AssertionError(migrated)
        if migrated["pagination_cap_reached"] or migrated["older_boundary_streak"] != 0:
            raise AssertionError(migrated)
        if wav_path.read_bytes() != wav_bytes or raw_path.read_bytes() != b"raw-artifact":
            raise AssertionError("migration rewrote verified media")
        if OWNED_MEDIA:
            raise AssertionError("old ownership survived migration")
        if media_hash not in (RUN_STATE.get("reusable_media_by_hash") or {}):
            raise AssertionError("verified PASS artifact was not cached by hash")
        if fail_hash in (RUN_STATE.get("reusable_media_by_hash") or {}):
            raise AssertionError("TECHNICAL_FAIL artifact entered the reuse cache")
        if verify_pass_artifact(RUN_STATE["reusable_media_by_hash"][media_hash]):
            raise AssertionError("cached PASS artifact no longer verifies")
        if migrate_url not in ARTICLE_CACHE or fail_url not in ARTICLE_CACHE:
            raise AssertionError("migration dropped the article fetch cache")
        if list(iter_discovered_articles()):
            raise AssertionError("1.0 discovery membership survived migration")
        if any(phase.get("articles") or phase.get("records") for phase in ACCUMULATED_PHASES):
            raise AssertionError("old phase membership survived migration")

        pipeline_calls = []

        def _offline_pipeline(row, candidates, err):
            pipeline_calls.append(row["source_id"])
            primary_url = candidates[0]["media_url"]
            return {
                "source_id": row["source_id"],
                "article_url": row["article_url"],
                "article_date": row["article_date"],
                "published_date_source": row["published_date_source"],
                "media_url_sha256": hashlib.sha256(primary_url.encode("utf-8")).hexdigest(),
                "technical_status": "PASS",
                "source_level_status": "ELIGIBLE_SOURCE",
                "duration_seconds": 0.1,
                "exclusion_reason": "",
            }, None

        def _save_beside_real(path=None):
            saved_save_checkpoint(migration_path if path is None else path)

        globals()["save_checkpoint"] = _save_beside_real
        globals()["run_article_through_pipeline"] = _offline_pipeline
        _reset_current_run()
        before = process_phase(PHASE_A_START, PHASE_A_END, PHASE_A_LABEL)
        if before["articles"] or before["records"] or pipeline_calls:
            raise AssertionError("an undiscovered 1.0 source entered the new phase")

        _only_discover(fail_url)
        failed = process_phase(PHASE_A_START, PHASE_A_END, PHASE_A_LABEL)
        if pipeline_calls != [fail_id]:
            raise AssertionError(pipeline_calls)
        if fail_hash in RUN_STATE["reusable_media_by_hash"]:
            raise AssertionError("failed media bypassed technical QA")
        failed_record = next(row for row in failed["records"] if row["source_id"] == fail_id)
        if failed_record["media_url_sha256"] != fail_hash:
            raise AssertionError(failed_record)

        _reset_current_run()
        pipeline_calls.clear()
        other_url = "https://vov4.vov.vn/bahnar/" + migrate_slug + "/migration-other.vov4"
        other_row = _cache_article(other_url, "2026-02-01", "metadata", "bai moi", media_url)
        _only_discover(other_url)
        reused_phase = process_phase(PHASE_A_START, PHASE_A_END, PHASE_A_LABEL)
        if pipeline_calls:
            raise AssertionError("current source downloaded instead of reusing hash H")
        if OWNED_MEDIA.get(media_hash) != other_row["source_id"]:
            raise AssertionError(OWNED_MEDIA)
        reused_record = next(row for row in reused_phase["records"] if row["source_id"] == other_row["source_id"])
        reused_article = next(row for row in reused_phase["articles"] if row["source_id"] == other_row["source_id"])
        if reused_record["article_date"] != "2026-02-01" or reused_record["published_date_source"] != "metadata":
            raise AssertionError(reused_record)
        if reused_article["article_date"] != reused_record["article_date"] or reused_article["published_date_source"] != reused_record["published_date_source"]:
            raise AssertionError((reused_article, reused_record))
        if reused_record["source_id"] != other_row["source_id"] or reused_record["article_url"] != other_url:
            raise AssertionError(reused_record)
        if reused_record["media_url_sha256"] != media_hash or reused_record["wav_sha256"] != pass_record["wav_sha256"]:
            raise AssertionError("reused record did not keep the verified artifact")
        if reused_record.get("published_date_source") == "body_fallback":
            raise AssertionError("stale body_fallback provenance survived reuse")
        expected_notes = (
            "content_class=news; technical_status=PASS; rule_reason=n/a"
        )
        if reused_record.get("verification_notes") != expected_notes:
            raise AssertionError(reused_record.get("verification_notes"))
        if reused_record.get("native_spotcheck_status") != "NOT_PERFORMED" or reused_record.get("native_spotcheck_notes"):
            raise AssertionError((reused_record.get("native_spotcheck_status"), reused_record.get("native_spotcheck_notes")))
        if any(row.get("source_id") == migrate_id for row in reused_phase["articles"]):
            raise AssertionError("old owner A reappeared in the current phase")

        _reset_current_run()
        pipeline_calls.clear()
        ARTICLE_CACHE[migrate_url]["media_candidates"] = [_public_audio(changed_media_url)]
        ARTICLE_CACHE[migrate_url]["article_row"]["article_date"] = "2026-02-01"
        ARTICLE_CACHE[migrate_url]["article_row"]["published_date_source"] = "metadata"
        _only_discover(migrate_url)
        changed = process_phase(PHASE_A_START, PHASE_A_END, PHASE_A_LABEL)
        if pipeline_calls != [migrate_id]:
            raise AssertionError(pipeline_calls)
        changed_record = next(row for row in changed["records"] if row["source_id"] == migrate_id)
        if changed_record["media_url_sha256"] != changed_hash or changed_hash in RUN_STATE["reusable_media_by_hash"]:
            raise AssertionError(changed_record)
        if changed_record["media_url_sha256"] == media_hash:
            raise AssertionError("H1 was reused for H2")

        _reset_current_run()
        pipeline_calls.clear()
        first_url = "https://vov4.vov.vn/bahnar/" + migrate_slug + "/migration-first.vov4"
        second_url = "https://vov4.vov.vn/bahnar/" + migrate_slug + "/migration-second.vov4"
        first_row = _cache_article(first_url, "2026-03-01", "metadata", "ban dau", media_url)
        second_row = _cache_article(second_url, "2026-03-02", "metadata", "ban sau", media_url)
        _only_discover(first_url, second_url)
        duplicate_phase = process_phase(PHASE_A_START, PHASE_A_END, PHASE_A_LABEL)
        current_ids = [row["source_id"] for row in duplicate_phase["articles"]]
        owner = OWNED_MEDIA.get(media_hash)
        if owner not in current_ids or owner == migrate_id:
            raise AssertionError((owner, current_ids))
        duplicates = [row for row in duplicate_phase["exclusions"] if row.get("exclusion_reason") == "DUPLICATE_MEDIA"]
        if len(duplicates) != 1 or duplicates[0]["canonical_source_id"] != owner:
            raise AssertionError(duplicates)
        if duplicates[0]["source_id"] not in current_ids or duplicates[0]["canonical_source_id"] not in current_ids:
            raise AssertionError(duplicates)
        if pipeline_calls:
            raise AssertionError("canonical current owner did not reuse verified media")

        _rewrite_compat(migration_path, {"crawl_pipeline_schema_version": "9.9"})
        try:
            load_checkpoint(migration_path)
            raise AssertionError("unsupported crawl schema was accepted")
        except CheckpointIncompatible:
            pass
    finally:
        globals()["save_checkpoint"] = saved_save_checkpoint
        globals()["run_article_through_pipeline"] = saved_pipeline
        VERIFY_CACHE.clear()
        VERIFY_CACHE.update(saved_verify)
        ARTICLE_CACHE.clear()
        ARTICLE_CACHE.update(saved_cache_m)
        SECTION_CRAWL_STATE.clear()
        SECTION_CRAWL_STATE.update(saved_sections_m)
        OWNED_MEDIA.clear()
        OWNED_MEDIA.update(saved_owned_m)
        NETWORK_STATE.clear()
        NETWORK_STATE.update(saved_network_m)
        RUN_STATE["processed_source_ids"] = saved_processed_m
        RUN_STATE["reusable_media_by_hash"] = saved_reusable_m
        RUN_STATE["total_budget_exhausted"] = saved_budget_m
        if saved_ceiling_m is None:
            RUN_STATE.pop("budget_exhausted_ceiling_bytes", None)
        else:
            RUN_STATE["budget_exhausted_ceiling_bytes"] = saved_ceiling_m
        ACCUMULATED_PHASES.clear()
        ACCUMULATED_PHASES.extend(saved_phases_m)
        shutil.rmtree(migration_dir, ignore_errors=True)

    saved_sections_phase_b = {slug: dict(state) for slug, state in SECTION_CRAWL_STATE.items()}
    phase_b_calls = []

    def _phase_b_crawl_should_not_run(*_args, **_kwargs):
        phase_b_calls.append("crawl")

    def _phase_b_branch(hours: float):
        if incomplete_section_slugs():
            return False
        triggered = hours < PHASE_B_TRIGGER_HOURS
        if triggered:
            _phase_b_crawl_should_not_run()
        return triggered

    try:
        SECTION_CRAWL_STATE.clear()
        for slug in ALL_SECTION_SLUGS:
            SECTION_CRAWL_STATE[slug] = blank_section_state()
        SECTION_CRAWL_STATE[ALL_SECTION_SLUGS[0]]["listing_scope_unverified"] = True
        if phase_b_allowed(0) or phase_b_allowed(15) or _phase_b_branch(0) or phase_b_calls:
            raise AssertionError("incomplete Phase A triggered Phase B")
        for slug in ALL_SECTION_SLUGS:
            state = blank_section_state()
            state["date_boundary_reached"] = True
            SECTION_CRAWL_STATE[slug] = state
        if incomplete_section_slugs() or not phase_b_allowed(6) or not _phase_b_branch(6) or phase_b_calls != ["crawl"]:
            raise AssertionError((incomplete_section_slugs(), phase_b_calls))
        phase_b_calls.clear()
        if phase_b_allowed(15) or _phase_b_branch(15) or phase_b_calls:
            raise AssertionError("complete Phase A at 15h triggered Phase B")
        SECTION_CRAWL_STATE[ALL_SECTION_SLUGS[0]]["listing_scope_unverified"] = True
        phase_b_calls.clear()
        if phase_b_allowed(15) or _phase_b_branch(15) or phase_b_calls:
            raise AssertionError("incomplete Phase A at 15h triggered Phase B")
    finally:
        SECTION_CRAWL_STATE.clear()
        SECTION_CRAWL_STATE.update(saved_sections_phase_b)

    guard_dir = OUT_DIR / "qa" / "_synthetic_final_guard"
    shutil.rmtree(guard_dir, ignore_errors=True)
    guard_dir.mkdir(parents=True, exist_ok=True)
    guard_summary = guard_dir / "summary.json"
    guard_summary.write_text(json.dumps({"status": "SUCCESS_RQ2_VOV4_FULL_SOURCE_POOL"}), encoding="utf-8")
    try:
        refuse_if_nb10_finalized(guard_summary)
        raise AssertionError("finalized SUCCESS allowed a network rerun")
    except RuntimeError as exc:
        if "finalized SUCCESS" not in str(exc):
            raise
    guard_summary.write_text(json.dumps({"status": "PARTIAL_RQ2_VOV4_FULL_SOURCE_POOL"}), encoding="utf-8")
    refuse_if_nb10_finalized(guard_summary)
    shutil.rmtree(guard_dir, ignore_errors=True)

    compiled = 0
    notebook_doc = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
    for index, cell in enumerate(notebook_doc.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue
        compile("".join(cell.get("source", [])), "cell_" + str(index), "exec")
        compiled += 1
    print("CODE_CELLS_COMPILED = " + str(compiled) + "/" + str(compiled))
    print("MAX_PAGES_PER_SECTION = " + str(MAX_PAGES_PER_SECTION))
    print("MAX_TOTAL_DOWNLOAD_BYTES = " + str(MAX_TOTAL_DOWNLOAD_BYTES))
    print("RUN_NETWORK_PIPELINE = " + str(RUN_NETWORK_PIPELINE))
    print("SYNTHETIC_CHECKS_PASS")


run_synthetic_checks()

DATE_PARSER_MIGRATION dropped 1 articles with untrusted dates; restarted 1 section cursors
RETRY download 1/2 in 2s: TransientHTTP https://cdn.example.com/a.mp3
RETRY download 2/2 in 4s: TransientHTTP https://cdn.example.com/a.mp3
CHECKPOINT_REPAIR: orphan_articles_removed = 0 processed_ids_released = 0 media_owners_released = 0
CHECKPOINT_REPAIR: orphan_articles_removed = 0 processed_ids_released = 0 media_owners_released = 0
CHECKPOINT_REPAIR: orphan_articles_removed = 0 processed_ids_released = 0 media_owners_released = 0
CHECKPOINT_ARTIFACT_REJECTED VOV4_23A5518B701085121391274F43D7F0A6854AF4A5
CHECKPOINT_REPAIR: orphan_articles_removed = 0 processed_ids_released = 0 media_owners_released = 0
CHECKPOINT_ARTIFACT_REJECTED VOV4_KEEP
CHECKPOINT_REPAIR: orphan_articles_removed = 0 processed_ids_released = 0 media_owners_released = 0
CHECKPOINT_REPAIR: orphan_articles_removed = 0 processed_ids_released = 0 media_owners_released = 0
download budget raised to 3221225472 bytes; interrupted

## 22. Quick audit

No network. Only shows what is currently on disk from the most recent completed run of Section 17-20
(empty until `RUN_NETWORK_PIPELINE` is flipped to `True` and the notebook is actually executed end to end).


In [22]:
manifest_path = OUT_DIR / "source_pool_manifest.csv"
if manifest_path.is_file():
    audit_df = pd.read_csv(manifest_path, dtype=str).fillna("")
    display(audit_df[["source_id", "article_date", "source_section", "content_class",
                       "artifact_origin", "media_type", "technical_status",
                       "source_level_status", "exclusion_reason"]])
    print(audit_df["source_level_status"].value_counts(dropna=False))
else:
    print("No source_pool_manifest.csv yet -- run the network pipeline (Section 17) first.")


,source_id,article_date,source_section,content_class,artifact_origin,media_type,technical_status,source_level_status,exclusion_reason
0,VOV4_000C97ECEDF969399D163187D048041974BBA7B1,2026-07-11,nor-pore-phat-thanh,radio_report,nb10_full,audio,PASS,ELIGIBLE_SOURCE,
1,VOV4_005132CEF4F7F7C7DE21937AAA3FD363AEE84596,2026-03-31,nor-pore-phat-thanh,radio_report,nb10_full,audio,PASS,ELIGIBLE_SOURCE,
2,VOV4_00988C15C111CBF0C2524C8ABC39E0ECDE96E685,2026-09-02,johngom-pran-suc-khoe,health,nb10_full,audio,PASS,ELIGIBLE_SOURCE,
3,VOV4_0169AD0B268DF93C463910E926DF7658100374D4,2026-01-12,video-clip-bai-joh-hori-video-clip-ca-nhac,music,nb10_full,video,PASS,REJECTED_SOURCE,MUSIC_CONTENT
4,VOV4_018F6A7650D2688842F80E3E5B9A6677FA534197,2026-01-26,trong-jang-teh-dak-tojur-an-pang-todrong-horih...,news,nb10_full,audio,PASS,ELIGIBLE_SOURCE,
...,...,...,...,...,...,...,...,...,...
675,VOV4_FD93BFA85367848236F0D89374AA620D280D7FCD,2026-07-19,pom-tong-san-khau-truyen-thanh-tieu-pham-cuoi-...,radio_drama,nb10_full,audio,PASS,NEEDS_REVIEW,AMBIGUOUS_CONTENT_CLASS:radio_drama
676,VOV4_FDEC5468F6240325AD7A0BCB73B6B5145392B43C,2026-03-02,video-clip-bai-joh-hori-video-clip-ca-nhac,music,nb10_full,video,PASS,REJECTED_SOURCE,MUSIC_CONTENT
677,VOV4_FEF440E7AE6BB5F5878E173D1396E28F5580AD3A,2026-03-28,toring-torang-nao-nong-thon-moi,rural_affairs,nb10_full,audio,PASS,ELIGIBLE_SOURCE,
678,VOV4_FF708E2757C0FEEC590E890EB5C8EB41518E7461,2026-07-01,todrong-hori-tay-nguyen-dan-ca-tay-nguyen,music,nb10_full,audio,PASS,REJECTED_SOURCE,MUSIC_CONTENT


source_level_status
ELIGIBLE_SOURCE    335
REJECTED_SOURCE    284
NEEDS_REVIEW        61
Name: count, dtype: int64
